In [ ]:
import os
import shutil
import sys
from pathlib import Path


# Try new-style mount first, fall back to old-style
def _find(slug, subdir=''):
    new = f'/kaggle/input/{slug}/{subdir}'.rstrip('/')
    old = f'/kaggle/input/datasets/rishig777/{slug}/{subdir}'.rstrip('/')
    return new if Path(new).exists() else old

AUDIO      = _find('calmsep-8k-slice', 'calmsep-kaggle')
MODEL_DS   = _find('calmsep-model',    'calmsep-tiny')
ADAPTERS   = _find('calmsep-stage1-adapters')
GATE_DS    = _find('calmsep-stage3-gate')
SRCORRNET  = f'{MODEL_DS}/sr_corrnet_src'
HF_CACHE   = f'{MODEL_DS}/hf_cache'
PROJ       = '/tmp/calmsep_project'
WORK       = '/kaggle/working'
CKPT_DIR   = f'{WORK}/checkpoints/stage4_joint'
STAGE1_DIR = f'{WORK}/checkpoints/stage1'

os.environ['HF_HOME']             = HF_CACHE
os.environ['HF_HUB_OFFLINE']      = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import torch

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
print('BF16:', torch.cuda.is_bf16_supported())

for label, path in [('AUDIO', AUDIO), ('MODEL_DS', MODEL_DS),
                     ('ADAPTERS', ADAPTERS), ('GATE_DS', GATE_DS)]:
    exists = Path(path).exists()
    print(f'  [{"OK" if exists else "MISSING"}] {label}: {path}')


In [ ]:
import base64
import os
import subprocess
from pathlib import Path

# SR-CorrNet: copy source tree (same method Stage 3 uses)
if Path(SRCORRNET).exists():
    shutil.copytree(SRCORRNET, '/tmp/sr_corrnet_src', dirs_exist_ok=True)
    print('SR-CorrNet src copied from:', SRCORRNET)
else:
    print('WARNING: sr_corrnet_src not found at', SRCORRNET)

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'soundfile', 'librosa', 'scipy', 'tqdm', '-q'], check=True)

sys.path.insert(0, '/tmp/sr_corrnet_src')

# loguru stub
os.makedirs('/tmp/loguru_stub/loguru', exist_ok=True)
open('/tmp/loguru_stub/loguru/__init__.py', 'w').write(
    'import logging as _l, sys as _s\n'
    '_l.basicConfig(level=_l.DEBUG, stream=_s.stderr, format="%(asctime)s %(levelname)s %(message)s")\n'
    'class _L:\n'
    '    def __call__(self, *a, **k): return self\n'
    '    def __getattr__(self, n): return self\n'
    '    def log(self, level, msg="", *a, **k): _l.getLogger("loguru").info(str(msg))\n'
    '    def info(self, msg="", *a, **k): _l.getLogger("loguru").info(str(msg))\n'
    '    def debug(self, msg="", *a, **k): _l.getLogger("loguru").debug(str(msg))\n'
    '    def warning(self, msg="", *a, **k): _l.getLogger("loguru").warning(str(msg))\n'
    '    def error(self, msg="", *a, **k): _l.getLogger("loguru").error(str(msg))\n'
    '    def disable(self, name=""): pass\n'
    '    def enable(self, name=""): pass\n'
    'logger = _L()\n'
)
sys.path.insert(0, '/tmp/loguru_stub')

# rotary_emb stub
os.makedirs('/tmp/rotary_stub/rotary_embedding_torch', exist_ok=True)
open('/tmp/rotary_stub/rotary_embedding_torch/__init__.py', 'w').write(
    'import torch, torch.nn as nn\n'
    'class RotaryEmbedding(nn.Module):\n'
    '    def __init__(self, *a, **kw): super().__init__()\n'
    '    def rotate_queries_or_keys(self, t): return t\n'
    '    def forward(self, t): return t\n'
)
sys.path.insert(0, '/tmp/rotary_stub')

try:
    from sr_corrnet import SSInference
    print('SSInference: OK')
except ImportError as e:
    print('SSInference import failed:', e)


In [ ]:
import os
import shutil
from pathlib import Path

os.makedirs(STAGE1_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Stage 1 adapters
for fname in ['best_reverb.pt', 'best_noise.pt', 'best_codec.pt']:
    src = os.path.join(ADAPTERS, fname)
    dst = os.path.join(STAGE1_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Copied {fname}: {round(os.path.getsize(dst)/1e3, 1)} KB')
    else:
        print(f'WARNING: {fname} not found in {ADAPTERS}')
        # Search recursively
        found = list(Path(ADAPTERS).rglob(fname))
        if found:
            shutil.copy2(found[0], dst)
            print(f'  Found at {found[0]} and copied')

# Stage 3 gate checkpoint
for gname in ['best_gate.pt', 'final_gate.pt']:
    src = os.path.join(GATE_DS, gname)
    dst = os.path.join(CKPT_DIR, gname.replace('gate', 'gate_init'))
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Copied {gname}: {round(os.path.getsize(src)/1e3, 1)} KB')
    else:
        found = list(Path(GATE_DS).rglob(gname))
        if found:
            shutil.copy2(found[0], dst)
            print(f'  {gname} found at {found[0]} and copied')
        else:
            print(f'WARNING: {gname} not found in {GATE_DS}')


In [ ]:
import os
from pathlib import Path

PROJ = '/tmp/calmsep_project'
for sub in ('train', 'models', 'data'):
    Path(f'{PROJ}/{sub}').mkdir(parents=True, exist_ok=True)

open(f'{PROJ}/train/__init__.py','wb').write(base64.b64decode('IiIiVHJhaW5pbmcgbG9vcHMgYW5kIGNvbXBvc2l0ZSBsb3NzIGFzc2VtYmx5IChEZXYgQikuIiIiCg=='))
open(f'{PROJ}/train/losses.py','wb').write(base64.b64decode('IiIiClNoYXJlZCBsb3NzIGZ1bmN0aW9ucyBmb3IgQ0FMTS1TZXAgYWRhcHRlciB0cmFpbmluZyAoRGV2IEIsIFAxLUIzKS4KClJldXNlcyB0aGUgYmFja2JvbmUgZW5naW5lIGxvc3NlcyB3aGVyZSBwb3NzaWJsZToKICBwcmltYXJ5OiBQSVQgU0ktU05SIG9uIHdhdmVmb3JtcyAodGltZSBkb21haW4pCiAgc2Vjb25kYXJ5OiAwLjUgw5cgUElUIFNJLVNOUiBvbiBtYWduaXR1ZGUgU1RGVCAoZnJlcXVlbmN5IGRvbWFpbikKICBhdHRyYWN0b3I6IEJDRSBvbiBwcmVzWyJsb2dpdHMiXSAoc2hhcGUgMSw3KQoKQ2FyZGluYWxpdHktYXdhcmUgZXh0ZW5zaW9uIChCTFVFUFJJTlQgwqc4LjIpOgogIE1pc3NlZCBzcGVha2VycyBzY29yZSAwIGRCLiBIYWxsdWNpbmF0ZWQgc3RyZWFtcyBpbmN1ciAtMSBkQiBwZXIgc3RyZWFtLgogIEFwcGxpZWQgYXQgUElUIHNlbGVjdGlvbiB0aW1lOiB0aGUgUElUIG1hdHJpeCBpcyBleHRlbmRlZCB3aXRoIHplcm8tY29sdW1ucwogIGZvciBtaXNzaW5nIHJlZmVyZW5jZXMgc28gdGhlIEh1bmdhcmlhbiBhc3NpZ25tZW50IGNhbiBtYXAgYSBzdHJlYW0gdG8gc2lsZW5jZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKX0VQUyA9IDFlLTEwCl9IQUxMX1BFTkFMVFlfREIgPSAtMS4wCl9TVEZUX1dJTiA9IDEyOApfU1RGVF9IT1AgPSA2NAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU0ktU05SCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKZGVmIHNpX3Nucihlc3RpbWF0ZTogdG9yY2guVGVuc29yLCB0YXJnZXQ6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiCiAgICBTY2FsZS1pbnZhcmlhbnQgU05SLCBzaGFwZS1hZ25vc3RpYy4KCiAgICBBcmdzOgogICAgICAgIGVzdGltYXRlOiBbLi4uLCBUX2VdCiAgICAgICAgdGFyZ2V0OiAgIFsuLi4sIFRfcl0gIChtYXkgZGlmZmVyIGZyb20gVF9lIGR1ZSB0byBTVEZUL2lTVEZUIGJvdW5kYXJ5KQogICAgUmV0dXJuczoKICAgICAgICBbLi4uXSBTSS1TTlIgaW4gZEIuCiAgICAiIiIKICAgIG1pbl90ID0gbWluKGVzdGltYXRlLnNoYXBlWy0xXSwgdGFyZ2V0LnNoYXBlWy0xXSkKICAgIGVzdGltYXRlID0gZXN0aW1hdGVbLi4uLCA6bWluX3RdCiAgICB0YXJnZXQgPSB0YXJnZXRbLi4uLCA6bWluX3RdCiAgICBlc3RpbWF0ZSA9IGVzdGltYXRlIC0gZXN0aW1hdGUubWVhbihkaW09LTEsIGtlZXBkaW09VHJ1ZSkKICAgIHRhcmdldCA9IHRhcmdldCAtIHRhcmdldC5tZWFuKGRpbT0tMSwga2VlcGRpbT1UcnVlKQogICAgZG90ID0gKGVzdGltYXRlICogdGFyZ2V0KS5zdW0oZGltPS0xLCBrZWVwZGltPVRydWUpCiAgICBzX3RhcmdldCA9IGRvdCAvICh0YXJnZXQucG93KDIpLnN1bShkaW09LTEsIGtlZXBkaW09VHJ1ZSkgKyBfRVBTKSAqIHRhcmdldAogICAgbm9pc2UgPSBlc3RpbWF0ZSAtIHNfdGFyZ2V0CiAgICByZXR1cm4gMTAuMCAqIHRvcmNoLmxvZzEwKHNfdGFyZ2V0LnBvdygyKS5zdW0oLTEpIC8gKG5vaXNlLnBvdygyKS5zdW0oLTEpICsgX0VQUykgKyBfRVBTKQoKCmRlZiBwaXRfc2lfc25yKAogICAgZXN0aW1hdGVzOiB0b3JjaC5UZW5zb3IsIHJlZmVyZW5jZXM6IHRvcmNoLlRlbnNvcgopIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgbGlzdFtsaXN0W2ludF1dXToKICAgICIiIgogICAgUGVybXV0YXRpb24tSW52YXJpYW50IFRyYWluaW5nIFNJLVNOUi4KCiAgICBBcmdzOgogICAgICAgIGVzdGltYXRlczogIChCLCBLX2hhdCwgVCkgc2VwYXJhdGVkIHN0cmVhbXMuCiAgICAgICAgcmVmZXJlbmNlczogKEIsIEtfcmVmLCBUKSBjbGVhbiByZWZlcmVuY2VzLgogICAgUmV0dXJuczoKICAgICAgICBtZWFuX3Npc25yOiAoQiwpIG1lYW4gU0ktU05SIGFmdGVyIG9wdGltYWwgcGVybXV0YXRpb24uCiAgICAgICAgcGVybXM6IExpc3Qgb2YgQiBwZXJtdXRhdGlvbiBsaXN0cyBtYXBwaW5nIEtfaGF0IOKGkiBLX3JlZi4KICAgICIiIgogICAgQiwgS19oYXQsIFRfZXN0ID0gZXN0aW1hdGVzLnNoYXBlCiAgICBLX3JlZiwgVF9yZWYgPSByZWZlcmVuY2VzLnNoYXBlWzFdLCByZWZlcmVuY2VzLnNoYXBlWzJdCiAgICBUID0gbWluKFRfZXN0LCBUX3JlZikgICMgYWxpZ24gbGVuZ3RoczogaVNURlQgY2FuIGRpZmZlciBieSBhIGZldyBzYW1wbGVzCiAgICBlc3RpbWF0ZXMgPSBlc3RpbWF0ZXNbLi4uLCA6VF0KICAgIHJlZmVyZW5jZXMgPSByZWZlcmVuY2VzWy4uLiwgOlRdCiAgICBLID0gbWF4KEtfaGF0LCBLX3JlZikKCiAgICAjIFBhZCBzbWFsbGVyIHRlbnNvciB0byBLLgogICAgaWYgS19oYXQgPCBLOgogICAgICAgIHBhZCA9IHRvcmNoLnplcm9zKEIsIEsgLSBLX2hhdCwgVCwgZGV2aWNlPWVzdGltYXRlcy5kZXZpY2UpCiAgICAgICAgZXN0aW1hdGVzX3BhZGRlZCA9IHRvcmNoLmNhdChbZXN0aW1hdGVzLCBwYWRdLCBkaW09MSkKICAgIGVsc2U6CiAgICAgICAgZXN0aW1hdGVzX3BhZGRlZCA9IGVzdGltYXRlcwoKICAgIGlmIEtfcmVmIDwgSzoKICAgICAgICBwYWQgPSB0b3JjaC56ZXJvcyhCLCBLIC0gS19yZWYsIFQsIGRldmljZT1yZWZlcmVuY2VzLmRldmljZSkKICAgICAgICByZWZzX3BhZGRlZCA9IHRvcmNoLmNhdChbcmVmZXJlbmNlcywgcGFkXSwgZGltPTEpCiAgICBlbHNlOgogICAgICAgIHJlZnNfcGFkZGVkID0gcmVmZXJlbmNlcwoKICAgICMgQnVpbGQgY29zdCBtYXRyaXggW0IsIEssIEtdLgogICAgY29zdCA9IHRvcmNoLnplcm9zKEIsIEssIEssIGRldmljZT1lc3RpbWF0ZXMuZGV2aWNlKQogICAgZm9yIGkgaW4gcmFuZ2UoSyk6CiAgICAgICAgZm9yIGogaW4gcmFuZ2UoSyk6CiAgICAgICAgICAgIGNvc3RbOiwgaSwgal0gPSAtc2lfc25yKGVzdGltYXRlc19wYWRkZWRbOiwgaV0sIHJlZnNfcGFkZGVkWzosIGpdKQoKICAgICMgSHVuZ2FyaWFuIHZpYSBzY2lweSAoQ1BVIG9ubHkgZm9yIG5vdyDigJQgc21hbGwgSykuCiAgICAjIElNUE9SVEFOVDogdXNlIC1jb3N0IHZhbHVlcyBmcm9tIHRoZSBwcmUtY29tcHV0ZWQgUHlUb3JjaCBjb3N0IG1hdHJpeAogICAgIyByYXRoZXIgdGhhbiBjYWxsaW5nIHNpX3NucigpIGFnYWluLCBzbyBncmFkaWVudCBmbG93cyB0aHJvdWdoIGNvc3Qg4oaSIGVzdGltYXRlcy4KICAgIGZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IGxpbmVhcl9zdW1fYXNzaWdubWVudAoKICAgIHBlcm1zOiBsaXN0W2xpc3RbaW50XV0gPSBbXQogICAgc2lzbnJfcGVyX3NhbXBsZTogbGlzdFt0b3JjaC5UZW5zb3JdID0gW10KCiAgICBmb3IgYiBpbiByYW5nZShCKToKICAgICAgICByb3dfaW5kLCBjb2xfaW5kID0gbGluZWFyX3N1bV9hc3NpZ25tZW50KGNvc3RbYl0uZGV0YWNoKCkuY3B1KCkubnVtcHkoKSkKICAgICAgICBwZXJtID0gbGlzdChjb2xfaW5kWzpLX2hhdF0pCiAgICAgICAgcGVybXMuYXBwZW5kKHBlcm0pCiAgICAgICAgbl9wYWlycyA9IG1pbihLX2hhdCwgS19yZWYpCiAgICAgICAgIyBLZWVwIHZhbHVlcyBhcyB0ZW5zb3JzIChOT1QgLml0ZW0oKSkgc28gYmFja3dhcmQgY2FuIGZsb3cgdGhyb3VnaCB0aGVtLgogICAgICAgIG1hdGNoZWQgPSB0b3JjaC5zdGFjayhbLWNvc3RbYiwgcm93X2luZFtpXSwgY29sX2luZFtpXV0gZm9yIGkgaW4gcmFuZ2Uobl9wYWlycyldKQogICAgICAgIG5faGFsbCA9IG1heCgwLCBLX2hhdCAtIEtfcmVmKQogICAgICAgIHNpc25yX3Blcl9zYW1wbGUuYXBwZW5kKG1hdGNoZWQubWVhbigpICsgbl9oYWxsICogX0hBTExfUEVOQUxUWV9EQikKCiAgICBzaXNucl92YWxzID0gdG9yY2guc3RhY2soc2lzbnJfcGVyX3NhbXBsZSkgICMgKEIsKSB3aXRoIGdyYWRfZm4KICAgIHJldHVybiBzaXNucl92YWxzLCBwZXJtcwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU1RGVCBtYWduaXR1ZGUgbG9zcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBfbWFnX3N0ZnQod2F2ZWZvcm06IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiW0IsIFRdIOKGkiBbQiwgRiwgZnJhbWVzXSBtYWduaXR1ZGUgU1RGVCBhdCA4IGtIeiBTUi1Db3JyTmV0IHBhcmFtcy4iIiIKICAgIEIsIFQgPSB3YXZlZm9ybS5zaGFwZQogICAgd2luZG93ID0gdG9yY2guaGFubl93aW5kb3coX1NURlRfV0lOLCBkZXZpY2U9d2F2ZWZvcm0uZGV2aWNlKQogICAgc3BlYyA9IHRvcmNoLnN0ZnQoCiAgICAgICAgd2F2ZWZvcm0sCiAgICAgICAgbl9mZnQ9X1NURlRfV0lOLAogICAgICAgIGhvcF9sZW5ndGg9X1NURlRfSE9QLAogICAgICAgIHdpbl9sZW5ndGg9X1NURlRfV0lOLAogICAgICAgIHdpbmRvdz13aW5kb3csCiAgICAgICAgcmV0dXJuX2NvbXBsZXg9VHJ1ZSwKICAgICAgICBjZW50ZXI9VHJ1ZSwKICAgICkKICAgIHJldHVybiBzcGVjLmFicygpICAjIFtCLCBGLCBmcmFtZXNdCgoKZGVmIHBpdF9zaV9zbnJfbWFnKGVzdGltYXRlczogdG9yY2guVGVuc29yLCByZWZlcmVuY2VzOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIlBJVCBTSS1TTlIgb24gbWFnbml0dWRlIFNURlQsIGF2ZXJhZ2VkIG92ZXIgQi4gUmV0dXJucyBzY2FsYXIuIiIiCiAgICBCLCBLX2hhdCwgVCA9IGVzdGltYXRlcy5zaGFwZQogICAgS19yZWYgPSByZWZlcmVuY2VzLnNoYXBlWzFdCgogICAgIyBGbGF0dGVuIHRvIFtCKkssIFRdIGZvciBiYXRjaCBTVEZULgogICAgZXN0X21hZyA9IF9tYWdfc3RmdChlc3RpbWF0ZXMucmVzaGFwZShCICogS19oYXQsIFQpKS5yZXNoYXBlKEIsIEtfaGF0LCAtMSkKICAgIHJlZl9tYWcgPSBfbWFnX3N0ZnQocmVmZXJlbmNlcy5yZXNoYXBlKEIgKiBLX3JlZiwgVCkpLnJlc2hhcGUoQiwgS19yZWYsIC0xKQoKICAgIHNpc25yX3ZhbHMsIF8gPSBwaXRfc2lfc25yKGVzdF9tYWcsIHJlZl9tYWcpCiAgICByZXR1cm4gc2lzbnJfdmFscy5tZWFuKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEF0dHJhY3RvciBCQ0UKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpkZWYgYXR0cmFjdG9yX2JjZShsb2dpdHM6IHRvcmNoLlRlbnNvciwgbl9zcGVha2VyczogbGlzdFtpbnRdKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiIKICAgIEJDRSBvbiBhdHRyYWN0b3IgcHJlc2VuY2UgbG9naXRzIChzaGFwZSBCLCA3IG9yIDEsIDcpLgoKICAgIEFyZ3M6CiAgICAgICAgbG9naXRzOiBSYXcgbG9naXRzIGZyb20gcHJlc1sibG9naXRzIl0sIHNoYXBlIChCLCA3KSBvciAoQiwgMSwgNykuCiAgICAgICAgbl9zcGVha2VyczogVHJ1ZSBzcGVha2VyIGNvdW50IHBlciBzYW1wbGUuCiAgICBSZXR1cm5zOgogICAgICAgIFNjYWxhciBCQ0UgbG9zcy4KICAgICIiIgogICAgbG9naXRzID0gbG9naXRzLnNxdWVlemUoMSkgaWYgbG9naXRzLm5kaW0gPT0gMyBlbHNlIGxvZ2l0cyAgIyAoQiwgNykKICAgIHRhcmdldHMgPSB0b3JjaC56ZXJvc19saWtlKGxvZ2l0cykKICAgIGZvciBiLCBuIGluIGVudW1lcmF0ZShuX3NwZWFrZXJzKToKICAgICAgICAjIFNsb3RzIDEuLm4gYXJlIGFjdGl2ZS4KICAgICAgICB0YXJnZXRzW2IsIDEgOiBuICsgMV0gPSAxLjAKICAgIHJldHVybiBGLmJpbmFyeV9jcm9zc19lbnRyb3B5X3dpdGhfbG9naXRzKGxvZ2l0cywgdGFyZ2V0cykKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENvbWJpbmVkIHRyYWluaW5nIGxvc3MKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpkZWYgY2FsbXNlcF9sb3NzKAogICAgZXN0aW1hdGVzOiB0b3JjaC5UZW5zb3IsCiAgICByZWZlcmVuY2VzOiB0b3JjaC5UZW5zb3IsCiAgICBsb2dpdHM6IHRvcmNoLlRlbnNvciB8IE5vbmUsCiAgICBuX3NwZWFrZXJzOiBsaXN0W2ludF0sCiAgICBtYWdfd2VpZ2h0OiBmbG9hdCA9IDAuNSwKICAgIGF0dF93ZWlnaHQ6IGZsb2F0ID0gMC4xLAopIC0+IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdOgogICAgIiIiCiAgICBDb21iaW5lZCBDQUxNLVNlcCBzZXBhcmF0aW9uIGxvc3MuCgogICAgPSBQSVRfU0lTTlJfdGltZSArIG1hZ193ZWlnaHQgw5cgUElUX1NJU05SX21hZyArIGF0dF93ZWlnaHQgw5cgQkNFKGF0dHJhY3RvcikKCiAgICBBcmdzOgogICAgICAgIGVzdGltYXRlczogIChCLCBLX2hhdCwgVCkgc2VwYXJhdGVkIHdhdmVmb3Jtcy4KICAgICAgICByZWZlcmVuY2VzOiAoQiwgS19yZWYsIFQpIGNsZWFuIHJlZmVyZW5jZSB3YXZlZm9ybXMuCiAgICAgICAgbG9naXRzOiAgICAgKEIsIDcpIG9yIE5vbmUuIElmIE5vbmUsIGF0dHJhY3RvciBsb3NzIGlzIHNraXBwZWQuCiAgICAgICAgbl9zcGVha2VyczogVHJ1ZSBjb3VudCBwZXIgc2FtcGxlLgogICAgICAgIG1hZ193ZWlnaHQ6IFdlaWdodCBvbiBTVEZUIG1hZ25pdHVkZSBsb3NzLgogICAgICAgIGF0dF93ZWlnaHQ6IFdlaWdodCBvbiBhdHRyYWN0b3IgQkNFLgoKICAgIFJldHVybnM6CiAgICAgICAgRGljdCB3aXRoICd0b3RhbCcsICd0aW1lJywgJ21hZycsICdhdHQnIHNjYWxhciB0ZW5zb3JzLgogICAgIiIiCiAgICAjIEFsaWduIHRpbWUgYXhpczogaVNURlQgbWF5IHByb2R1Y2UgwrFmZXcgc2FtcGxlcyB2cyB0aGUgcmVmZXJlbmNlCiAgICBtaW5fdCA9IG1pbihlc3RpbWF0ZXMuc2hhcGVbLTFdLCByZWZlcmVuY2VzLnNoYXBlWy0xXSkKICAgIGVzdGltYXRlcyA9IGVzdGltYXRlc1suLi4sIDptaW5fdF0KICAgIHJlZmVyZW5jZXMgPSByZWZlcmVuY2VzWy4uLiwgOm1pbl90XQoKICAgIHNpc25yX3ZhbHMsIF8gPSBwaXRfc2lfc25yKGVzdGltYXRlcywgcmVmZXJlbmNlcykKICAgIGxvc3NfdGltZSA9IC1zaXNucl92YWxzLm1lYW4oKQogICAgbG9zc19tYWcgPSAtcGl0X3NpX3Nucl9tYWcoZXN0aW1hdGVzLCByZWZlcmVuY2VzKQoKICAgIHRvdGFsID0gbG9zc190aW1lICsgbWFnX3dlaWdodCAqIGxvc3NfbWFnCgogICAgbG9zc19hdHQgPSB0b3JjaC50ZW5zb3IoMC4wLCBkZXZpY2U9ZXN0aW1hdGVzLmRldmljZSkKICAgIGlmIGxvZ2l0cyBpcyBub3QgTm9uZToKICAgICAgICBsb3NzX2F0dCA9IGF0dHJhY3Rvcl9iY2UobG9naXRzLCBuX3NwZWFrZXJzKQogICAgICAgIHRvdGFsID0gdG90YWwgKyBhdHRfd2VpZ2h0ICogbG9zc19hdHQKCiAgICByZXR1cm4geyJ0b3RhbCI6IHRvdGFsLCAidGltZSI6IGxvc3NfdGltZSwgIm1hZyI6IGxvc3NfbWFnLCAiYXR0IjogbG9zc19hdHR9Cg=='))
open(f'{PROJ}/train/stage1_single.py','wb').write(base64.b64decode('...'))
open(f'{PROJ}/train/stage3_gate.py','wb').write(base64.b64decode('...'))
open(f'{PROJ}/train/stage4_joint.py','wb').write(base64.b64decode('IiIiClN0YWdlIDQ6IEpvaW50IGdhdGVkIGVuZC10by1lbmQgcG9saXNoaW5nIChEZXYgQiwgUDMtQjEpLgoKR2F0ZSArIExldmVsLTIgYW5hbHl6ZXIgY29udGludWUgZnJvbSBTdGFnZSAzLiBMb1JBIGFkYXB0ZXJzIGZyb20gU3RhZ2UgMQphcmUgbm93IHVwZGF0ZWQgYnkgZ3JhZGllbnRzIGZsb3dpbmcgZnJvbSB0aGUgc2VwYXJhdGlvbiBsb3NzIHRocm91Z2ggdGhlCmRpZmZlcmVudGlhYmxlICh0ZW5zb3IpIGdhdGUgd2VpZ2h0czoKCiAgICB5ID0gVzAgeCArIGdfcsK3QnIoQXIgeCkgKyBnX27Ct0JuKEFuIHgpICsgZ19jwrdCYyhBYyB4KQogICAgd2hlcmUgKGdfciwgZ19uLCBnX2MpID0gZ2F0ZV9uZXQoY29uZGl0aW9uKSDigJQgZ3JhZC1lbmFibGVkIHRlbnNvcnMuCgpDcml0aWNhbCBkaWZmZXJlbmNlcyBmcm9tIFN0YWdlIDM6CiAg4oCiIE5vIHRvcmNoLm5vX2dyYWQoKSBhcm91bmQgdGhlIFNSLUNvcnJOZXQgZm9yd2FyZCDihpIgc2VwX2xvc3MgaGFzIGdyYWQKICDigKIgR2F0ZSB2YWx1ZXMgc3RheSBhcyBUZW5zb3JzIChub3QgY29udmVydGVkIHRvIGZsb2F0KSDihpIgZ3JhZCBmbG93cyBpbnRvIGFkYXB0ZXJzCiAg4oCiIEYucGFkICsgc3RhY2sgZm9yIGVzdGltYXRlcyBpbnN0ZWFkIG9mIGluLXBsYWNlIHplcm9zIOKGkiBncmFkaWVudCBncmFwaCBpbnRhY3QKICDigKIgaW5uZXIuZXZhbCgpIGtlZXBzIGJhc2UgQk4vZHJvcG91dCBkZXRlcm1pbmlzdGljOyBMb1JBIHBhcmFtcyBzdGlsbCBnZXQgZ3JhZAogIOKAoiBPLUxvUkEgcGVuYWx0eSBhbHdheXMgYWN0aXZlIChhZGFwdGVyIEEtbWF0cml4IG9ydGhvZ29uYWxpdHkpCgpJbml0aWFsaXNhdGlvbjoKICBnYXRlICsgYW5hbHl6ZXIgICDihpAgU3RhZ2UgMyBiZXN0X2dhdGUucHQgICAoLS1zdGFnZTMtZGlyIG9yIC0tZ2F0ZS1jaGVja3BvaW50KQogIGFkYXB0ZXJzICAgICAgICAgIOKGkCBTdGFnZSAxIGJlc3RfKi5wdCAgICAgICAoLS1zdGFnZTEtZGlyKQogIFNSLUNvcnJOZXQgYmFzZSAgIEZST1pFTiB0aHJvdWdob3V0CgpPcHRpbWlzZXI6IEFkYW1XLCB0d28gcGFyYW0gZ3JvdXBzCiAgYWRhcHRlcnMgICAgICAgICAgbHIgPSAxZS01ICAgd2QgPSAxZS00CiAgZ2F0ZSArIGFuYWx5emVyICAgbHIgPSAyZS01ICAgd2QgPSAxZS01CgpTYXZlcyBwZXIgZXBvY2ggKHdoZW4gbG9zcyBpbXByb3Zlcyk6CiAgYmVzdF9qb2ludC5wdCAgIOKAlCB7IGdhdGUsIGFuYWx5emVyLCBhZGFwdGVyX3N0YXRlIChmbGF0IEEvQiBkaWN0KSB9CiAgZmluYWxfam9pbnQucHQgIOKAlCBzYW1lLCBhZnRlciBsYXN0IGVwb2NoCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBsb2dnaW5nCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKaW1wb3J0IHRvcmNoLm9wdGltIGFzIG9wdGltCgpmcm9tIG1vZGVscy5jb25kaXRpb24gaW1wb3J0IExldmVsMkFuYWx5emVyLCBsZXZlbDFfdGVuc29yLCBsZXZlbDJfbG9zcwpmcm9tIG1vZGVscy5nYXRlIGltcG9ydCBHYXRlTmV0d29yaywgb3JhY2xlX2dhdGUKZnJvbSBtb2RlbHMubG9yYSBpbXBvcnQgQURBUFRFUl9OQU1FUywgTG9SQUxpYnJhcnksIExvUkFMaW5lYXIsIG9sb3JhX3BlbmFsdHkKZnJvbSB0cmFpbi5sb3NzZXMgaW1wb3J0IGNhbG1zZXBfbG9zcwpmcm9tIHRyYWluLnN0YWdlMV9zaW5nbGUgaW1wb3J0ICgKICAgIF9mb3J3YXJkX3dpdGhfZ3JhZCwKICAgIF9nZXRfaW5uZXJfbW9kdWxlLAogICAgX2xvYWRfbW9kZWwsCiAgICBfc2VlZF9ldmVyeXRoaW5nLAopCmZyb20gdHJhaW4uc3RhZ2UzX2dhdGUgaW1wb3J0IF9idWlsZF9nYXRlX2RhdGFzZXQsIF9sb2FkX2FkYXB0ZXJzCgpsb2dnaW5nLmJhc2ljQ29uZmlnKGxldmVsPWxvZ2dpbmcuSU5GTywgZm9ybWF0PSIlKGFzY3RpbWUpcyAlKGxldmVsbmFtZSlzICUobWVzc2FnZSlzIikKbG9nID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pCgpfT0xPUkFfQUxQSEE6IGZsb2F0ID0gMWUtMwpfTDFfTEFNQkRBOiBmbG9hdCA9IDFlLTMKCgpkZWYgX2xvYWRfc3RhZ2UzX2NrcHQoCiAgICBwYXRoOiBQYXRoLAogICAgYW5hbHl6ZXI6IExldmVsMkFuYWx5emVyLAogICAgZ2F0ZV9uZXQ6IEdhdGVOZXR3b3JrLAogICAgZGV2aWNlOiB0b3JjaC5kZXZpY2UsCikgLT4gTm9uZToKICAgIGNrcHQgPSB0b3JjaC5sb2FkKHN0cihwYXRoKSwgbWFwX2xvY2F0aW9uPWRldmljZSkKICAgIGFuYWx5emVyLmxvYWRfc3RhdGVfZGljdChja3B0WyJhbmFseXplciJdKQogICAgZ2F0ZV9uZXQubG9hZF9zdGF0ZV9kaWN0KGNrcHRbImdhdGUiXSkKICAgIGxvZy5pbmZvKCJMb2FkZWQgU3RhZ2UgMyBjaGVja3BvaW50OiAlcyIsIHBhdGgpCgoKZGVmIF9zYXZlX2pvaW50X2NrcHQoCiAgICBwYXRoOiBQYXRoLAogICAgZ2F0ZV9uZXQ6IEdhdGVOZXR3b3JrLAogICAgYW5hbHl6ZXI6IExldmVsMkFuYWx5emVyLAogICAgaW5uZXI6IHRvcmNoLm5uLk1vZHVsZSwKKSAtPiBOb25lOgogICAgIiIiU2F2ZSBnYXRlLCBhbmFseXplciwgYW5kIGFsbCByZWZpbmVkIExvUkEgQS9CIHdlaWdodHMuIiIiCiAgICBhZGFwdGVyX3N0YXRlOiBkaWN0W3N0ciwgdG9yY2guVGVuc29yXSA9IHt9CiAgICBmb3IgbW9kX25hbWUsIG1vZCBpbiBpbm5lci5uYW1lZF9tb2R1bGVzKCk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShtb2QsIExvUkFMaW5lYXIpOgogICAgICAgICAgICBmb3IgYWRhcHRlcl9uYW1lLCBicmFuY2ggaW4gbW9kLmJyYW5jaGVzLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBmb3IgcGFyYW1fbmFtZSwgcGFyYW0gaW4gYnJhbmNoLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICAgICAgICAgICAgICBrZXkgPSBmInttb2RfbmFtZX0uYnJhbmNoZXMue2FkYXB0ZXJfbmFtZX0ue3BhcmFtX25hbWV9IgogICAgICAgICAgICAgICAgICAgIGFkYXB0ZXJfc3RhdGVba2V5XSA9IHBhcmFtLmRhdGEuY2xvbmUoKQogICAgdG9yY2guc2F2ZSgKICAgICAgICB7CiAgICAgICAgICAgICJnYXRlIjogZ2F0ZV9uZXQuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAiYW5hbHl6ZXIiOiBhbmFseXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICJhZGFwdGVyX3N0YXRlIjogYWRhcHRlcl9zdGF0ZSwKICAgICAgICB9LAogICAgICAgIHBhdGgsCiAgICApCiAgICBsb2cuaW5mbygiU2F2ZWQ6ICVzICAoJWQgYWRhcHRlciB0ZW5zb3JzKSIsIHBhdGgsIGxlbihhZGFwdGVyX3N0YXRlKSkKCgpkZWYgdHJhaW5fam9pbnQoYXJnczogYXJncGFyc2UuTmFtZXNwYWNlKSAtPiBOb25lOgogICAgX3NlZWRfZXZlcnl0aGluZyhnZXRhdHRyKGFyZ3MsICJzZWVkIiwgNDIpKQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKGdldGF0dHIoYXJncywgImRldmljZSIsICJjcHUiKSkKICAgIF93YW50X2FtcCA9IGdldGF0dHIoYXJncywgImJmMTYiLCBUcnVlKSBhbmQgZGV2aWNlLnR5cGUgPT0gImN1ZGEiCiAgICB1c2VfYmYxNiA9IF93YW50X2FtcCBhbmQgdG9yY2guY3VkYS5pc19iZjE2X3N1cHBvcnRlZCgpCiAgICBfYW1wX2R0eXBlID0gdG9yY2guYmZsb2F0MTYgaWYgdXNlX2JmMTYgZWxzZSAodG9yY2guZmxvYXQxNiBpZiBfd2FudF9hbXAgZWxzZSBOb25lKQogICAgbG9nLmluZm8oIlByZWNpc2lvbjogJXMiLCBmIkFNUCB7X2FtcF9kdHlwZX0iIGlmIF9hbXBfZHR5cGUgZWxzZSAiRlAzMiIpCgogICAgb3V0X2RpciA9IFBhdGgoYXJncy5vdXRwdXRfZGlyKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyBMb2FkIGZyb3plbiBTUi1Db3JyTmV0ICsgYXR0YWNoIExvUkEgYnJhbmNoZXMKICAgIHNzX21vZGVsID0gX2xvYWRfbW9kZWwoCiAgICAgICAgZ2V0YXR0cihhcmdzLCAiaGZfbW9kZWwiLCAic2hpbnVoL3NyLWNvcnJuZXQtc3MtMWNoLXdzai12YXItMi01c3BrIiksIGRldmljZQogICAgKQogICAgaW5uZXIgPSBfZ2V0X2lubmVyX21vZHVsZShzc19tb2RlbCkKICAgIGxpYiA9IExvUkFMaWJyYXJ5KGlubmVyKQogICAgbGliLmZyZWV6ZV9iYXNlKCkKICAgIF9sb2FkX2FkYXB0ZXJzKGlubmVyLCBsaWIsIGFyZ3MpCiAgICBpbm5lci50byhkZXZpY2UpCiAgICBlbmdpbmUgPSBnZXRhdHRyKHNzX21vZGVsLCAiZW5naW5lIiwgTm9uZSkKICAgIGlmIGVuZ2luZSBpcyBub3QgTm9uZToKICAgICAgICBmb3IgX2F0dHIgaW4gKCJzdGZ0IiwgImlzdGZ0Iik6CiAgICAgICAgICAgIF9tb2QgPSBnZXRhdHRyKGVuZ2luZSwgX2F0dHIsIE5vbmUpCiAgICAgICAgICAgIGlmIF9tb2QgaXMgbm90IE5vbmUgYW5kIGhhc2F0dHIoX21vZCwgInRvIik6CiAgICAgICAgICAgICAgICBfbW9kLnRvKGRldmljZSkKCiAgICAjIEdhdGUgKyBBbmFseXplciDigJQgd2FybS1zdGFydCBmcm9tIFN0YWdlIDMKICAgIGFuYWx5emVyID0gTGV2ZWwyQW5hbHl6ZXIoKS50byhkZXZpY2UpCiAgICBnYXRlX25ldCA9IEdhdGVOZXR3b3JrKCkudG8oZGV2aWNlKQogICAgZ2F0ZV9ja3B0X3BhdGggPSBQYXRoKGdldGF0dHIoYXJncywgImdhdGVfY2hlY2twb2ludCIsICIiKSkKICAgIGlmIGdhdGVfY2twdF9wYXRoLmV4aXN0cygpOgogICAgICAgIF9sb2FkX3N0YWdlM19ja3B0KGdhdGVfY2twdF9wYXRoLCBhbmFseXplciwgZ2F0ZV9uZXQsIGRldmljZSkKICAgIGVsc2U6CiAgICAgICAgbG9nLndhcm5pbmcoIlN0YWdlIDMgY2hlY2twb2ludCBub3QgZm91bmQgYXQgJXM7IHN0YXJ0aW5nIGdhdGUgZnJvbSBzY3JhdGNoIiwgZ2F0ZV9ja3B0X3BhdGgpCgogICAgIyBUd28gcGFyYW0gZ3JvdXBzOiBhZGFwdGVycyBnZXQgdmVyeSBsb3cgTFIgKGZpbmUtZ3JhaW5lZCBjb3JyZWN0aW9uKQogICAgYWRhcHRlcl9wYXJhbXM6IGxpc3RbdG9yY2gubm4uUGFyYW1ldGVyXSA9IFtdCiAgICBmb3IgbmFtZSBpbiBBREFQVEVSX05BTUVTOgogICAgICAgIGFkYXB0ZXJfcGFyYW1zICs9IGxpYi5hZGFwdGVyX3BhcmFtZXRlcnMobmFtZSkKCiAgICBscl9hZGFwdGVyID0gZ2V0YXR0cihhcmdzLCAibHJfYWRhcHRlciIsIDFlLTUpCiAgICBscl9nYXRlID0gZ2V0YXR0cihhcmdzLCAibHJfZ2F0ZSIsIDJlLTUpCiAgICAjIFN1cHBvcnQgbGVnYWN5IC0tc3RhZ2UxLWxyIC8gLS1sciBhbGlhc2VzIGZyb20gb2xkZXIgbm90ZWJvb2tzCiAgICBpZiBnZXRhdHRyKGFyZ3MsICJsciIsIDAuMCkgPiAwOgogICAgICAgIGxyX2dhdGUgPSBhcmdzLmxyCiAgICAgICAgbHJfYWRhcHRlciA9IGFyZ3MubHIgLyAyLjAKICAgIGVsaWYgZ2V0YXR0cihhcmdzLCAic3RhZ2UxX2xyIiwgMC4wKSA+IDA6CiAgICAgICAgbHJfZ2F0ZSA9IGFyZ3Muc3RhZ2UxX2xyIC8gMTAuMAogICAgICAgIGxyX2FkYXB0ZXIgPSBscl9nYXRlIC8gMi4wCgogICAgb3B0aW1pemVyID0gb3B0aW0uQWRhbVcoCiAgICAgICAgWwogICAgICAgICAgICB7InBhcmFtcyI6IGFkYXB0ZXJfcGFyYW1zLCAibHIiOiBscl9hZGFwdGVyLCAid2VpZ2h0X2RlY2F5IjogMWUtNH0sCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJwYXJhbXMiOiBsaXN0KGFuYWx5emVyLnBhcmFtZXRlcnMoKSkgKyBsaXN0KGdhdGVfbmV0LnBhcmFtZXRlcnMoKSksCiAgICAgICAgICAgICAgICAibHIiOiBscl9nYXRlLAogICAgICAgICAgICAgICAgIndlaWdodF9kZWNheSI6IDFlLTUsCiAgICAgICAgICAgIH0sCiAgICAgICAgXQogICAgKQogICAgZXBvY2hzID0gZ2V0YXR0cihhcmdzLCAiZXBvY2hzIiwgMjApCiAgICBzY2hlZHVsZXIgPSBvcHRpbS5scl9zY2hlZHVsZXIuQ29zaW5lQW5uZWFsaW5nTFIob3B0aW1pemVyLCBUX21heD1lcG9jaHMpCiAgICBsb2FkZXIgPSBfYnVpbGRfZ2F0ZV9kYXRhc2V0KGFyZ3MpCiAgICBiZXN0X2xvc3MgPSBmbG9hdCgiaW5mIikKCiAgICBsb2cuaW5mbygKICAgICAgICAiU3RhZ2UgNCB8IGVwb2Nocz0lZCAgc2FtcGxlcy9lcG9jaD0lZCAgbHJfYWRhcHRlcj0lLjJlICBscl9nYXRlPSUuMmUiLAogICAgICAgIGVwb2NocywgZ2V0YXR0cihhcmdzLCAic2FtcGxlc19wZXJfZXBvY2giLCAxMDAwKSwgbHJfYWRhcHRlciwgbHJfZ2F0ZSwKICAgICkKCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoMSwgZXBvY2hzICsgMSk6CiAgICAgICAgIyBldmFsKCk6IGtlZXBzIGJhc2UgQk4vZHJvcG91dCBkZXRlcm1pbmlzdGljLiBMb1JBIHBhcmFtcyBzdGlsbCByZWNlaXZlIGdyYWQKICAgICAgICAjIGJlY2F1c2UgcmVxdWlyZXNfZ3JhZD1UcnVlIGlzIGluZGVwZW5kZW50IG9mIHRyYWluL2V2YWwgbW9kZS4KICAgICAgICBpbm5lci5ldmFsKCkKICAgICAgICBhbmFseXplci50cmFpbigpCiAgICAgICAgZ2F0ZV9uZXQudHJhaW4oKQogICAgICAgIGVwb2NoX2xvc3MgPSAwLjAKICAgICAgICB0MCA9IHRpbWUudGltZSgpCgogICAgICAgIGZvciBiYXRjaCBpbiBsb2FkZXI6CiAgICAgICAgICAgIG1peHR1cmUgPSBiYXRjaFsibWl4dHVyZSJdLnRvKGRldmljZSkKICAgICAgICAgICAgcmVmZXJlbmNlcyA9IGJhdGNoWyJyZWZlcmVuY2VzIl0udG8oZGV2aWNlKQogICAgICAgICAgICBuX3Nwa3MgPSBiYXRjaFsibl9zcGVha2VycyJdCiAgICAgICAgICAgIHJlY2lwZXMgPSBiYXRjaFsicmVjaXBlIl0KICAgICAgICAgICAgQiA9IG1peHR1cmUuc2hhcGVbMF0KCiAgICAgICAgICAgICMgUGVyLXNhbXBsZSBiYWNrd2FyZDogemVybyBvbmNlLCBiYWNrd2FyZCBwZXIgc2FtcGxlLCBzdGVwIG9uY2UuCiAgICAgICAgICAgICMgVGhpcyBrZWVwcyBwZWFrIG1lbW9yeSBhdCAxIGdyYXBoIGF0IGEgdGltZSBpbnN0ZWFkIG9mIEIgZ3JhcGhzLgogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAgICAgYmF0Y2hfbG9zcyA9IDAuMAogICAgICAgICAgICBlMF9saXN0OiBsaXN0W3RvcmNoLlRlbnNvcl0gPSBbXSAgIyBmbG9hdDMyLCBmb3IgbGV2ZWwtMiBjYXVzYWwgaW5pdAoKICAgICAgICAgICAgZm9yIGIgaW4gcmFuZ2UoQik6CiAgICAgICAgICAgICAgICB3YXYgPSBtaXh0dXJlW2JdLnVuc3F1ZWV6ZSgwKQogICAgICAgICAgICAgICAgIyBDbGlwIGF1ZGlvIOKAlCBncmFkaWVudCB0YXBlIGZvciBTUi1Db3JyTmV0IGlzIGxhcmdlOyAyIHMgZml0cyAxNiBHQiBHUFUKICAgICAgICAgICAgICAgIF9NQVhfU0FNUExFUyA9IGdldGF0dHIoYXJncywgIm1heF9hdWRpb19zYW1wbGVzIiwgODAwMCkKICAgICAgICAgICAgICAgIGlmIHdhdi5zaGFwZVstMV0gPiBfTUFYX1NBTVBMRVM6CiAgICAgICAgICAgICAgICAgICAgd2F2ID0gd2F2Wy4uLiwgOl9NQVhfU0FNUExFU10KCiAgICAgICAgICAgICAgICBsMV9mZWF0ID0gbGV2ZWwxX3RlbnNvcih3YXYuc3F1ZWV6ZSgwKSkudG8oZGV2aWNlKQoKICAgICAgICAgICAgICAgICMgSG9vazogY2FwdHVyZSBFKDApIGFzIGZsb2F0MzIgcmVnYXJkbGVzcyBvZiBhdXRvY2FzdCBjb250ZXh0CiAgICAgICAgICAgICAgICBlMF9jYXB0dXJlOiBkaWN0ID0ge30KCiAgICAgICAgICAgICAgICBkZWYgX2UwX2hvb2soCiAgICAgICAgICAgICAgICAgICAgbTogb2JqZWN0LCBpbnA6IG9iamVjdCwgb3V0OiBvYmplY3QsIF9jYXA6IGRpY3QgPSBlMF9jYXB0dXJlCiAgICAgICAgICAgICAgICApIC0+IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgdCA9IG91dCBpZiBpc2luc3RhbmNlKG91dCwgdG9yY2guVGVuc29yKSBlbHNlIG91dFswXQogICAgICAgICAgICAgICAgICAgIF9jYXBbImUwIl0gPSB0LmRldGFjaCgpLmZsb2F0KCkKCiAgICAgICAgICAgICAgICBob29rX2hhbmRsZSA9IE5vbmUKICAgICAgICAgICAgICAgIGlmIGhhc2F0dHIoaW5uZXIsICJlbmNvZGVyIik6CiAgICAgICAgICAgICAgICAgICAgaG9va19oYW5kbGUgPSBpbm5lci5lbmNvZGVyLnJlZ2lzdGVyX2ZvcndhcmRfaG9vayhfZTBfaG9vaykKCiAgICAgICAgICAgICAgICAjIExldmVsLTIgY2F1c2FsIGluaXQgZnJvbSBwcmV2aW91cyBzYW1wbGUncyBlMAogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICAgICAgICAgaWYgZTBfbGlzdDoKICAgICAgICAgICAgICAgICAgICAgICAgcHJldiA9IGUwX2xpc3RbLTFdCiAgICAgICAgICAgICAgICAgICAgICAgIGwyX2luaXQgPSBhbmFseXplci5mZWF0dXJlX3ZlY3RvcigKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByZXYudW5zcXVlZXplKDApIGlmIHByZXYubmRpbSA9PSAzIGVsc2UgcHJldgogICAgICAgICAgICAgICAgICAgICAgICApLnNxdWVlemUoMCkKICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICBsMl9pbml0ID0gdG9yY2guemVyb3MoNiwgZGV2aWNlPWRldmljZSkKCiAgICAgICAgICAgICAgICBjb25kX2IgPSB0b3JjaC5jYXQoCiAgICAgICAgICAgICAgICAgICAgW2wxX2ZlYXQuZmxvYXQoKSwgbDJfaW5pdC5mbG9hdCgpXSwgZGltPS0xCiAgICAgICAgICAgICAgICApLnVuc3F1ZWV6ZSgwKSAgIyAoMSwgMTApCgogICAgICAgICAgICAgICAgIyBHYXRlICsgbW9kZWwgZm9yd2FyZCBpbnNpZGUgYXV0b2Nhc3QgKGJmMTYgYWN0aXZhdGlvbnMg4oaSIGhhbGYgbWVtb3J5KQogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgKICAgICAgICAgICAgICAgICAgICAiY3VkYSIsIGR0eXBlPV9hbXBfZHR5cGUgb3IgdG9yY2guZmxvYXQzMiwgZW5hYmxlZD1fYW1wX2R0eXBlIGlzIG5vdCBOb25lCiAgICAgICAgICAgICAgICApOgogICAgICAgICAgICAgICAgICAgIGdhdGVfYiA9IGdhdGVfbmV0KGNvbmRfYikuc3F1ZWV6ZSgwKSAgIyAoMywpIGJmMTYgd2l0aCBncmFkCiAgICAgICAgICAgICAgICAgICAgbGliLnNldF9nYXRlcyh7QURBUFRFUl9OQU1FU1tpXTogZ2F0ZV9iW2ldIGZvciBpIGluIHJhbmdlKDMpfSkKICAgICAgICAgICAgICAgICAgICBsaWIuaW5qZWN0X2dhdGVzKCkKICAgICAgICAgICAgICAgICAgICB3YXZlc19zZXAsIGxnX3NlcCA9IF9mb3J3YXJkX3dpdGhfZ3JhZCgKICAgICAgICAgICAgICAgICAgICAgICAgc3NfbW9kZWwsIHdhdiwgbl9zcGtzPXRvcmNoLnRlbnNvcihuX3Nwa3NbYl0pCiAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgICAgIGlmIGhvb2tfaGFuZGxlOgogICAgICAgICAgICAgICAgICAgIGhvb2tfaGFuZGxlLnJlbW92ZSgpCgogICAgICAgICAgICAgICAgZTAgPSBlMF9jYXB0dXJlLmdldCgiZTAiKQogICAgICAgICAgICAgICAgaWYgZTAgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgZTBfbGlzdC5hcHBlbmQoZTApCgogICAgICAgICAgICAgICAgIyAtLS0gUGVyLXNhbXBsZSBsb3NzZXMgKGZwMzIgYWZ0ZXIgZXhwbGljaXQgY2FzdHMpIC0tLQogICAgICAgICAgICAgICAgd2F2ZXNfYiA9IHdhdmVzX3NlcC51bnNxdWVlemUoMCkgICMgKDEsIEssIFQpCiAgICAgICAgICAgICAgICByZWZfYiA9IHJlZmVyZW5jZXNbYiA6IGIgKyAxXSAgIyAoMSwgS19yZWYsIFQpIOKAlCBjYWxtc2VwX2xvc3MgdHJpbXMKICAgICAgICAgICAgICAgIGxnX2IgPSBsZ19zZXAuZmxvYXQoKS51bnNxdWVlemUoMCkgaWYgbGdfc2VwIGlzIG5vdCBOb25lIGVsc2UgTm9uZQogICAgICAgICAgICAgICAgc2VwX2xvc3NfYiA9IGNhbG1zZXBfbG9zcyh3YXZlc19iLCByZWZfYiwgbGdfYiwgbl9zcGtzW2IgOiBiICsgMV0pWyJ0b3RhbCJdCgogICAgICAgICAgICAgICAgZ2F0ZV9iX2YgPSBnYXRlX2IuZmxvYXQoKSAgIyBiZjE2IOKGkiBmcDMyOyBncmFkIHN0aWxsIGZsb3dzCiAgICAgICAgICAgICAgICBvcmFjbGVfYiA9IG9yYWNsZV9nYXRlKFtyZWNpcGVzW2JdXSwgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgICAgIGJjZV9iID0gRi5iaW5hcnlfY3Jvc3NfZW50cm9weSgKICAgICAgICAgICAgICAgICAgICBnYXRlX2JfZi51bnNxdWVlemUoMCkgLyBnYXRlX25ldC5nYXRlX3NjYWxlLCBvcmFjbGVfYgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgbDFfYiA9IF9MMV9MQU1CREEgKiBnYXRlX2JfZi5hYnMoKS5tZWFuKCkKCiAgICAgICAgICAgICAgICBsMl9iID0gdG9yY2gudGVuc29yKDAuMCwgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgICAgIGlmIGUwIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoCiAgICAgICAgICAgICAgICAgICAgICAgICJjdWRhIiwgZHR5cGU9X2FtcF9kdHlwZSBvciB0b3JjaC5mbG9hdDMyLCBlbmFibGVkPV9hbXBfZHR5cGUgaXMgbm90IE5vbmUKICAgICAgICAgICAgICAgICAgICApOgogICAgICAgICAgICAgICAgICAgICAgICBlMF9pbiA9IGUwLnVuc3F1ZWV6ZSgwKSBpZiBlMC5uZGltID09IDMgZWxzZSBlMAogICAgICAgICAgICAgICAgICAgICAgICBsMl9iID0gbGV2ZWwyX2xvc3MoYW5hbHl6ZXIsIGUwX2luLCBbcmVjaXBlc1tiXV0pCgogICAgICAgICAgICAgICAgIyBOb3JtYWxpc2UgYnkgQiBzbyBlZmZlY3RpdmUgc3RlcCBlcXVhbHMgYSBmdWxsLWJhdGNoIGJhY2t3YXJkCiAgICAgICAgICAgICAgICBsb3NzX2IgPSAoc2VwX2xvc3NfYiArIGJjZV9iICsgbDFfYiArIGwyX2IpIC8gQgogICAgICAgICAgICAgICAgbG9zc19iLmJhY2t3YXJkKCkgICMgZnJlZXMgdGhpcyBzYW1wbGUncyBncmFwaCBpbW1lZGlhdGVseQogICAgICAgICAgICAgICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgICAgICAgICAgICAgYmF0Y2hfbG9zcyArPSBsb3NzX2IuaXRlbSgpCgogICAgICAgICAgICAjIENsZWFyIHN0YWxlIGluamVjdGVkIGdhdGVzCiAgICAgICAgICAgIGxpYi5zZXRfZ2F0ZXMoe246IDAuMCBmb3IgbiBpbiBBREFQVEVSX05BTUVTfSkKICAgICAgICAgICAgbGliLmluamVjdF9nYXRlcygpCgogICAgICAgICAgICAjIE8tTG9SQSBwZW5hbHR5IOKAlCBtb2RlbCB3ZWlnaHRzIGFyZSBmcDMyLCBjb21wdXRlZCBvbmNlIHBlciBiYXRjaAogICAgICAgICAgICBvbG8gPSBvbG9yYV9wZW5hbHR5KGlubmVyLCBhbHBoYT1fT0xPUkFfQUxQSEEpCiAgICAgICAgICAgIG9sby5iYWNrd2FyZCgpCiAgICAgICAgICAgIGJhdGNoX2xvc3MgKz0gb2xvLml0ZW0oKQoKICAgICAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKAogICAgICAgICAgICAgICAgYWRhcHRlcl9wYXJhbXMgKyBsaXN0KGFuYWx5emVyLnBhcmFtZXRlcnMoKSkgKyBsaXN0KGdhdGVfbmV0LnBhcmFtZXRlcnMoKSksCiAgICAgICAgICAgICAgICA1LjAsCiAgICAgICAgICAgICkKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgICAgICBlcG9jaF9sb3NzICs9IGJhdGNoX2xvc3MKCiAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQogICAgICAgIGF2ZyA9IGVwb2NoX2xvc3MgLyBtYXgobGVuKGxvYWRlciksIDEpCiAgICAgICAgbG9nLmluZm8oIkVwb2NoICVkLyVkICBsb3NzPSUuNGYgIHRpbWU9JS4xZnMiLCBlcG9jaCwgZXBvY2hzLCBhdmcsIHRpbWUudGltZSgpIC0gdDApCgogICAgICAgIGlmIGF2ZyA8IGJlc3RfbG9zczoKICAgICAgICAgICAgYmVzdF9sb3NzID0gYXZnCiAgICAgICAgICAgIF9zYXZlX2pvaW50X2NrcHQob3V0X2RpciAvICJiZXN0X2pvaW50LnB0IiwgZ2F0ZV9uZXQsIGFuYWx5emVyLCBpbm5lcikKCiAgICBfc2F2ZV9qb2ludF9ja3B0KG91dF9kaXIgLyAiZmluYWxfam9pbnQucHQiLCBnYXRlX25ldCwgYW5hbHl6ZXIsIGlubmVyKQogICAgbG9nLmluZm8oIkpvaW50IHRyYWluaW5nIGRvbmUuIEJlc3QgbG9zczogJS40ZiIsIGJlc3RfbG9zcykKCgpkZWYgX3BhcnNlX2FyZ3MoKSAtPiBhcmdwYXJzZS5OYW1lc3BhY2U6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbGlicmlzcGVlY2gtOGsiLCBkZWZhdWx0PSIiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgZGVmYXVsdD0iIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXJpci1iYW5rIiwgZGVmYXVsdD0iZGF0YS9yaXJzL2JhbmsuanNvbiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1ub2lzZS1kaXIiLCBkZWZhdWx0PSIiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tYWRhcHRlci1yZXZlcmIiLCBkZWZhdWx0PSIiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tYWRhcHRlci1ub2lzZSIsIGRlZmF1bHQ9IiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1hZGFwdGVyLWNvZGVjIiwgZGVmYXVsdD0iIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLXN0YWdlMS1kaXIiLCBkZWZhdWx0PSIiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc3RhZ2UzLWRpciIsIGRlZmF1bHQ9IiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1nYXRlLWNoZWNrcG9pbnQiLCBkZWZhdWx0PSIiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIGRlZmF1bHQ9IiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1jaGVja3BvaW50LWRpciIsIGRlZmF1bHQ9IiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1oZi1tb2RlbCIsIGRlZmF1bHQ9InNoaW51aC9zci1jb3JybmV0LXNzLTFjaC13c2otdmFyLTItNXNwayIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PSJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1lcG9jaHMiLCB0eXBlPWludCwgZGVmYXVsdD0yMCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgcC5hZGRfYXJndW1lbnQoIi0tbHItYWRhcHRlciIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MWUtNSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWxyLWdhdGUiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTJlLTUpCiAgICAjIExlZ2FjeSBhbGlhc2VzIGtlcHQgZm9yIGJhY2t3YXJkcy1jb21wYXQgd2l0aCBvbGRlciBub3RlYm9vayB2ZXJzaW9ucwogICAgcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXN0YWdlMS1sciIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc2FtcGxlcy1wZXItZXBvY2giLCB0eXBlPWludCwgZGVmYXVsdD0xMDAwKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbnVtLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD0wKQogICAgcC5hZGRfYXJndW1lbnQoIi0tYmYxNiIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGRlZmF1bHQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLW5vLWJmMTYiLCBkZXN0PSJiZjE2IiwgYWN0aW9uPSJzdG9yZV9mYWxzZSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1tYXgtYXVkaW8tc2FtcGxlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTgwMDApCiAgICBhcmdzID0gcC5wYXJzZV9hcmdzKCkKICAgIGlmIG5vdCBhcmdzLmxpYnJpc3BlZWNoXzhrIGFuZCBhcmdzLmRhdGFfcm9vdDoKICAgICAgICBhcmdzLmxpYnJpc3BlZWNoXzhrID0gYXJncy5kYXRhX3Jvb3QKICAgIGlmIG5vdCBhcmdzLm91dHB1dF9kaXIgYW5kIGFyZ3MuY2hlY2twb2ludF9kaXI6CiAgICAgICAgYXJncy5vdXRwdXRfZGlyID0gYXJncy5jaGVja3BvaW50X2RpcgogICAgaWYgYXJncy5zdGFnZTFfZGlyOgogICAgICAgIHMxID0gUGF0aChhcmdzLnN0YWdlMV9kaXIpCiAgICAgICAgaWYgbm90IGFyZ3MuYWRhcHRlcl9yZXZlcmI6CiAgICAgICAgICAgIGFyZ3MuYWRhcHRlcl9yZXZlcmIgPSBzdHIoczEgLyAiYmVzdF9yZXZlcmIucHQiKQogICAgICAgIGlmIG5vdCBhcmdzLmFkYXB0ZXJfbm9pc2U6CiAgICAgICAgICAgIGFyZ3MuYWRhcHRlcl9ub2lzZSA9IHN0cihzMSAvICJiZXN0X25vaXNlLnB0IikKICAgICAgICBpZiBub3QgYXJncy5hZGFwdGVyX2NvZGVjOgogICAgICAgICAgICBhcmdzLmFkYXB0ZXJfY29kZWMgPSBzdHIoczEgLyAiYmVzdF9jb2RlYy5wdCIpCiAgICBpZiBhcmdzLnN0YWdlM19kaXIgYW5kIG5vdCBhcmdzLmdhdGVfY2hlY2twb2ludDoKICAgICAgICBhcmdzLmdhdGVfY2hlY2twb2ludCA9IHN0cihQYXRoKGFyZ3Muc3RhZ2UzX2RpcikgLyAiYmVzdF9nYXRlLnB0IikKICAgIGlmIG5vdCBhcmdzLmxpYnJpc3BlZWNoXzhrOgogICAgICAgIHAuZXJyb3IoIi0tbGlicmlzcGVlY2gtOGsgb3IgLS1kYXRhLXJvb3QgaXMgcmVxdWlyZWQiKQogICAgaWYgbm90IGFyZ3Mub3V0cHV0X2RpcjoKICAgICAgICBwLmVycm9yKCItLW91dHB1dC1kaXIgb3IgLS1jaGVja3BvaW50LWRpciBpcyByZXF1aXJlZCIpCiAgICByZXR1cm4gYXJncwoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICB0cmFpbl9qb2ludChfcGFyc2VfYXJncygpKQo='))
open(f'{PROJ}/models/__init__.py','wb').write(base64.b64decode('IiIiRXhwZXJ0IG1vZGVscywgY2FzY2FkZSBnYXRlLCBmdXNpb24gaGVhZCAoRGV2IEIpLiIiIgo='))
open(f'{PROJ}/models/lora.py','wb').write(base64.b64decode('...'))
open(f'{PROJ}/models/condition.py','wb').write(base64.b64decode('IiIiClR3by1sZXZlbCBjb25kaXRpb24gYW5hbHl6ZXIgKERldiBCLCBQMi1CMSAvIFAyLUIyKS4KCkxldmVsIDEg4oCUIHJhdyBTVEZUIERTUCBmZWF0dXJlcywgZGV0ZXJtaW5pc3RpYywgbm8gdHJhaW5pbmc6CiAg4oCiIFNOUiBlc3RpbWF0ZSB2aWEgU2lsZXJvVkFEIHZvaWNlZC1mcmFtZSBlbmVyZ3kgcmF0aW8KICDigKIgQ29kZWMgYmFuZHdpZHRoOiBmcmFjdGlvbiBvZiBlbmVyZ3kgaW4gMy00IGtIeiBiYW5kIChkcm9wcyB3aXRoIGNvZGVjIGRhbWFnZSkKICDigKIgVm9pY2VkLWZyYW1lIGRlbnNpdHk6IGZyYWN0aW9uIG9mIGZyYW1lcyBmbGFnZ2VkIGFzIHZvaWNlZCBieSBTaWxlcm9WQUQKICAgIChmYWxsYmFjazogdm9pY2VkLWVuZXJneSBmcmFjdGlvbiB3aGVuIFNpbGVyb1ZBRCBpcyBub3QgaW5zdGFsbGVkKQoKTGV2ZWwgMiDigJQgcG9vbGVkIEUoMCkgaGVhZHMsIHRyYWluZWQgYWxvbmdzaWRlIHRoZSBnYXRlOgogIOKAoiBUNjAgcmV2ZXJiIGhlYWQ6IHByZWRpY3QgbG9nLVQ2MCBmcm9tIHRlbXBvcmFsIG1lYW4gb2YgRSgwKSBvdmVyIHZvaWNlZCBmcmFtZXMKICDigKIgQ291bnQgcHJpb3IgTUxQOiBwcmVkaWN0IE4g4oiIIHsyLDMsNCw1fSBmcm9tIEUoMCkgc3RhdGlzdGljcwoKVGhlIGNvbWJpbmVkIGZlYXR1cmUgdmVjdG9yIGRyaXZlcyB0aGUgZ2F0ZSBuZXR3b3JrIChtb2RlbHMvZ2F0ZS5weSkuIER1cmluZwppbmZlcmVuY2UgYm90aCBsZXZlbHMgcnVuIHNlcXVlbnRpYWxseTsgZHVyaW5nIGdhdGUgdHJhaW5pbmcgdGhlIExldmVsLTEgb3V0cHV0cwphcmUgZml4ZWQgKG5vIGdyYWQpIGFuZCBMZXZlbC0yIGhlYWRzIGFyZSB0cmFpbmVkLgoKRnJlZSBzdXBlcnZpc2lvbiBmcm9tIE1peHR1cmVSZWNpcGUuY29uZGl0aW9uX3ZlY3RvcigpIChCTFVFUFJJTlQgwqc1LjQpOgogIHNucl9kYiwgdDYwX3MsIGNvZGVjX2NsYXNzLCBjb2RlY19iaXRyYXRlX2ticHMsIG5fc3BlYWtlcnMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHdhcm5pbmdzCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENvbnN0YW50cwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKQ0FMTVNFUF9TUiA9IDhfMDAwClNURlRfV0lOID0gMTI4ClNURlRfSE9QID0gNjQKU1RGVF9CSU5TID0gNjUgICMgbl9mZnQvLzIgKyAxCgojIENvZGVjIGJhbmR3aWR0aCBzZW50aW5lbDogZW5lcmd5IGFib3ZlIHRoaXMgZnJhY3Rpb24gb2YgTnlxdWlzdCBkcm9wcyBhdCBsb3cgYml0cmF0ZS4KX0JBTkRXSURUSF9DVVRPRkZfSFogPSAzXzIwMApfQkFORFdJRFRIX0JJTiA9IGludChyb3VuZChfQkFORFdJRFRIX0NVVE9GRl9IWiAvIChDQUxNU0VQX1NSIC8gMikgKiAoU1RGVF9CSU5TIC0gMSkpKQoKX0VQUyA9IDFlLTEwCl9MT0dfVDYwX01JTiA9IGZsb2F0KG5wLmxvZygwLjA1KSkKX0xPR19UNjBfTUFYID0gZmxvYXQobnAubG9nKDIuMCkpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBTaWxlcm9WQUQgd3JhcHBlciAob3B0aW9uYWwpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpfc2lsZXJvX21vZGVsOiBvYmplY3QgfCBOb25lID0gTm9uZQpfc2lsZXJvX2F2YWlsYWJsZTogYm9vbCB8IE5vbmUgPSBOb25lCgoKZGVmIF90cnlfbG9hZF9zaWxlcm8oKSAtPiBib29sOgogICAgZ2xvYmFsIF9zaWxlcm9fbW9kZWwsIF9zaWxlcm9fYXZhaWxhYmxlCiAgICBpZiBfc2lsZXJvX2F2YWlsYWJsZSBpcyBub3QgTm9uZToKICAgICAgICByZXR1cm4gX3NpbGVyb19hdmFpbGFibGUKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBtb2RlbCwgdXRpbHMgPSB0b3JjaC5odWIubG9hZCgKICAgICAgICAgICAgcmVwb19vcl9kaXI9InNuYWtlcnM0L3NpbGVyby12YWQiLAogICAgICAgICAgICBtb2RlbD0ic2lsZXJvX3ZhZCIsCiAgICAgICAgICAgIGZvcmNlX3JlbG9hZD1GYWxzZSwKICAgICAgICAgICAgdHJ1c3RfcmVwbz1UcnVlLAogICAgICAgICAgICB2ZXJib3NlPUZhbHNlLAogICAgICAgICkKICAgICAgICBfc2lsZXJvX21vZGVsID0gbW9kZWwuY3B1KCkgICMga2VlcCBWQUQgb24gQ1BVOyBTUi1Db3JyTmV0IG5lZWRzIEdQVSBWUkFNCiAgICAgICAgX3NpbGVyb19hdmFpbGFibGUgPSBUcnVlCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIF9zaWxlcm9fYXZhaWxhYmxlID0gRmFsc2UKICAgIHJldHVybiBfc2lsZXJvX2F2YWlsYWJsZSAgIyB0eXBlOiBpZ25vcmVbcmV0dXJuLXZhbHVlXQoKCmRlZiB2b2ljZWRfZGVuc2l0eV9zaWxlcm8od2F2ZWZvcm06IHRvcmNoLlRlbnNvciwgc3I6IGludCA9IENBTE1TRVBfU1IpIC0+IGZsb2F0OgogICAgIiIiCiAgICBGcmFjdGlvbiBvZiAzMCBtcyBmcmFtZXMgU2lsZXJvVkFEIGNsYXNzaWZpZXMgYXMgdm9pY2VkLgoKICAgIFJldHVybnMgZmxvYXQgaW4gWzAsIDFdLiBGYWxscyBiYWNrIHRvIHZvaWNlZC1lbmVyZ3kgZnJhY3Rpb24gaWYgU2lsZXJvVkFECiAgICBpcyBub3QgaW5zdGFsbGVkLgogICAgIiIiCiAgICBpZiBub3QgX3RyeV9sb2FkX3NpbGVybygpOgogICAgICAgIHJldHVybiB2b2ljZWRfZGVuc2l0eV9lbmVyZ3kod2F2ZWZvcm0pCiAgICB0cnk6CiAgICAgICAgYXNzZXJ0IF9zaWxlcm9fbW9kZWwgaXMgbm90IE5vbmUKICAgICAgICB3YXYgPSB3YXZlZm9ybS5mbG9hdCgpLnNxdWVlemUoKQogICAgICAgIGlmIHdhdi5uZGltICE9IDE6CiAgICAgICAgICAgIHdhdiA9IHdhdi5tZWFuKDApCiAgICAgICAgaWYgc3IgIT0gMTZfMDAwIGFuZCBzciAhPSA4XzAwMDoKICAgICAgICAgICAgd2FybmluZ3Mud2FybihmIlNpbGVyb1ZBRCBwcmVmZXJzIDhrSHogb3IgMTZrSHo7IGdvdCB7c3J9IEh6IiwgUnVudGltZVdhcm5pbmcpCiAgICAgICAgIyBTaWxlcm8gcmV0dXJucyBwcm9iYWJpbGl0aWVzIHBlciAzMG1zIHdpbmRvdwogICAgICAgIGZyYW1lX2xlbiA9IGludCgwLjAzMCAqIHNyKQogICAgICAgIGhvcCA9IGZyYW1lX2xlbiAgIyBub24tb3ZlcmxhcHBpbmcKICAgICAgICBuX2ZyYW1lcyA9IHdhdi5zaGFwZVswXSAvLyBob3AKICAgICAgICB2b2ljZWQgPSAwCiAgICAgICAgZm9yIGkgaW4gcmFuZ2Uobl9mcmFtZXMpOgogICAgICAgICAgICBzZWcgPSB3YXZbaSAqIGhvcCA6IChpICsgMSkgKiBob3BdLnVuc3F1ZWV6ZSgwKQogICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgIHByb2IgPSBmbG9hdChfc2lsZXJvX21vZGVsKHNlZy5jcHUoKSwgc3IpLml0ZW0oKSkgICMgdHlwZTogaWdub3JlW29wZXJhdG9yXQogICAgICAgICAgICB2b2ljZWQgKz0gaW50KHByb2IgPiAwLjUpCiAgICAgICAgcmV0dXJuIHZvaWNlZCAvIG1heChuX2ZyYW1lcywgMSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIHZvaWNlZF9kZW5zaXR5X2VuZXJneSh3YXZlZm9ybSkKCgpkZWYgdm9pY2VkX2RlbnNpdHlfZW5lcmd5KHdhdmVmb3JtOiB0b3JjaC5UZW5zb3IpIC0+IGZsb2F0OgogICAgIiIiCiAgICBWb2ljZWQtZW5lcmd5IGZyYWN0aW9uOiBmcmFjdGlvbiBvZiAzMG1zIGZyYW1lcyB3aG9zZSBlbmVyZ3kgZXhjZWVkcwogICAgdGhlIG1lZGlhbiBmcmFtZSBlbmVyZ3kuIEZhbGxiYWNrIGZvciB3aGVuIFNpbGVyb1ZBRCBpcyB1bmF2YWlsYWJsZS4KICAgICIiIgogICAgd2F2ID0gd2F2ZWZvcm0uZmxvYXQoKS5zcXVlZXplKCkKICAgIGZyYW1lX2xlbiA9IGludCgwLjAzMCAqIENBTE1TRVBfU1IpCiAgICBuX2ZyYW1lcyA9IHdhdi5zaGFwZVswXSAvLyBmcmFtZV9sZW4KICAgIGlmIG5fZnJhbWVzID09IDA6CiAgICAgICAgcmV0dXJuIDAuNQogICAgZnJhbWVzID0gd2F2Wzogbl9mcmFtZXMgKiBmcmFtZV9sZW5dLnJlc2hhcGUobl9mcmFtZXMsIGZyYW1lX2xlbikKICAgIGVuZXJnaWVzID0gZnJhbWVzLnBvdygyKS5tZWFuKGRpbT0xKQogICAgbWVkaWFuID0gZW5lcmdpZXMubWVkaWFuKCkKICAgIHJldHVybiBmbG9hdCgoZW5lcmdpZXMgPiBtZWRpYW4pLmZsb2F0KCkubWVhbigpLml0ZW0oKSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIExldmVsIDE6IERTUCBmZWF0dXJlcyAobm8gcGFyYW1ldGVycywgZGV0ZXJtaW5pc3RpYykKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpkZWYgbGV2ZWwxX2ZlYXR1cmVzKG1peHR1cmVfOGs6IHRvcmNoLlRlbnNvcikgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICIiIgogICAgRXh0cmFjdCByYXcgRFNQIGZlYXR1cmVzIGZyb20gYW4gOCBrSHogbWl4dHVyZSB3YXZlZm9ybS4KCiAgICBBcmdzOgogICAgICAgIG1peHR1cmVfOGs6IFtUXSBvciBbMSwgVF0gZmxvYXQzMiBtb25vIHdhdmVmb3JtIGF0IDgga0h6LgoKICAgIFJldHVybnM6CiAgICAgICAgRGljdCB3aXRoIGtleXM6IHNucl9lc3RfZGIsIGNvZGVjX2J3X3JhdGlvLCB2b2ljZWRfZGVuc2l0eSwgdG90YWxfZW5lcmd5X2RiLgogICAgICAgIEFsbCB2YWx1ZXMgYXJlIHNjYWxhciBmbG9hdHMgdXNhYmxlIGFzIGdhdGUgaW5wdXRzLgogICAgIiIiCiAgICB3YXYgPSBtaXh0dXJlXzhrLmZsb2F0KCkuc3F1ZWV6ZSgpCiAgICBpZiB3YXYubmRpbSAhPSAxOgogICAgICAgIHdhdiA9IHdhdi5tZWFuKDApCgogICAgIyBTVEZUIGF0IDgga0h6IHdpdGggU1ItQ29yck5ldCB3aW5kb3cvaG9wCiAgICBzcGVjID0gdG9yY2guc3RmdCgKICAgICAgICB3YXYsCiAgICAgICAgbl9mZnQ9U1RGVF9XSU4sCiAgICAgICAgaG9wX2xlbmd0aD1TVEZUX0hPUCwKICAgICAgICB3aW5fbGVuZ3RoPVNURlRfV0lOLAogICAgICAgIHdpbmRvdz10b3JjaC5oYW5uX3dpbmRvdyhTVEZUX1dJTiwgZGV2aWNlPXdhdi5kZXZpY2UpLAogICAgICAgIHJldHVybl9jb21wbGV4PVRydWUsCiAgICAgICAgY2VudGVyPVRydWUsCiAgICApICAjIFtGLCBUX2ZyYW1lc10sIEY9NjUKCiAgICBwb3dlciA9IHNwZWMuYWJzKCkucG93KDIpICAjIFs2NSwgVF9mcmFtZXNdCiAgICB0b3RhbF9wb3dlciA9IGZsb2F0KHBvd2VyLnN1bSgpLml0ZW0oKSkKCiAgICAjIC0tLSBTTlIgZXN0aW1hdGUgdmlhIHZvaWNlZC91bnZvaWNlZCBlbmVyZ3kgc3BsaXQgLS0tCiAgICB2b2ljZWRfZGVuID0gdm9pY2VkX2RlbnNpdHlfc2lsZXJvKHdhdikKICAgICMgUHJveHkgU05SOiByYXRpbyBvZiBmcmFtZSBlbmVyZ2llcyBhYm92ZS9iZWxvdyB2b2ljZWQgdGhyZXNob2xkCiAgICBmcmFtZV9lbmVyZ2llcyA9IHBvd2VyLnN1bSgwKSAgIyBbVF9mcmFtZXNdCiAgICBpZiBmcmFtZV9lbmVyZ2llcy5udW1lbCgpID4gMToKICAgICAgICB0aHJlc2hvbGQgPSBmcmFtZV9lbmVyZ2llcy5tZWRpYW4oKQogICAgICAgIHZvaWNlZF9lbmVyZ3kgPSBmbG9hdChmcmFtZV9lbmVyZ2llc1tmcmFtZV9lbmVyZ2llcyA+IHRocmVzaG9sZF0ubWVhbigpLml0ZW0oKSkKICAgICAgICB1bnZvaWNlZF9lbmVyZ3kgPSBmbG9hdChmcmFtZV9lbmVyZ2llc1tmcmFtZV9lbmVyZ2llcyA8PSB0aHJlc2hvbGRdLm1lYW4oKS5pdGVtKCkpCiAgICAgICAgc25yX2VzdCA9IDEwLjAgKiBucC5sb2cxMChtYXgodm9pY2VkX2VuZXJneSwgX0VQUykgLyBtYXgodW52b2ljZWRfZW5lcmd5LCBfRVBTKSkKICAgIGVsc2U6CiAgICAgICAgc25yX2VzdCA9IDAuMAoKICAgICMgLS0tIENvZGVjIGJhbmR3aWR0aCByYXRpbyAtLS0KICAgICMgRW5lcmd5IGFib3ZlIF9CQU5EV0lEVEhfQ1VUT0ZGX0haIHJlbGF0aXZlIHRvIGZ1bGwtYmFuZCBlbmVyZ3kuCiAgICAjIFRoaXMgZHJvcHMgd2hlbiBjb2RlYyBkYW1hZ2UgcmVtb3ZlcyBoaWdoLWZyZXF1ZW5jeSBjb250ZW50LgogICAgaGlnaGJhbmRfcG93ZXIgPSBmbG9hdChwb3dlcltfQkFORFdJRFRIX0JJTjosIDpdLnN1bSgpLml0ZW0oKSkKICAgIGJ3X3JhdGlvID0gaGlnaGJhbmRfcG93ZXIgLyBtYXgodG90YWxfcG93ZXIsIF9FUFMpCgogICAgIyAtLS0gVG90YWwgZW5lcmd5IGluIGRCIC0tLQogICAgdG90YWxfZGIgPSAxMC4wICogbnAubG9nMTAobWF4KHRvdGFsX3Bvd2VyIC8gbWF4KHBvd2VyLm51bWVsKCksIDEpLCBfRVBTKSkKCiAgICByZXR1cm4gewogICAgICAgICJzbnJfZXN0X2RiIjogZmxvYXQoc25yX2VzdCksCiAgICAgICAgImNvZGVjX2J3X3JhdGlvIjogZmxvYXQobnAuY2xpcChid19yYXRpbywgMC4wLCAxLjApKSwKICAgICAgICAidm9pY2VkX2RlbnNpdHkiOiBmbG9hdCh2b2ljZWRfZGVuKSwKICAgICAgICAidG90YWxfZW5lcmd5X2RiIjogZmxvYXQodG90YWxfZGIpLAogICAgfQoKCmRlZiBsZXZlbDFfdGVuc29yKG1peHR1cmVfOGs6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiUmV0dXJuIExldmVsLTEgZmVhdHVyZXMgYXMgYSBbNF0gZmxvYXQzMiB0ZW5zb3IgKGZvciBnYXRlIGlucHV0KS4iIiIKICAgIGZlYXRzID0gbGV2ZWwxX2ZlYXR1cmVzKG1peHR1cmVfOGspCiAgICByZXR1cm4gdG9yY2gudGVuc29yKAogICAgICAgIFtmZWF0c1sic25yX2VzdF9kYiJdLCBmZWF0c1siY29kZWNfYndfcmF0aW8iXSwgZmVhdHNbInZvaWNlZF9kZW5zaXR5Il0sIGZlYXRzWyJ0b3RhbF9lbmVyZ3lfZGIiXV0sCiAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQzMiwKICAgICkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIExldmVsIDI6IExlYXJuZWQgaGVhZHMgb24gcG9vbGVkIEUoMCkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpjbGFzcyBUNjBIZWFkKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFByZWRpY3QgbG9nLVQ2MCBmcm9tIHRoZSB0ZW1wb3JhbCBtZWFuIG9mIGVuY29kZXIgb3V0cHV0IEUoMCkuCgogICAgRSgwKSBzaGFwZTogKDEsIFQsIDY1LCAxMjgpIGZyb20gbW9kZWwuZW5jb2RlciBmb3J3YXJkIGhvb2suCiAgICBQb29sZWQgdG8gKDEyOCwpIGJ5IGF2ZXJhZ2luZyBvdmVyIFQgYW5kIEYsIHRoZW4gdGhyb3VnaCBhIDItbGF5ZXIgTUxQLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWw6IGludCA9IDEyOCwgaGlkZGVuOiBpbnQgPSA2NCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5ldCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihkX21vZGVsLCBoaWRkZW4pLAogICAgICAgICAgICBubi5HRUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW4sIDEpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBlMDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgZTA6IChCLCBULCBGLCBEKSBlbmNvZGVyIG91dHB1dC4KICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICAoQiwpIHByZWRpY3RlZCBsb2ctVDYwIGluIHNlY29uZHMuCiAgICAgICAgIiIiCiAgICAgICAgIyBNZWFuIG92ZXIgVCBhbmQgRi4KICAgICAgICBwb29sZWQgPSBlMC5tZWFuKGRpbT0oMSwgMikpICAjIChCLCBEKQogICAgICAgIGxvZ190NjAgPSBzZWxmLm5ldChwb29sZWQpLnNxdWVlemUoLTEpICAjIChCLCkKICAgICAgICByZXR1cm4gbG9nX3Q2MAoKICAgIGRlZiB0NjBfc2Vjb25kcyhzZWxmLCBlMDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiUmV0dXJuIHByZWRpY3RlZCBUNjAgaW4gc2Vjb25kcyAoY2xhbXBlZCB0byBwbGF1c2libGUgcmFuZ2UpLiIiIgogICAgICAgIGxvZ190NjAgPSBzZWxmLmZvcndhcmQoZTApCiAgICAgICAgcmV0dXJuIHRvcmNoLmV4cChsb2dfdDYwLmNsYW1wKF9MT0dfVDYwX01JTiwgX0xPR19UNjBfTUFYKSkKCgpjbGFzcyBDb3VudFByaW9yTUxQKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFNwZWFrZXIgY291bnQgcHJpb3IgZnJvbSBFKDApOiBQKE4gfCBFKDApKSwgTiDiiIggezIsIDMsIDQsIDV9LgoKICAgIFJldHVybnMgbG9naXRzIG92ZXIgNCBjbGFzc2VzOyBtYXBzIHRvIE4gYnkgYXJnbWF4ICsgMi4KICAgICIiIgoKICAgIF9OX0NMQVNTRVMgPSA0ICAjIE4gaW4gezIsMyw0LDV9CgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWw6IGludCA9IDEyOCwgaGlkZGVuOiBpbnQgPSA2NCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5ldCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxpbmVhcihkX21vZGVsLCBoaWRkZW4pLAogICAgICAgICAgICBubi5HRUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbiksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKGhpZGRlbiwgc2VsZi5fTl9DTEFTU0VTKSwKICAgICAgICApCgogICAgZGVmIGZvcndhcmQoc2VsZiwgZTA6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGUwOiAoQiwgVCwgRiwgRCkKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICAoQiwgNCkgbG9naXRzIGZvciBOIOKIiCB7MiwzLDQsNX0KICAgICAgICAiIiIKICAgICAgICBwb29sZWQgPSBlMC5tZWFuKGRpbT0oMSwgMikpICAjIChCLCBEKQogICAgICAgIHJldHVybiBzZWxmLm5ldChwb29sZWQpCgogICAgZGVmIGNvdW50X2VzdGltYXRlKHNlbGYsIGUwOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJSZXR1cm4gKEIsKSBpbnRlZ2VyIGNvdW50IGVzdGltYXRlcyBpbiB7MiwzLDQsNX0uIiIiCiAgICAgICAgbG9naXRzID0gc2VsZi5mb3J3YXJkKGUwKQogICAgICAgIHJldHVybiBsb2dpdHMuYXJnbWF4KGRpbT0tMSkgKyAyCgogICAgZGVmIGNvdW50X3Byb2JzKHNlbGYsIGUwOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJSZXR1cm4gKEIsIDQpIHNvZnRtYXggcHJvYmFiaWxpdGllcyBvdmVyIHsyLDMsNCw1fS4iIiIKICAgICAgICByZXR1cm4gRi5zb2Z0bWF4KHNlbGYuZm9yd2FyZChlMCksIGRpbT0tMSkKCgpjbGFzcyBMZXZlbDJBbmFseXplcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBDb21iaW5lcyBUNjBIZWFkIGFuZCBDb3VudFByaW9yTUxQIGludG8gb25lIG1vZHVsZS4KCiAgICBQcm9kdWNlcyBhIChCLCA2KSBmZWF0dXJlIHZlY3RvcjoKICAgICAgW2xvZ190NjAsIHQ2MF9zZWNvbmRzLCBjb3VudF9sb2dpdF8yLCBjb3VudF9sb2dpdF8zLCBjb3VudF9sb2dpdF80LCBjb3VudF9sb2dpdF81XQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWw6IGludCA9IDEyOCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnQ2MF9oZWFkID0gVDYwSGVhZChkX21vZGVsKQogICAgICAgIHNlbGYuY291bnRfcHJpb3IgPSBDb3VudFByaW9yTUxQKGRfbW9kZWwpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgZTA6IHRvcmNoLlRlbnNvcikgLT4gZGljdFtzdHIsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgZTA6IChCLCBULCBGLCBEKSBlbmNvZGVyIG91dHB1dCBmcm9tIGhvb2suCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgRGljdCB3aXRoIGtleXM6IGxvZ190NjAgKEIsKSwgdDYwX3MgKEIsKSwgY291bnRfbG9naXRzIChCLDQpLCBjb3VudF9wcm9icyAoQiw0KS4KICAgICAgICAiIiIKICAgICAgICBsb2dfdDYwID0gc2VsZi50NjBfaGVhZChlMCkKICAgICAgICB0NjBfcyA9IHRvcmNoLmV4cChsb2dfdDYwLmNsYW1wKF9MT0dfVDYwX01JTiwgX0xPR19UNjBfTUFYKSkKICAgICAgICBjb3VudF9sb2dpdHMgPSBzZWxmLmNvdW50X3ByaW9yKGUwKQogICAgICAgIGNvdW50X3Byb2JzID0gRi5zb2Z0bWF4KGNvdW50X2xvZ2l0cywgZGltPS0xKQogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJsb2dfdDYwIjogbG9nX3Q2MCwKICAgICAgICAgICAgInQ2MF9zIjogdDYwX3MsCiAgICAgICAgICAgICJjb3VudF9sb2dpdHMiOiBjb3VudF9sb2dpdHMsCiAgICAgICAgICAgICJjb3VudF9wcm9icyI6IGNvdW50X3Byb2JzLAogICAgICAgIH0KCiAgICBkZWYgZmVhdHVyZV92ZWN0b3Ioc2VsZiwgZTA6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgICIiIlJldHVybiAoQiwgNikgZmxvYXQgdGVuc29yIGZvciBnYXRlIGlucHV0LiIiIgogICAgICAgIG91dCA9IHNlbGYuZm9yd2FyZChlMCkKICAgICAgICByZXR1cm4gdG9yY2guY2F0KFtvdXRbImxvZ190NjAiXS51bnNxdWVlemUoLTEpLCBvdXRbInQ2MF9zIl0udW5zcXVlZXplKC0xKSwgb3V0WyJjb3VudF9wcm9icyJdXSwgZGltPS0xKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU3VwZXJ2aXNlZCBsb3NzIGZvciBMZXZlbC0yIGhlYWRzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKZGVmIGxldmVsMl9sb3NzKAogICAgYW5hbHl6ZXI6IExldmVsMkFuYWx5emVyLAogICAgZTA6IHRvcmNoLlRlbnNvciwKICAgIHJlY2lwZV92ZWN0b3JzOiBsaXN0W2RpY3Rbc3RyLCBmbG9hdF1dLAopIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIgogICAgU3VwZXJ2aXNlZCBsb3NzIG9uIGJvdGggaGVhZHMgdXNpbmcgZnJlZSBjb25kaXRpb24gbGFiZWxzIGZyb20gdGhlIHJlY2lwZS4KCiAgICB0NjAgaGVhZDogTDEgbG9zcyBvbiBsb2ctVDYwIChyb2J1c3QgdG8gc2NhbGUpLgogICAgY291bnQgcHJpb3I6IGNyb3NzLWVudHJvcHkgb3ZlciB7MiwzLDQsNX0uCgogICAgQXJnczoKICAgICAgICBhbmFseXplcjogTGV2ZWwyQW5hbHl6ZXIgbW9kdWxlLgogICAgICAgIGUwOiAoQiwgVCwgRiwgRCkgZW5jb2RlciBmZWF0dXJlcy4KICAgICAgICByZWNpcGVfdmVjdG9yczogTGlzdCBvZiBCIGNvbmRpdGlvbl92ZWN0b3IoKSBkaWN0cyBmcm9tIE1peHR1cmVSZWNpcGUuCgogICAgUmV0dXJuczoKICAgICAgICBTY2FsYXIgbG9zcyB0ZW5zb3IuCiAgICAiIiIKICAgIG91dCA9IGFuYWx5emVyLmZvcndhcmQoZTApCiAgICBkZXZpY2UgPSBvdXRbImxvZ190NjAiXS5kZXZpY2UKCiAgICB0NjBfdGFyZ2V0cyA9IHRvcmNoLnRlbnNvcigKICAgICAgICBbbnAubG9nKG1heChydlsidDYwX3MiXSwgMC4wNSkpIGZvciBydiBpbiByZWNpcGVfdmVjdG9yc10sCiAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPWRldmljZSwKICAgICkKICAgIGNvdW50X3RhcmdldHMgPSB0b3JjaC50ZW5zb3IoCiAgICAgICAgW2ludChydlsibl9zcGVha2VycyJdKSAtIDIgZm9yIHJ2IGluIHJlY2lwZV92ZWN0b3JzXSwKICAgICAgICBkdHlwZT10b3JjaC5sb25nLCBkZXZpY2U9ZGV2aWNlLAogICAgKQoKICAgIGxvc3NfdDYwID0gRi5sMV9sb3NzKG91dFsibG9nX3Q2MCJdLCB0NjBfdGFyZ2V0cykKICAgIGxvc3NfY291bnQgPSBGLmNyb3NzX2VudHJvcHkob3V0WyJjb3VudF9sb2dpdHMiXSwgY291bnRfdGFyZ2V0cykKICAgIHJldHVybiBsb3NzX3Q2MCArIGxvc3NfY291bnQK'))
open(f'{PROJ}/models/gate.py','wb').write(base64.b64decode('IiIiCkdhdGUgbmV0d29yayBmb3IgQ0FMTS1TZXAgYWRhcHRlciByb3V0aW5nIChEZXYgQiwgUDItQjMpLgoKQXJjaGl0ZWN0dXJlOiBNTFAgd2l0aCB0d28gaGlkZGVuIGxheWVycyBvZiAyNTYsIEdFTFUsIHNpZ21vaWQgc2NhbGVkIHRvIDEuNS4KSW5wdXQ6IGNvbmNhdGVuYXRpb24gb2YgTGV2ZWwtMSBmZWF0dXJlcyAoNC1EKSArIExldmVsLTIgZmVhdHVyZXMgKDYtRCkgPSAxMC1ELgpPdXRwdXQ6IDMtRCBnYXRlIHZlY3Rvciwgb25lIHNjYWxhciBwZXIgYWRhcHRlciAocmV2ZXJiLCBub2lzZSwgY29kZWMpLgoKUmVndWxhcmlzYXRpb246CiAgTDEgc3BhcnNpdHkgKGxhbWJkYT0xZS0zKTogZW5jb3VyYWdlcyB0aGUgZ2F0ZSB0byBzZWxlY3QgZXhhY3RseSB0aGUgYWRhcHRlcnMKICB0aGF0IGFyZSBuZWVkZWQsIG5vdCBoZWRnaW5nIGFjcm9zcyBhbGwgdGhyZWUuCiAgRU1BIHNtb290aGluZyAoYWxwaGE9MC43KTogYXBwbGllZCB0byB0aGUgZ2F0ZSBvdXRwdXQgZHVyaW5nIGluZmVyZW5jZSB0bwogIHByZXZlbnQgcGVyLWNodW5rIGZsaWNrZXJpbmcgb24gbG9uZyByZWNvcmRpbmdzLgoKU3VwZXJ2aXNpb24gKFN0YWdlIDMsIEJMVUVQUklOVCDCpzYuMSk6IHRoZSBnYXRlIGlzIHRyYWluZWQgam9pbnRseSB3aXRoIHRoZQpMZXZlbC0yIGFuYWx5emVyIG9uIHRoZSBTQU1FIGZyZWUgbGFiZWxzIGZyb20gTWl4dHVyZVJlY2lwZS5jb25kaXRpb25fdmVjdG9yKCkuClRoZSAib3JhY2xlIGdhdGUiIGlzIGRlcml2ZWQgZGlyZWN0bHkgZnJvbSB0aGUgcmVjaXBlOiByZXZlcmIgYWRhcHRlciBpcyBvbiBpZmYKdDYwX3MgPiAwLCBub2lzZSBhZGFwdGVyIGlmZiBzbnJfZGIgPCA2MC4wLCBjb2RlYyBhZGFwdGVyIGlmZiBjb2RlY19jbGFzcyA+IDAuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgpmcm9tIG1vZGVscy5sb3JhIGltcG9ydCBBREFQVEVSX05BTUVTCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENvbnN0YW50cwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKX0wxX0xBTUJEQTogZmxvYXQgPSAxZS0zCl9HQVRFX1NDQUxFOiBmbG9hdCA9IDEuNSAgICMgc2lnbW9pZCAqIHNjYWxlLCBvdXRwdXQgcmFuZ2UgWzAsIDEuNV0KX0VNQV9BTFBIQTogZmxvYXQgPSAwLjcgICAgIyBFTUEgc21vb3RoaW5nIGNvZWZmaWNpZW50IGZvciBzdHJlYW1pbmcgaW5mZXJlbmNlCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBHYXRlIE1MUAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmNsYXNzIEdhdGVOZXR3b3JrKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFJvdXRlcyB0aGUgMTAtRCBjb25kaXRpb24gdmVjdG9yIHRvIDMgYWRhcHRlciBnYXRlIHZhbHVlcyBpbiBbMCwgMS41XS4KCiAgICBJbnB1dDogIGNhdChsZXZlbDFfZmVhdHMgWzRdLCBsZXZlbDJfZmVhdHMgWzZdKSA9IFsxMF0KICAgIEhpZGRlbjogTGluZWFyKDEwLCAyNTYpIOKGkiBHRUxVIOKGkiBMaW5lYXIoMjU2LCAyNTYpIOKGkiBHRUxVCiAgICBPdXRwdXQ6IExpbmVhcigyNTYsIDMpIOKGkiBzaWdtb2lkIMOXIDEuNQoKICAgIEF0dHJpYnV0ZXM6CiAgICAgICAgYWRhcHRlcl9uYW1lczogTmFtZXMgb2YgdGhlIDMgYWRhcHRlcnMgaW4gZ2F0ZS12ZWN0b3Igb3JkZXIuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbl9mZWF0dXJlczogaW50ID0gMTAsCiAgICAgICAgaGlkZGVuOiBpbnQgPSAyNTYsCiAgICAgICAgbl9hZGFwdGVyczogaW50ID0gMywKICAgICAgICBnYXRlX3NjYWxlOiBmbG9hdCA9IF9HQVRFX1NDQUxFLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYWRhcHRlcl9uYW1lcyA9IGxpc3QoQURBUFRFUl9OQU1FUykKICAgICAgICBzZWxmLmdhdGVfc2NhbGUgPSBnYXRlX3NjYWxlCgogICAgICAgIHNlbGYubmV0ID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGluX2ZlYXR1cmVzLCBoaWRkZW4pLAogICAgICAgICAgICBubi5HRUxVKCksCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW4sIGhpZGRlbiksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKGhpZGRlbiwgbl9hZGFwdGVycyksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGNvbmRpdGlvbjogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiCiAgICAgICAgQXJnczoKICAgICAgICAgICAgY29uZGl0aW9uOiAoQiwgMTApIGNvbmNhdGVuYXRlZCBMZXZlbC0xICsgTGV2ZWwtMiBmZWF0dXJlcy4KICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICAoQiwgMykgZ2F0ZSB2YWx1ZXMgaW4gWzAsIGdhdGVfc2NhbGVdLgogICAgICAgICIiIgogICAgICAgIHJldHVybiB0b3JjaC5zaWdtb2lkKHNlbGYubmV0KGNvbmRpdGlvbikpICogc2VsZi5nYXRlX3NjYWxlCgogICAgZGVmIGdhdGVfZGljdChzZWxmLCBjb25kaXRpb246IHRvcmNoLlRlbnNvcikgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiIKICAgICAgICBDb21wdXRlIGFuZCByZXR1cm4gYSB7YWRhcHRlcl9uYW1lOiBnYXRlX3ZhbHVlfSBkaWN0IGZvciBhIHNpbmdsZSBzYW1wbGUuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIGNvbmRpdGlvbjogKDEsIDEwKSBvciAoMTAsKSBjb25kaXRpb24gdmVjdG9yLgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIERpY3QgbWFwcGluZyBlYWNoIGFkYXB0ZXIgbmFtZSB0byBpdHMgZ2F0ZSB2YWx1ZS4KICAgICAgICAiIiIKICAgICAgICBnID0gc2VsZi5mb3J3YXJkKGNvbmRpdGlvbi51bnNxdWVlemUoMCkgaWYgY29uZGl0aW9uLm5kaW0gPT0gMSBlbHNlIGNvbmRpdGlvbikKICAgICAgICByZXR1cm4ge25hbWU6IGZsb2F0KGdbMCwgaV0uaXRlbSgpKSBmb3IgaSwgbmFtZSBpbiBlbnVtZXJhdGUoc2VsZi5hZGFwdGVyX25hbWVzKX0KCiAgICBkZWYgbDFfcGVuYWx0eShzZWxmLCBnYXRlczogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgIiIiTDEgc3BhcnNpdHkgcGVuYWx0eSBvbiB0aGUgZ2F0ZSB2ZWN0b3IgKEJMVUVQUklOVCDCpzYuMSkuIiIiCiAgICAgICAgcmV0dXJuIF9MMV9MQU1CREEgKiBnYXRlcy5hYnMoKS5tZWFuKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIE9yYWNsZSBnYXRlIChmb3Igc3VwZXJ2aXNlZCBTdGFnZS0zIHRyYWluaW5nKQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBvcmFjbGVfZ2F0ZShyZWNpcGVfdmVjdG9yczogbGlzdFtkaWN0W3N0ciwgZmxvYXRdXSwgZGV2aWNlOiB0b3JjaC5kZXZpY2UgfCBzdHIgPSAiY3B1IikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiCiAgICBEZXJpdmUgdGhlIG9yYWNsZSBnYXRlIGZyb20gZ3JvdW5kLXRydXRoIGNvbmRpdGlvbiBsYWJlbHMuCgogICAgVGhlIG9yYWNsZSBpcyBhIGhhcmQgYmluYXJ5IGdhdGU6CiAgICAgIHJldmVyYiA9IDEgaWZmIHQ2MF9zID4gMC4wCiAgICAgIG5vaXNlICA9IDEgaWZmIHNucl9kYiA8IDYwLjAgICg2MCBkQiDiiaEgIm5vIG5vaXNlIiBpbiB0aGUgbWl4ZXIpCiAgICAgIGNvZGVjICA9IDEgaWZmIGNvZGVjX2NsYXNzID4gMAoKICAgIEFyZ3M6CiAgICAgICAgcmVjaXBlX3ZlY3RvcnM6IExpc3Qgb2YgQiBkaWN0cyBmcm9tIE1peHR1cmVSZWNpcGUuY29uZGl0aW9uX3ZlY3RvcigpLgogICAgUmV0dXJuczoKICAgICAgICAoQiwgMykgZmxvYXQzMiB0ZW5zb3Igd2l0aCB2YWx1ZXMgaW4gezAuMCwgMS4wfS4KICAgICIiIgogICAgcm93czogbGlzdFtsaXN0W2Zsb2F0XV0gPSBbXQogICAgZm9yIHJ2IGluIHJlY2lwZV92ZWN0b3JzOgogICAgICAgIHJldmVyYiA9IGZsb2F0KHJ2LmdldCgidDYwX3MiLCAwLjApID4gMC4wKQogICAgICAgIG5vaXNlID0gZmxvYXQocnYuZ2V0KCJzbnJfZGIiLCA2MC4wKSA8IDYwLjApCiAgICAgICAgY29kZWMgPSBmbG9hdChydi5nZXQoImNvZGVjX2NsYXNzIiwgMC4wKSA+IDAuMCkKICAgICAgICByb3dzLmFwcGVuZChbcmV2ZXJiLCBub2lzZSwgY29kZWNdKQogICAgcmV0dXJuIHRvcmNoLnRlbnNvcihyb3dzLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9ZGV2aWNlKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgR2F0ZSB0cmFpbmluZyBsb3NzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKZGVmIGdhdGVfbG9zcygKICAgIGdhdGVfbmV0OiBHYXRlTmV0d29yaywKICAgIGNvbmRpdGlvbjogdG9yY2guVGVuc29yLAogICAgcmVjaXBlX3ZlY3RvcnM6IGxpc3RbZGljdFtzdHIsIGZsb2F0XV0sCiAgICBzZXBfbG9zczogdG9yY2guVGVuc29yLAopIC0+IHRvcmNoLlRlbnNvcjoKICAgICIiIgogICAgVG90YWwgZ2F0ZSB0cmFpbmluZyBsb3NzID0gc2VwX2xvc3MgKyBCQ0Ugc3VwZXJ2aXNpb24gKyBMMSBzcGFyc2l0eS4KCiAgICBCTFVFUFJJTlQgwqc2LjE6IHRoZSBnYXRlIGlzIHRyYWluZWQgam9pbnRseSB3aXRoIHRoZSBzZXBhcmF0aW9uIGxvc3MuCiAgICBUaGUgQkNFIHRlcm0gdGVhY2hlcyB0aGUgZ2F0ZSB0byByZXByb2R1Y2UgdGhlIG9yYWNsZSBnaXZlbiB0aGUgY29uZGl0aW9uLgogICAgVGhlIEwxIHRlcm0gcHVzaGVzIHNwYXJzZSwgY2xlYW4gcm91dGluZyByYXRoZXIgdGhhbiBkaWZmdXNlIGhlZGdpbmcuCgogICAgQXJnczoKICAgICAgICBnYXRlX25ldDogR2F0ZU5ldHdvcmsgbW9kdWxlLgogICAgICAgIGNvbmRpdGlvbjogKEIsIDEwKSBjb25kaXRpb24gZmVhdHVyZSB2ZWN0b3IuCiAgICAgICAgcmVjaXBlX3ZlY3RvcnM6IEdyb3VuZC10cnV0aCByZWNpcGUgbGFiZWxzIChCIGRpY3RzKS4KICAgICAgICBzZXBfbG9zczogU2NhbGFyIHNlcGFyYXRpb24gbG9zcyBmcm9tIHRoZSB1cHN0cmVhbSBtb2RlbC4KCiAgICBSZXR1cm5zOgogICAgICAgIFNjYWxhciB0b3RhbCBsb3NzLgogICAgIiIiCiAgICBnYXRlcyA9IGdhdGVfbmV0KGNvbmRpdGlvbikgICMgKEIsIDMpCiAgICBvcmFjbGUgPSBvcmFjbGVfZ2F0ZShyZWNpcGVfdmVjdG9ycywgZGV2aWNlPWNvbmRpdGlvbi5kZXZpY2UpCgogICAgYmNlID0gRi5iaW5hcnlfY3Jvc3NfZW50cm9weSgKICAgICAgICBnYXRlcyAvIGdhdGVfbmV0LmdhdGVfc2NhbGUsICAjIG5vcm1hbGlzZSB0byBbMCwxXSBmb3IgQkNFCiAgICAgICAgb3JhY2xlLAogICAgKQogICAgbDEgPSBnYXRlX25ldC5sMV9wZW5hbHR5KGdhdGVzKQogICAgcmV0dXJuIHNlcF9sb3NzICsgYmNlICsgbDEKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEVNQSBzbW9vdGhlciAoc3RyZWFtaW5nIGluZmVyZW5jZSkKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpjbGFzcyBHYXRlU21vb3RoZXI6CiAgICAiIiIKICAgIEV4cG9uZW50aWFsIG1vdmluZyBhdmVyYWdlIG9mIGdhdGUgdmFsdWVzIGFjcm9zcyBjb25zZWN1dGl2ZSBjaHVua3MuCgogICAgUHJldmVudHMgcGVyLWNodW5rIGZsaWNrZXJpbmcgb24gbG9uZyByZWNvcmRpbmdzLiBCTFVFUFJJTlQgwqc2LjMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgYWxwaGE6IGZsb2F0ID0gX0VNQV9BTFBIQSkgLT4gTm9uZToKICAgICAgICBzZWxmLmFscGhhID0gYWxwaGEKICAgICAgICBzZWxmLl9zdGF0ZTogZGljdFtzdHIsIGZsb2F0XSB8IE5vbmUgPSBOb25lCgogICAgZGVmIHNtb290aChzZWxmLCBnYXRlczogZGljdFtzdHIsIGZsb2F0XSkgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiJVcGRhdGUgRU1BIGFuZCByZXR1cm4gc21vb3RoZWQgZ2F0ZSBkaWN0LiIiIgogICAgICAgIGlmIHNlbGYuX3N0YXRlIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuX3N0YXRlID0gZGljdChnYXRlcykKICAgICAgICAgICAgcmV0dXJuIGRpY3QoZ2F0ZXMpCiAgICAgICAgcmVzdWx0OiBkaWN0W3N0ciwgZmxvYXRdID0ge30KICAgICAgICBmb3IgayBpbiBnYXRlczoKICAgICAgICAgICAgc21vb3RoZWQgPSBzZWxmLmFscGhhICogc2VsZi5fc3RhdGUuZ2V0KGssIGdhdGVzW2tdKSArICgxLjAgLSBzZWxmLmFscGhhKSAqIGdhdGVzW2tdCiAgICAgICAgICAgIHJlc3VsdFtrXSA9IHNtb290aGVkCiAgICAgICAgICAgIHNlbGYuX3N0YXRlW2tdID0gc21vb3RoZWQKICAgICAgICByZXR1cm4gcmVzdWx0CgogICAgZGVmIHJlc2V0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgIiIiUmVzZXQgc3RhdGUgYXQgdGhlIHN0YXJ0IG9mIGEgbmV3IHJlY29yZGluZy4iIiIKICAgICAgICBzZWxmLl9zdGF0ZSA9IE5vbmUK'))
open(f'{PROJ}/data/__init__.py','wb').write(base64.b64decode('IiIiRGF0YSBwaXBlbGluZTogZHluYW1pYyBtaXhlciwgYXVnbWVudGF0aW9uLCBhbmQgZGF0YXNldCBwcmVwYXJhdGlvbiAoRGV2IEEpLiIiIgo='))
open(f'{PROJ}/data/calmsep_mixer.py','wb').write(base64.b64decode('IiIiCkNBTE0tU2VwIDgga0h6IGR5bmFtaWMgbWl4ZXIgd2l0aCBkZWdyYWRhdGlvbiByZWNpcGUgbG9nZ2luZyAoRGV2IEEsIFAwLUExKS4KClRoZSBmcm96ZW4gU1ItQ29yck5ldCB2YXItMi01IGNoZWNrcG9pbnQgb3BlcmF0ZXMgYXQgOCBrSHogKFNURlQgd2luZG93IDEyOCwKaG9wIDY0KS4gRXZlcnkgdHJhaW5pbmcgYW5kIGV2YWx1YXRpb24gbWl4dHVyZSBpbiBDQUxNLVNlcCBpcyB0aGVyZWZvcmUgbWl4ZWQKYXQgOCBrSHouIFRoaXMgbW9kdWxlIGlzIHRoZSA4IGtIeiBjb3VudGVycGFydCB0byBkYXRhL21peGVyLnB5LCB3aGljaCBzdGF5cwphdCAxNiBrSHogZm9yIHRoZSBsZWdhY3kgMTYga0h6IHBhdGggYW5kIHRoZSBiYW5kLXJlY292ZXJ5IHRhcmdldHMuCgpUaGUgY3JpdGljYWwgZGlmZmVyZW5jZSBmcm9tIGRhdGEvbWl4ZXIucHkgaXMgdGhlIHJlY2lwZSBsb2cuIEJMVUVQUklOVCBzZWN0aW9uCjUuNCByZXF1aXJlcyB0aGF0IGV2ZXJ5IGNvbmRpdGlvbiBsYWJlbCAoU05SLCBUNjAsIGNvZGVjIGZhbWlseSBhbmQgYml0cmF0ZSwKc3BlYWtlciBjb3VudCkgY29tZSBmcmVlIGZyb20gdGhlIHN5bnRoZXNpcyByZWNpcGUgcmF0aGVyIHRoYW4gZnJvbSBhIG5ldXJhbAplc3RpbWF0ZS4gVGhpcyBtaXhlciByZXR1cm5zIGEgTWl4dHVyZVJlY2lwZSBhbG9uZ3NpZGUgZXZlcnkgbWl4dHVyZSByZWNvcmRpbmcKZXhhY3RseSB3aGF0IHdhcyBhcHBsaWVkLCBzbyB0aGUgY29uZGl0aW9uIGFuYWx5emVyIGFuZCBnYXRlIHRyYWluIGFnYWluc3QKZ3JvdW5kIHRydXRoIHRoYXQgd2FzIG5ldmVyIGVzdGltYXRlZC4KCkRlZ3JhZGF0aW9ucyBhcmUgYXBwbGllZCBieSBkYXRhL2RlZ3JhZGF0aW9ucy5weTsgdGhpcyBtb2R1bGUgb3ducyB0aGUgc291cmNlCmRyYXcsIGxldmVsIG9mZnNldHMsIHN1bW1hdGlvbiwgYW5kIHRoZSByZWNpcGUgcmVjb3JkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB1dWlkCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBkYXRhLm1peGVyX3N0dWIgaW1wb3J0IE1peHR1cmVTYW1wbGUsIF9sb2FkX3dhdgoKQ0FMTVNFUF9TQU1QTEVfUkFURTogaW50ID0gOF8wMDAKIiIiTG9ja2VkIGJ5IHRoZSBmcm96ZW4gY2hlY2twb2ludC4gQkxVRVBSSU5UIGZpeGVkIGNvbnN0cmFpbnRzLiBOZXZlciBjaGFuZ2UuIiIiCgpCQU5EX1JFQ09WRVJZX1NBTVBMRV9SQVRFOiBpbnQgPSAxNl8wMDAKIiIiUmF0ZSBmb3IgYmFuZC1yZWNvdmVyeSB0YXJnZXRzIGFuZCBETlNNT1Mgc2NvcmluZyBvbmx5LiBOZXZlciBmZWQgdG8gdGhlIGJhc2UuIiIiCgpfREVGQVVMVF9BTExPV0VEX046IGxpc3RbaW50XSA9IFsyLCAzLCA0LCA1XQoiIiJOIGluIHsyLDMsNCw1fS4gSzA9NSBpbiB0aGUgY2hlY2twb2ludDsgdGhlcmUgaXMgbm8gNisgc3BlYWtlciByZWdpbWUuIiIiCgoKQGRhdGFjbGFzcwpjbGFzcyBNaXh0dXJlUmVjaXBlOgogICAgIiIiCiAgICBHcm91bmQtdHJ1dGggcmVjb3JkIG9mIGV2ZXJ5dGhpbmcgYXBwbGllZCB0byBvbmUgbWl4dHVyZS4KCiAgICBFdmVyeSBmaWVsZCBoZXJlIGlzIGEgZnJlZSBzdXBlcnZpc2lvbiB0YXJnZXQ6IGl0IGlzIGtub3duIGJlY2F1c2UgdGhpcwogICAgY29kZSBjaG9zZSBpdCwgbm90IGJlY2F1c2UgYSBtb2RlbCBlc3RpbWF0ZWQgaXQuIFRoZSBjb25kaXRpb24gYW5hbHl6ZXIKICAgIChCTFVFUFJJTlQgNS40KSBhbmQgdGhlIGdhdGUgKDUuNSkgdHJhaW4gYWdhaW5zdCB0aGVzZSB2YWx1ZXMuCgogICAgQXR0cmlidXRlczoKICAgICAgICBuX3NwZWFrZXJzOiBUcnVlIHNwZWFrZXIgY291bnQsIHRoZSBwcmltYXJ5IGNvdW50aW5nIGxhYmVsLgogICAgICAgIHNwZWFrZXJfaWRzOiBTb3VyY2Ugc3BlYWtlciBJRHMsIGluIHJlZmVyZW5jZS1zdHJlYW0gb3JkZXIuCiAgICAgICAgc291cmNlX2ZpbGVzOiBTb3VyY2UgdXR0ZXJhbmNlIHBhdGhzLCBpbiByZWZlcmVuY2Utc3RyZWFtIG9yZGVyLgogICAgICAgIGxldmVsX29mZnNldHNfZGI6IFBlci1zcGVha2VyIGdhaW4gYXBwbGllZCwgaW4gcmVmZXJlbmNlLXN0cmVhbSBvcmRlci4KICAgICAgICBzbnJfZGI6IE5vaXNlIFNOUiBpbiBkQiwgb3IgTm9uZSB3aGVuIG5vIG5vaXNlIHdhcyBhZGRlZC4KICAgICAgICBub2lzZV9maWxlOiBOb2lzZSBzb3VyY2UgcGF0aCwgb3IgTm9uZS4KICAgICAgICB0NjBfczogUmV2ZXJiZXJhdGlvbiB0aW1lIGluIHNlY29uZHMsIG9yIE5vbmUgd2hlbiBhbmVjaG9pYy4KICAgICAgICByaXJfZmlsZTogUklSIHBhdGggdXNlZCwgb3IgTm9uZS4KICAgICAgICBjb2RlY19uYW1lOiBDb2RlYyBmYW1pbHkgYXBwbGllZCAoIm9wdXMiLCAiYWFjIiwgImFtci1uYiIsICJhbXItd2IiKSwKICAgICAgICAgICAgb3IgTm9uZSB3aGVuIHVuY29tcHJlc3NlZC4KICAgICAgICBjb2RlY19iaXRyYXRlX2JwczogQ29kZWMgYml0cmF0ZSBpbiBiaXRzL3NlYywgb3IgTm9uZS4KICAgICAgICBzZWVkOiBSTkcgc2VlZCB0aGF0IHByb2R1Y2VkIHRoaXMgbWl4dHVyZSwgd2hlbiB0aGUgbWl4ZXIgd2FzIHNlZWRlZC4KICAgICAgICBzYW1wbGVfcmF0ZTogQWx3YXlzIENBTE1TRVBfU0FNUExFX1JBVEUuCiAgICAiIiIKCiAgICBuX3NwZWFrZXJzOiBpbnQKICAgIHNwZWFrZXJfaWRzOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHNvdXJjZV9maWxlczogbGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBsZXZlbF9vZmZzZXRzX2RiOiBsaXN0W2Zsb2F0XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgc25yX2RiOiBmbG9hdCB8IE5vbmUgPSBOb25lCiAgICBub2lzZV9maWxlOiBzdHIgfCBOb25lID0gTm9uZQogICAgdDYwX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgIHJpcl9maWxlOiBzdHIgfCBOb25lID0gTm9uZQogICAgY29kZWNfbmFtZTogc3RyIHwgTm9uZSA9IE5vbmUKICAgIGNvZGVjX2JpdHJhdGVfYnBzOiBpbnQgfCBOb25lID0gTm9uZQogICAgc2VlZDogaW50IHwgTm9uZSA9IE5vbmUKICAgIHNhbXBsZV9yYXRlOiBpbnQgPSBDQUxNU0VQX1NBTVBMRV9SQVRFCgogICAgZGVmIHRvX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIEFueV06CiAgICAgICAgIiIiU2VyaWFsaXplIGZvciBtYW5pZmVzdCB3cml0aW5nIGFuZCBjb25kaXRpb24tbGFiZWwgZXh0cmFjdGlvbi4iIiIKICAgICAgICByZXR1cm4gYXNkaWN0KHNlbGYpCgogICAgZGVmIGNvbmRpdGlvbl92ZWN0b3Ioc2VsZikgLT4gZGljdFtzdHIsIGZsb2F0XToKICAgICAgICAiIiIKICAgICAgICBUaGUgc3VwZXJ2aXNlZCBjb25kaXRpb24gdGFyZ2V0cywgYXMgdGhlIGFuYWx5emVyIGNvbnN1bWVzIHRoZW0uCgogICAgICAgIEFic2VudCBjb25kaXRpb25zIG1hcCB0byB0aGVpciBuZXV0cmFsIHZhbHVlIHJhdGhlciB0aGFuIE5vbmUgc28gdGhlCiAgICAgICAgdmVjdG9yIGlzIGFsd2F5cyBkZW5zZTogbm8gbm9pc2UgbWVhbnMgYSBoaWdoIFNOUiwgYW5lY2hvaWMgbWVhbnMgYQogICAgICAgIG5lYXItemVybyBUNjAsIHVuY29tcHJlc3NlZCBtZWFucyBjb2RlYyBjbGFzcyAwLiBUaGlzIGlzIHdoYXQgbWFrZXMKICAgICAgICB0aGUgZ2F0ZSdzIGNsZWFuLWlucHV0IHRhcmdldCAoYWxsIGdhdGVzIG5lYXIgemVybykgd2VsbCBkZWZpbmVkLgoKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBEaWN0IHdpdGgga2V5cyBzbnJfZGIsIHQ2MF9zLCBjb2RlY19jbGFzcywgY29kZWNfYml0cmF0ZV9rYnBzLAogICAgICAgICAgICBuX3NwZWFrZXJzLiBDb25zdW1lZCBieSBtb2RlbHMvY29uZGl0aW9uLnB5LgogICAgICAgICIiIgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJzbnJfZGIiOiA2MC4wIGlmIHNlbGYuc25yX2RiIGlzIE5vbmUgZWxzZSBmbG9hdChzZWxmLnNucl9kYiksCiAgICAgICAgICAgICJ0NjBfcyI6IDAuMCBpZiBzZWxmLnQ2MF9zIGlzIE5vbmUgZWxzZSBmbG9hdChzZWxmLnQ2MF9zKSwKICAgICAgICAgICAgImNvZGVjX2NsYXNzIjogZmxvYXQoX0NPREVDX0NMQVNTX0lOREVYLmdldChzZWxmLmNvZGVjX25hbWUgb3IgIm5vbmUiLCAwKSksCiAgICAgICAgICAgICJjb2RlY19iaXRyYXRlX2ticHMiOiAoCiAgICAgICAgICAgICAgICAwLjAgaWYgc2VsZi5jb2RlY19iaXRyYXRlX2JwcyBpcyBOb25lIGVsc2Ugc2VsZi5jb2RlY19iaXRyYXRlX2JwcyAvIDEwMDAuMAogICAgICAgICAgICApLAogICAgICAgICAgICAibl9zcGVha2VycyI6IGZsb2F0KHNlbGYubl9zcGVha2VycyksCiAgICAgICAgfQoKCl9DT0RFQ19DTEFTU19JTkRFWDogZGljdFtzdHIsIGludF0gPSB7CiAgICAibm9uZSI6IDAsCiAgICAib3B1cyI6IDEsCiAgICAiYWFjIjogMiwKICAgICJhbXItbmIiOiAzLAogICAgImFtci13YiI6IDQsCn0KIiIiQ29kZWMgZmFtaWx5IHRvIGNsYXNzIGluZGV4LiBJbmRleCAwIChub25lKSBpcyB0aGUgY2xlYW4vbmV1dHJhbCBjbGFzcy4iIiIKCgpAZGF0YWNsYXNzCmNsYXNzIENhbG1TZXBNaXh0dXJlOgogICAgIiIiCiAgICBPbmUgOCBrSHogbWl4dHVyZSB3aXRoIGl0cyBzdGVtcyBhbmQgaXRzIGdyb3VuZC10cnV0aCByZWNpcGUuCgogICAgQXR0cmlidXRlczoKICAgICAgICBzYW1wbGU6IFRoZSBtaXh0dXJlIGFuZCByZWZlcmVuY2Ugc3RlbXMgKE1peHR1cmVTYW1wbGUsIDgga0h6KS4KICAgICAgICByZWNpcGU6IFdoYXQgd2FzIGFwcGxpZWQsIGZvciBzdXBlcnZpc2lvbiBhbmQgbWFuaWZlc3RzLgogICAgIiIiCgogICAgc2FtcGxlOiBNaXh0dXJlU2FtcGxlCiAgICByZWNpcGU6IE1peHR1cmVSZWNpcGUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBtaXh0dXJlKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIHNlbGYuc2FtcGxlLm1peHR1cmUKCiAgICBAcHJvcGVydHkKICAgIGRlZiByZWZlcmVuY2VzKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIHNlbGYuc2FtcGxlLnJlZmVyZW5jZXMKCiAgICBAcHJvcGVydHkKICAgIGRlZiBuX3NwZWFrZXJzKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5yZWNpcGUubl9zcGVha2VycwoKCmRlZiBfc3BlYWtlcl9pZChwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICAiIiIKICAgIFNwZWFrZXIgSUQgZnJvbSBhIExpYnJpU3BlZWNoLXN0eWxlIGZpbGVuYW1lLgoKICAgIExpYnJpU3BlZWNoIG5hbWVzIGFyZSB7c3BlYWtlcn0te2NoYXB0ZXJ9LXt1dHRlcmFuY2V9LmV4dCwgc28gdGhlIElEIGlzIHRoZQogICAgY29tcG9uZW50IGJlZm9yZSB0aGUgZmlyc3QgZGFzaC4gTm9uLWNvbmZvcm1pbmcgbmFtZXMgZmFsbCBiYWNrIHRvIHRoZSBmdWxsCiAgICBzdGVtLCB3aGljaCBrZWVwcyBzcGVha2VyIGlzb2xhdGlvbiBjb25zZXJ2YXRpdmU6IGFuIHVucGFyc2VkIG5hbWUgaXMgaXRzCiAgICBvd24gc3BlYWtlciByYXRoZXIgdGhhbiBzaWxlbnRseSBjb2xsaWRpbmcgd2l0aCBhbm90aGVyLgogICAgIiIiCiAgICByZXR1cm4gcGF0aC5zdGVtLnNwbGl0KCItIilbMF0KCgpjbGFzcyBDYWxtU2VwTWl4ZXI6CiAgICAiIiIKICAgIERyYXdzIE4gY2xlYW4gOCBrSHogdXR0ZXJhbmNlcyBhbmQgbWl4ZXMgdGhlbSwgbG9nZ2luZyB0aGUgcmVjaXBlLgoKICAgIFNwZWFrZXIgaXNvbGF0aW9uIGlzIGVuZm9yY2VkIGF0IGNvbnN0cnVjdGlvbjogZmlsZXMgYmVsb25naW5nIHRvIGhlbGQtb3V0CiAgICBzcGVha2VycyBhcmUgcmVtb3ZlZCBmcm9tIHRoZSB0cmFpbmluZyBwb29sIGVudGlyZWx5LCBzbyBhIGRldi1jbGVhbiBvcgogICAgdGVzdC1jbGVhbiBzcGVha2VyIGNhbiBuZXZlciBsZWFrIGludG8gdHJhaW5pbmcgKEJMVUVQUklOVCA3LjUsIGhvbGRvdXQgMSkuCgogICAgUGFyYW1ldGVycwogICAgLS0tLS0tLS0tLQogICAgc291cmNlX2ZpbGVzOgogICAgICAgIENsZWFuIHNpbmdsZS1zcGVha2VyIFdBVi9GTEFDIGZpbGVzLCBhbHJlYWR5IGF0IDgga0h6LgogICAgYWxsb3dlZF9uOgogICAgICAgIFNwZWFrZXIgY291bnRzIHRvIGRyYXcgZnJvbS4gRGVmYXVsdHMgdG8gWzIsIDMsIDQsIDVdLgogICAgZGJfbWluLCBkYl9tYXg6CiAgICAgICAgUGVyLXNwZWFrZXIgbGV2ZWwgb2Zmc2V0IHJhbmdlIGluIGRCLCBkcmF3biBpbmRlcGVuZGVudGx5IHBlciBzcGVha2VyLgogICAgaGVsZF9vdXRfc3BlYWtlcl9pZHM6CiAgICAgICAgU3BlYWtlcnMgcmVzZXJ2ZWQgZm9yIHZhbGlkYXRpb24gYW5kIGV2YWx1YXRpb24uIEV4Y2x1ZGVkIGZyb20gdGhlCiAgICAgICAgdHJhaW5pbmcgcG9vbCBhbmQgcmVhY2hhYmxlIG9ubHkgdmlhIGBgbWl4KHNwbGl0PSJoZWxkb3V0IilgYC4KICAgIHNhbXBsZV9yYXRlOgogICAgICAgIEV4cGVjdGVkIGlucHV0IHJhdGUuIERlZmF1bHRzIHRvIENBTE1TRVBfU0FNUExFX1JBVEUgKDgwMDApLiBBIGZpbGUgYXQKICAgICAgICBhbnkgb3RoZXIgcmF0ZSByYWlzZXMgcmF0aGVyIHRoYW4gYmVpbmcgc2lsZW50bHkgcmVzYW1wbGVkLCBiZWNhdXNlIGEKICAgICAgICBzaWxlbnQgcmVzYW1wbGUgaXMgaG93IGEgMTYga0h6IGZpbGUgZW5kcyB1cCBpbnRlcnByZXRlZCBhcyA4IGtIei4KICAgIHJuZzoKICAgICAgICBTZWVkZWQgZ2VuZXJhdG9yIGZvciByZXByb2R1Y2libGUgbWl4ZXMuCiAgICBzZWVkOgogICAgICAgIFJlY29yZGVkIGludG8gZXZlcnkgcmVjaXBlIGZvciB0cmFjZWFiaWxpdHkuIFBhc3MgdGhlIHNhbWUgdmFsdWUgdXNlZAogICAgICAgIHRvIGJ1aWxkIGBgcm5nYGAuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBzb3VyY2VfZmlsZXM6IFNlcXVlbmNlW1BhdGggfCBzdHJdLAogICAgICAgIGFsbG93ZWRfbjogbGlzdFtpbnRdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgZGJfbWluOiBmbG9hdCA9IDAuMCwKICAgICAgICBkYl9tYXg6IGZsb2F0ID0gNS4wLAogICAgICAgIGhlbGRfb3V0X3NwZWFrZXJfaWRzOiBzZXRbc3RyXSB8IE5vbmUgPSBOb25lLAogICAgICAgIHNhbXBsZV9yYXRlOiBpbnQgPSBDQUxNU0VQX1NBTVBMRV9SQVRFLAogICAgICAgIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciB8IE5vbmUgPSBOb25lLAogICAgICAgIHNlZWQ6IGludCB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBOb25lOgogICAgICAgIGlmIGRiX21pbiA+IGRiX21heDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImRiX21pbiAoe2RiX21pbn0pIG11c3QgYmUgPD0gZGJfbWF4ICh7ZGJfbWF4fSkiKQoKICAgICAgICBzZWxmLl9hbGxvd2VkX24gPSBsaXN0KGFsbG93ZWRfbikgaWYgYWxsb3dlZF9uIGlzIG5vdCBOb25lIGVsc2UgbGlzdChfREVGQVVMVF9BTExPV0VEX04pCiAgICAgICAgaWYgbm90IHNlbGYuX2FsbG93ZWRfbjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYWxsb3dlZF9uIG11c3QgY29udGFpbiBhdCBsZWFzdCBvbmUgdmFsdWUiKQogICAgICAgIGJhZCA9IFtuIGZvciBuIGluIHNlbGYuX2FsbG93ZWRfbiBpZiBuIG5vdCBpbiAoMiwgMywgNCwgNSldCiAgICAgICAgaWYgYmFkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJhbGxvd2VkX24gY29udGFpbnMge2JhZH07IENBTE0tU2VwIHN1cHBvcnRzIE4gaW4ge3syLDMsNCw1fX0gb25seSAiCiAgICAgICAgICAgICAgICBmIihLMD01IGluIHRoZSBmcm96ZW4gY2hlY2twb2ludCkiCiAgICAgICAgICAgICkKCiAgICAgICAgc2VsZi5fZGJfbWluID0gZmxvYXQoZGJfbWluKQogICAgICAgIHNlbGYuX2RiX21heCA9IGZsb2F0KGRiX21heCkKICAgICAgICBzZWxmLl9zYW1wbGVfcmF0ZSA9IGludChzYW1wbGVfcmF0ZSkKICAgICAgICBzZWxmLl9ybmcgPSBybmcgaWYgcm5nIGlzIG5vdCBOb25lIGVsc2UgbnAucmFuZG9tLmRlZmF1bHRfcm5nKCkKICAgICAgICBzZWxmLl9zZWVkID0gc2VlZAoKICAgICAgICBhbGxfZmlsZXMgPSBbUGF0aChmKSBmb3IgZiBpbiBzb3VyY2VfZmlsZXNdCiAgICAgICAgaGVsZCA9IHNldChoZWxkX291dF9zcGVha2VyX2lkcykgaWYgaGVsZF9vdXRfc3BlYWtlcl9pZHMgZWxzZSBzZXQoKQoKICAgICAgICBzZWxmLl9oZWxkb3V0X2ZpbGVzOiBsaXN0W1BhdGhdID0gW2YgZm9yIGYgaW4gYWxsX2ZpbGVzIGlmIF9zcGVha2VyX2lkKGYpIGluIGhlbGRdCiAgICAgICAgc2VsZi5fdHJhaW5fZmlsZXM6IGxpc3RbUGF0aF0gPSBbZiBmb3IgZiBpbiBhbGxfZmlsZXMgaWYgX3NwZWFrZXJfaWQoZikgbm90IGluIGhlbGRdCgogICAgICAgIG1heF9uID0gbWF4KHNlbGYuX2FsbG93ZWRfbikKICAgICAgICBpZiBsZW4oc2VsZi5fdHJhaW5fZmlsZXMpIDwgbWF4X246CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmInRyYWluaW5nIHBvb2wgaGFzIHtsZW4oc2VsZi5fdHJhaW5fZmlsZXMpfSBmaWxlKHMpIGJ1dCAiCiAgICAgICAgICAgICAgICBmIm1heChhbGxvd2VkX24pPXttYXhfbn07IGFkZCBzb3VyY2VzIG9yIHJlZHVjZSBhbGxvd2VkX24iCiAgICAgICAgICAgICkKCiAgICAjIOKUgOKUgCBQdWJsaWMgQVBJIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBtaXgoc2VsZiwgc3BsaXQ6IHN0ciA9ICJ0cmFpbiIsIG46IGludCB8IE5vbmUgPSBOb25lKSAtPiBDYWxtU2VwTWl4dHVyZToKICAgICAgICAiIiIKICAgICAgICBQcm9kdWNlIG9uZSBjbGVhbiA4IGtIeiBtaXh0dXJlIGFuZCBpdHMgcmVjaXBlLgoKICAgICAgICBEZWdyYWRhdGlvbnMgKHJldmVyYiwgbm9pc2UsIGNvZGVjKSBhcmUgYXBwbGllZCBhZnRlcndhcmRzIGJ5CiAgICAgICAgZGF0YS9kZWdyYWRhdGlvbnMucHksIHdoaWNoIGV4dGVuZHMgdGhlIHJldHVybmVkIHJlY2lwZSBpbiBwbGFjZS4gVGhpcwogICAgICAgIHNwbGl0IGtlZXBzIHRoZSBzb3VyY2UgZHJhdyBpbmRlcGVuZGVudCBvZiB0aGUgY29uZGl0aW9uIHNhbXBsaW5nLCBzbwogICAgICAgIHRoZSBzYW1lIG1peHR1cmUgY2FuIGJlIHJlbmRlcmVkIHVuZGVyIHNldmVyYWwgY29uZGl0aW9ucyBmb3IgdGhlCiAgICAgICAgbWF0Y2hlZC1wYWlyIGFuYWx5c2VzIGluIEJMVUVQUklOVCA5LjUuCgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHNwbGl0OiAidHJhaW4iIGRyYXdzIGZyb20gdGhlIHRyYWluaW5nIHBvb2wgKGhlbGQtb3V0IHNwZWFrZXJzCiAgICAgICAgICAgICAgICBleGNsdWRlZCksICJoZWxkb3V0IiBkcmF3cyBvbmx5IGZyb20gaGVsZC1vdXQgc3BlYWtlcnMsIGFueQogICAgICAgICAgICAgICAgb3RoZXIgdmFsdWUgZHJhd3MgZnJvbSBib3RoLgogICAgICAgICAgICBuOiBTcGVha2VyIGNvdW50IG92ZXJyaWRlLiBNdXN0IGJlIGluIGFsbG93ZWRfbi4gRHJhd24gdW5pZm9ybWx5CiAgICAgICAgICAgICAgICBmcm9tIGFsbG93ZWRfbiB3aGVuIE5vbmUuCgogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIENhbG1TZXBNaXh0dXJlIHdpdGggYW4gYW5lY2hvaWMsIG5vaXNlLWZyZWUsIHVuY29tcHJlc3NlZCBtaXh0dXJlCiAgICAgICAgICAgIGFuZCBhIHJlY2lwZSByZWNvcmRpbmcgdGhlIHNvdXJjZXMgYW5kIGxldmVsIG9mZnNldHMuCiAgICAgICAgIiIiCiAgICAgICAgY2hvc2VuX24gPSBzZWxmLl9yZXNvbHZlX24obikKICAgICAgICBwb29sID0gc2VsZi5fc2VsZWN0X3Bvb2woc3BsaXQpCgogICAgICAgIGlmIGxlbihwb29sKSA8IGNob3Nlbl9uOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJwb29sIGZvciBzcGxpdD17c3BsaXQhcn0gaGFzIHtsZW4ocG9vbCl9IGZpbGUocykgYnV0IG49e2Nob3Nlbl9ufSByZXF1ZXN0ZWQiCiAgICAgICAgICAgICkKCiAgICAgICAgaW5kaWNlcyA9IHNlbGYuX3JuZy5jaG9pY2UobGVuKHBvb2wpLCBzaXplPWNob3Nlbl9uLCByZXBsYWNlPUZhbHNlKQogICAgICAgIGNob3NlbiA9IFtwb29sW2ludChpKV0gZm9yIGkgaW4gaW5kaWNlc10KCiAgICAgICAgd2F2ZWZvcm1zOiBsaXN0W25wLm5kYXJyYXldID0gW10KICAgICAgICBvZmZzZXRzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgZm9yIHBhdGggaW4gY2hvc2VuOgogICAgICAgICAgICBhdWRpbywgc3IgPSBfbG9hZF93YXYocGF0aCkKICAgICAgICAgICAgaWYgc3IgIT0gc2VsZi5fc2FtcGxlX3JhdGU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgICAgIGYic2FtcGxlIHJhdGUgbWlzbWF0Y2g6IHtwYXRofSBpcyB7c3J9IEh6LCBleHBlY3RlZCB7c2VsZi5fc2FtcGxlX3JhdGV9IEh6LiAiCiAgICAgICAgICAgICAgICAgICAgZiJSZXNhbXBsZSB0aGUgY29ycHVzIHdpdGggZGF0YS9wcmVwYXJlX2xpYnJpc3BlZWNoXzhrLnB5IHJhdGhlciB0aGFuICIKICAgICAgICAgICAgICAgICAgICBmInJlc2FtcGxpbmcgaGVyZSwgc28gdGhlIHdob2xlIHBvb2wgc3RheXMgY29uc2lzdGVudC4iCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGRiID0gZmxvYXQoc2VsZi5fcm5nLnVuaWZvcm0oc2VsZi5fZGJfbWluLCBzZWxmLl9kYl9tYXgpKQogICAgICAgICAgICBvZmZzZXRzLmFwcGVuZChkYikKICAgICAgICAgICAgd2F2ZWZvcm1zLmFwcGVuZCgoYXVkaW8gKiAoMTAuMCAqKiAoZGIgLyAyMC4wKSkpLmFzdHlwZShucC5mbG9hdDMyKSkKCiAgICAgICAgcmVmcywgbWl4dHVyZSA9IHNlbGYuX3BhZF9hbmRfc3VtKHdhdmVmb3JtcykKICAgICAgICB1aWQgPSBmImNhbG1zZXBfe2Nob3Nlbl9ufXNwa197dXVpZC51dWlkNCgpLmhleFs6OF19IgoKICAgICAgICByZWNpcGUgPSBNaXh0dXJlUmVjaXBlKAogICAgICAgICAgICBuX3NwZWFrZXJzPWNob3Nlbl9uLAogICAgICAgICAgICBzcGVha2VyX2lkcz1bX3NwZWFrZXJfaWQocCkgZm9yIHAgaW4gY2hvc2VuXSwKICAgICAgICAgICAgc291cmNlX2ZpbGVzPVtzdHIocCkgZm9yIHAgaW4gY2hvc2VuXSwKICAgICAgICAgICAgbGV2ZWxfb2Zmc2V0c19kYj1vZmZzZXRzLAogICAgICAgICAgICBzZWVkPXNlbGYuX3NlZWQsCiAgICAgICAgICAgIHNhbXBsZV9yYXRlPXNlbGYuX3NhbXBsZV9yYXRlLAogICAgICAgICkKICAgICAgICBzYW1wbGUgPSBNaXh0dXJlU2FtcGxlKAogICAgICAgICAgICBtaXh0dXJlPW1peHR1cmUsCiAgICAgICAgICAgIHJlZmVyZW5jZXM9cmVmcywKICAgICAgICAgICAgc2FtcGxlX3JhdGU9c2VsZi5fc2FtcGxlX3JhdGUsCiAgICAgICAgICAgIHV0dGVyYW5jZV9pZD11aWQsCiAgICAgICAgKQogICAgICAgIHJldHVybiBDYWxtU2VwTWl4dHVyZShzYW1wbGU9c2FtcGxlLCByZWNpcGU9cmVjaXBlKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHRyYWluX3Bvb2xfc2l6ZShzZWxmKSAtPiBpbnQ6CiAgICAgICAgIiIiTnVtYmVyIG9mIHNvdXJjZSBmaWxlcyBlbGlnaWJsZSBmb3IgdHJhaW5pbmcgbWl4ZXMuIiIiCiAgICAgICAgcmV0dXJuIGxlbihzZWxmLl90cmFpbl9maWxlcykKCiAgICBAcHJvcGVydHkKICAgIGRlZiBoZWxkb3V0X3Bvb2xfc2l6ZShzZWxmKSAtPiBpbnQ6CiAgICAgICAgIiIiTnVtYmVyIG9mIHNvdXJjZSBmaWxlcyByZXNlcnZlZCBmb3IgdmFsaWRhdGlvbiBhbmQgZXZhbHVhdGlvbi4iIiIKICAgICAgICByZXR1cm4gbGVuKHNlbGYuX2hlbGRvdXRfZmlsZXMpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdHJhaW5fc3BlYWtlcnMoc2VsZikgLT4gc2V0W3N0cl06CiAgICAgICAgIiIiU3BlYWtlciBJRHMgcHJlc2VudCBpbiB0aGUgdHJhaW5pbmcgcG9vbC4iIiIKICAgICAgICByZXR1cm4ge19zcGVha2VyX2lkKGYpIGZvciBmIGluIHNlbGYuX3RyYWluX2ZpbGVzfQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGhlbGRvdXRfc3BlYWtlcnMoc2VsZikgLT4gc2V0W3N0cl06CiAgICAgICAgIiIiU3BlYWtlciBJRHMgcmVzZXJ2ZWQgZm9yIHZhbGlkYXRpb24gYW5kIGV2YWx1YXRpb24uIiIiCiAgICAgICAgcmV0dXJuIHtfc3BlYWtlcl9pZChmKSBmb3IgZiBpbiBzZWxmLl9oZWxkb3V0X2ZpbGVzfQoKICAgIGRlZiBhc3NlcnRfc3BlYWtlcl9pc29sYXRpb24oc2VsZikgLT4gTm9uZToKICAgICAgICAiIiIKICAgICAgICBQcm92ZSBubyBzcGVha2VyIGFwcGVhcnMgaW4gYm90aCBwb29scy4KCiAgICAgICAgQ2FsbGVkIGJ5IHRoZSBwcmVmbGlnaHQgY2hlY2sgYW5kIGJ5IHRlc3RzLiBBIHZpb2xhdGlvbiBoZXJlIG1lYW5zCiAgICAgICAgQkxVRVBSSU5UIGhvbGRvdXQgMSBpcyBicm9rZW4gYW5kIGV2ZXJ5IGRvd25zdHJlYW0gbnVtYmVyIGlzIHN1c3BlY3QsCiAgICAgICAgc28gdGhpcyByYWlzZXMgcmF0aGVyIHRoYW4gd2FybnMuCiAgICAgICAgIiIiCiAgICAgICAgb3ZlcmxhcCA9IHNlbGYudHJhaW5fc3BlYWtlcnMgJiBzZWxmLmhlbGRvdXRfc3BlYWtlcnMKICAgICAgICBpZiBvdmVybGFwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJzcGVha2VyIGlzb2xhdGlvbiB2aW9sYXRlZDoge3NvcnRlZChvdmVybGFwKX0gYXBwZWFyIGluIGJvdGggdGhlICIKICAgICAgICAgICAgICAgIGYidHJhaW5pbmcgYW5kIGhlbGQtb3V0IHBvb2xzIgogICAgICAgICAgICApCgogICAgIyDilIDilIAgUHJpdmF0ZSBoZWxwZXJzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBfcmVzb2x2ZV9uKHNlbGYsIG46IGludCB8IE5vbmUpIC0+IGludDoKICAgICAgICBpZiBuIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBpbnQoc2VsZi5fcm5nLmNob2ljZShzZWxmLl9hbGxvd2VkX24pKQogICAgICAgIGlmIG4gbm90IGluIHNlbGYuX2FsbG93ZWRfbjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIm49e259IGlzIG5vdCBpbiBhbGxvd2VkX249e3NlbGYuX2FsbG93ZWRfbn0iKQogICAgICAgIHJldHVybiBuCgogICAgZGVmIF9zZWxlY3RfcG9vbChzZWxmLCBzcGxpdDogc3RyKSAtPiBsaXN0W1BhdGhdOgogICAgICAgIGlmIHNwbGl0ID09ICJoZWxkb3V0IjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2hlbGRvdXRfZmlsZXMKICAgICAgICBpZiBzcGxpdCA9PSAidHJhaW4iOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fdHJhaW5fZmlsZXMKICAgICAgICByZXR1cm4gc2VsZi5fdHJhaW5fZmlsZXMgKyBzZWxmLl9oZWxkb3V0X2ZpbGVzCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9wYWRfYW5kX3N1bSh3YXZlZm9ybXM6IGxpc3RbbnAubmRhcnJheV0pIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgICAgICIiIlplcm8tcGFkIHRvIHRoZSBsb25nZXN0IHN0ZW0sIHN0YWNrIHRvIFtOLCBUXSwgc3VtIHRvIFtUXS4iIiIKICAgICAgICBtYXhfbGVuID0gbWF4KHcuc2hhcGVbMF0gZm9yIHcgaW4gd2F2ZWZvcm1zKQogICAgICAgIHBhZGRlZCA9IFtucC5wYWQodywgKDAsIG1heF9sZW4gLSB3LnNoYXBlWzBdKSkuYXN0eXBlKG5wLmZsb2F0MzIpIGZvciB3IGluIHdhdmVmb3Jtc10KICAgICAgICByZWZzID0gbnAuc3RhY2socGFkZGVkLCBheGlzPTApCiAgICAgICAgcmV0dXJuIHJlZnMsIHJlZnMuc3VtKGF4aXM9MCkK'))
open(f'{PROJ}/data/degradations.py','wb').write(base64.b64decode('IiIiCkRlZ3JhZGF0aW9uIGFwcGxpY2F0aW9uIHdpdGggd2V0LXJlZmVyZW5jZSBwb2xpY3kgKERldiBBLCBQMC1BNyBhbmQgUDEtQTEvQTIpLgoKQXBwbGllcyByZXZlcmJlcmF0aW9uLCBub2lzZSwgYW5kIGNvZGVjIGRhbWFnZSB0byBhIGNsZWFuIDgga0h6IG1peHR1cmUgYW5kCmV4dGVuZHMgaXRzIE1peHR1cmVSZWNpcGUgd2l0aCB0aGUgZ3JvdW5kLXRydXRoIGxhYmVscy4gVGhpcyBpcyB0aGUgbW9kdWxlIHRoYXQKdHVybnMgYSBjbGVhbiBDYWxtU2VwTWl4dHVyZSBpbnRvIGEgdHJhaW5pbmcgb3IgZXZhbHVhdGlvbiBleGFtcGxlIHVuZGVyIGEKbmFtZWQgY29uZGl0aW9uLgoKVGhlIHJlZmVyZW5jZSBwb2xpY3kgaXMgdGhlIHN1YnRsZSBwYXJ0LiBCTFVFUFJJTlQgNy42OiBmb3IgcmV2ZXJiZXJhbnQgZGF0YQp0aGUgdGFyZ2V0IGlzIHRoZSAqKndldCBzb3VyY2UqKiAodGhlIHNwZWFrZXIgY29udm9sdmVkIHdpdGggdGhlIFJJUiwgdHJ1bmNhdGVkCmF0IG5fcGVhayArIDUxMiBzYW1wbGVzKSwgbm90IHRoZSBkcnkgc291cmNlLiBUaGUgc3lzdGVtIGlzIGFza2VkIHRvIHNlcGFyYXRlCnNwZWFrZXJzLCBub3QgdG8gZGVyZXZlcmJlcmF0ZSB0aGVtLiBTY29yaW5nIGFnYWluc3QgZHJ5IHNvdXJjZXMgd291bGQgY29uZmxhdGUKdHdvIHRhc2tzIGFuZCBtYWtlIHJldmVyYmVyYW50IFNJLVNEUmkgdW5pbnRlcnByZXRhYmxlOiBhIHBlcmZlY3Qgc2VwYXJhdG9yCnRoYXQgbGVhdmVzIHJldmVyYiBpbnRhY3Qgd291bGQgc2NvcmUgYmFkbHksIHdoaWNoIGlzIHRoZSB3cm9uZyBpbmNlbnRpdmUuCgpUaGUgdHJ1bmNhdGlvbiBvZmZzZXQgKDUxMiBzYW1wbGVzIGF0IDgga0h6ID0gNjQgbXMpIGtlZXBzIHRoZSBkaXJlY3QgcGF0aCBhbmQKdGhlIGVhcmx5IHJlZmxlY3Rpb25zIHRoYXQgYXJyaXZlIHdpdGggaXQsIGFuZCBkaXNjYXJkcyB0aGUgbGF0ZSB0YWlsLiBFYXJseQpyZWZsZWN0aW9ucyBhcmUgcGVyY2VwdHVhbGx5IGZ1c2VkIHdpdGggdGhlIGRpcmVjdCBzb3VuZCBhbmQgY2FycnkgdGhlIHNwZWFrZXIncwp0aW1icmU7IHRoZSBsYXRlIHRhaWwgaXMgd2hhdCBhIGRlcmV2ZXJiZXJhdG9yIHdvdWxkIHJlbW92ZS4gS2VlcGluZyB0aGUgZWFybHkKcGFydCBpbiB0aGUgdGFyZ2V0IGlzIHdoYXQgbWFrZXMgInNlcGFyYXRlIGJ1dCBkbyBub3QgZGVyZXZlcmJlcmF0ZSIgcHJlY2lzZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCByZXBsYWNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkgaW1wb3J0IHNpZ25hbAoKZnJvbSBkYXRhLmNhbG1zZXBfbWl4ZXIgaW1wb3J0IENBTE1TRVBfU0FNUExFX1JBVEUsIENhbG1TZXBNaXh0dXJlLCBNaXh0dXJlUmVjaXBlCmZyb20gZGF0YS5taXhlcl9zdHViIGltcG9ydCBNaXh0dXJlU2FtcGxlCmZyb20gZGF0YS5yaXJfYmFuayBpbXBvcnQgUmlyQmFuaywgUmlyUmVjb3JkLCBzYW1wbGVfdDYwCgpXRVRfUkVGRVJFTkNFX09GRlNFVF9TQU1QTEVTOiBpbnQgPSA1MTIKIiIiCkJMVUVQUklOVCA3LjY6IHdldCByZWZlcmVuY2VzIGFyZSB0cnVuY2F0ZWQgYXQgbl9wZWFrICsgNTEyLiBBdCA4IGtIeiB0aGlzIGlzCjY0IG1zIG9mIGVhcmx5IHJlZmxlY3Rpb25zIHJldGFpbmVkIHBhc3QgdGhlIGRpcmVjdCBwYXRoLiBNYXRjaGVzIHRoZSBzb3VyY2UKcGFwZXIncyBzaW5nbGUtY2hhbm5lbCBuX29mZnNldC4KIiIiCgpTTlJfTUlOX0RCOiBmbG9hdCA9IC02LjAKU05SX01BWF9EQjogZmxvYXQgPSAxMC4wCiIiIkJMVUVQUklOVCA1LjM6IGFkYXB0ZXJfbm9pc2UgdHJhaW5zIG9uIFNOUiB1bmlmb3JtIC02IHRvICsxMCBkQi4iIiIKClNFVkVSRV9TTlJfREI6IGZsb2F0ID0gLTQuMAoiIiIKQkxVRVBSSU5UIDcuNSBob2xkb3V0IDM6IFNOUiBiZWxvdyAtNCBkQiBpcyBhIHNldmVyaXR5IGhvbGRvdXQsIGtlcHQgdG8gMTAlIG9mCm5vaXNlIHRyYWluaW5nIHNhbXBsZXMgYW5kIHByb2JlZCBpbiBldmFsdWF0aW9uLgoiIiIKClNFVkVSRV9GUkFDVElPTjogZmxvYXQgPSAwLjEwCiIiIkZyYWN0aW9uIG9mIHRyYWluaW5nIGRyYXdzIGFsbG93ZWQgaW50byB0aGUgc2V2ZXJlIChTTlIgPCAtNCBkQikgYmFuZC4iIiIKCkVQUzogZmxvYXQgPSAxZS0xMAoiIiJFbmVyZ3kgZ3VhcmQuIE1hdGNoZXMgZXZhbC9tZXRyaWNzLnB5IEVQUyBzY2FsZS4iIiIKCgpkZWYgc2FtcGxlX3NucigKICAgIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwKICAgIGFsbG93X3NldmVyZTogYm9vbCA9IFRydWUsCiAgICBzZXZlcmVfZnJhY3Rpb246IGZsb2F0ID0gU0VWRVJFX0ZSQUNUSU9OLAopIC0+IGZsb2F0OgogICAgIiIiCiAgICBEcmF3IGEgdHJhaW5pbmcgU05SLCBob25vdXJpbmcgdGhlIHNldmVyaXR5IGhvbGRvdXQuCgogICAgTWlycm9ycyBkYXRhLnJpcl9iYW5rLnNhbXBsZV90NjAgZm9yIHRoZSBub2lzZSBheGlzLiBCTFVFUFJJTlQgNy41CiAgICBob2xkb3V0IDMga2VlcHMgU05SIGJlbG93IC00IGRCIHJhcmUgaW4gdHJhaW5pbmcgc28gZXZhbHVhdGlvbiBhdCBsb3cgU05SCiAgICBtZWFzdXJlcyBleHRyYXBvbGF0aW9uLgoKICAgIEFyZ3M6CiAgICAgICAgcm5nOiBTZWVkZWQgZ2VuZXJhdG9yLgogICAgICAgIGFsbG93X3NldmVyZTogV2hlbiBGYWxzZSwgbmV2ZXIgZHJhd3MgYmVsb3cgU0VWRVJFX1NOUl9EQi4KICAgICAgICBzZXZlcmVfZnJhY3Rpb246IFByb2JhYmlsaXR5IG9mIGRyYXdpbmcgZnJvbSB0aGUgc2V2ZXJlIGJhbmQuCgogICAgUmV0dXJuczoKICAgICAgICBTTlIgaW4gZEIsIHdpdGhpbiBbU05SX01JTl9EQiwgU05SX01BWF9EQl0uCiAgICAiIiIKICAgIGlmIG5vdCBhbGxvd19zZXZlcmU6CiAgICAgICAgcmV0dXJuIGZsb2F0KHJuZy51bmlmb3JtKFNFVkVSRV9TTlJfREIsIFNOUl9NQVhfREIpKQogICAgaWYgcm5nLnJhbmRvbSgpIDwgc2V2ZXJlX2ZyYWN0aW9uOgogICAgICAgIHJldHVybiBmbG9hdChybmcudW5pZm9ybShTTlJfTUlOX0RCLCBTRVZFUkVfU05SX0RCKSkKICAgIHJldHVybiBmbG9hdChybmcudW5pZm9ybShTRVZFUkVfU05SX0RCLCBTTlJfTUFYX0RCKSkKCgpkZWYgbWFrZV93ZXRfcmVmZXJlbmNlKAogICAgZHJ5OiBucC5uZGFycmF5LAogICAgcmlyOiBucC5uZGFycmF5LAogICAgbl9wZWFrOiBpbnQsCiAgICBvZmZzZXQ6IGludCA9IFdFVF9SRUZFUkVOQ0VfT0ZGU0VUX1NBTVBMRVMsCiAgICB0YXJnZXRfbGVuZ3RoOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBucC5uZGFycmF5OgogICAgIiIiCiAgICBDb252b2x2ZSBhIGRyeSBzb3VyY2Ugd2l0aCBhIHRydW5jYXRlZCBSSVIgdG8gbWFrZSB0aGUgd2V0IHJlZmVyZW5jZS4KCiAgICBUaGUgUklSIGlzIGN1dCBhdCBuX3BlYWsgKyBvZmZzZXQgYmVmb3JlIGNvbnZvbHV0aW9uLCBzbyB0aGUgcmVmZXJlbmNlCiAgICBjb250YWlucyB0aGUgZGlyZWN0IHBhdGggYW5kIGVhcmx5IHJlZmxlY3Rpb25zIGJ1dCBub3QgdGhlIGxhdGUgdGFpbC4gU2VlCiAgICB0aGUgbW9kdWxlIGRvY3N0cmluZyBmb3Igd2h5IHRoaXMgaXMgdGhlIGNvcnJlY3QgdGFyZ2V0LgoKICAgIEFyZ3M6CiAgICAgICAgZHJ5OiBDbGVhbiBzb3VyY2Ugd2F2ZWZvcm0gW1RdLgogICAgICAgIHJpcjogRnVsbCByb29tIGltcHVsc2UgcmVzcG9uc2UgW1JdLgogICAgICAgIG5fcGVhazogSW5kZXggb2YgdGhlIGRpcmVjdC1wYXRoIHBlYWsgaW4gcmlyIChmcm9tIFJpclJlY29yZC5uX3BlYWspLgogICAgICAgIG9mZnNldDogU2FtcGxlcyBrZXB0IHBhc3QgdGhlIHBlYWsuIERlZmF1bHRzIHRvIDUxMiAoNjQgbXMgYXQgOCBrSHopLgogICAgICAgIHRhcmdldF9sZW5ndGg6IENyb3Agb3IgcGFkIHRoZSByZXN1bHQgdG8gZXhhY3RseSB0aGlzIG1hbnkgc2FtcGxlcy4KICAgICAgICAgICAgRGVmYXVsdHMgdG8gbGVuKGRyeSksIHdoaWNoIGtlZXBzIHRoZSByZWZlcmVuY2UgdGltZS1hbGlnbmVkIHdpdGgKICAgICAgICAgICAgdGhlIGRyeSBzb3VyY2Ugc28gU0ktU0RSIGlzIG1lYW5pbmdmdWwuCgogICAgUmV0dXJuczoKICAgICAgICBXZXQgcmVmZXJlbmNlIFt0YXJnZXRfbGVuZ3RoXSBmbG9hdDMyLgogICAgIiIiCiAgICBkID0gbnAuYXNhcnJheShkcnksIGR0eXBlPW5wLmZsb2F0MzIpLnNxdWVlemUoKQogICAgaCA9IG5wLmFzYXJyYXkocmlyLCBkdHlwZT1ucC5mbG9hdDMyKS5zcXVlZXplKCkKICAgIGlmIGQubmRpbSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJkcnkgbXVzdCBiZSAxLUQsIGdvdCBzaGFwZSB7ZC5zaGFwZX0iKQogICAgaWYgaC5uZGltICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInJpciBtdXN0IGJlIDEtRCwgZ290IHNoYXBlIHtoLnNoYXBlfSIpCgogICAgY3V0ID0gbWluKG5fcGVhayArIG9mZnNldCwgaC5zaGFwZVswXSkKICAgIGlmIGN1dCA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ0cnVuY2F0aW9uIGluZGV4IHtjdXR9IGlzIG5vdCBwb3NpdGl2ZSAobl9wZWFrPXtuX3BlYWt9KSIpCiAgICBoX2Vhcmx5ID0gaFs6Y3V0XQoKICAgIHdldCA9IHNpZ25hbC5mZnRjb252b2x2ZShkLCBoX2Vhcmx5LCBtb2RlPSJmdWxsIikKICAgIGxlbmd0aCA9IGludCh0YXJnZXRfbGVuZ3RoKSBpZiB0YXJnZXRfbGVuZ3RoIGlzIG5vdCBOb25lIGVsc2UgZC5zaGFwZVswXQogICAgcmV0dXJuIF9maXRfbGVuZ3RoKHdldCwgbGVuZ3RoKS5hc3R5cGUobnAuZmxvYXQzMikKCgpkZWYgbWFrZV93ZXRfbWl4dHVyZSgKICAgIGRyeTogbnAubmRhcnJheSwKICAgIHJpcjogbnAubmRhcnJheSwKICAgIHRhcmdldF9sZW5ndGg6IGludCB8IE5vbmUgPSBOb25lLAopIC0+IG5wLm5kYXJyYXk6CiAgICAiIiIKICAgIENvbnZvbHZlIGEgZHJ5IHNvdXJjZSB3aXRoIHRoZSAqKmZ1bGwqKiBSSVIgdG8gbWFrZSB0aGUgb2JzZXJ2ZWQgc2lnbmFsLgoKICAgIFRoZSBtaXh0dXJlIHRoZSBzeXN0ZW0gaGVhcnMgY2FycmllcyB0aGUgY29tcGxldGUgcmV2ZXJiZXJhdGlvbiBpbmNsdWRpbmcKICAgIHRoZSBsYXRlIHRhaWwuIE9ubHkgdGhlICpyZWZlcmVuY2UqIGlzIHRydW5jYXRlZC4gVXNpbmcgdGhlIHRydW5jYXRlZCBSSVIKICAgIGZvciBib3RoIHdvdWxkIG1lYW4gdGhlIHN5c3RlbSBuZXZlciBzZWVzIHRoZSB0YWlsIGl0IG11c3QgYmUgcm9idXN0IHRvLgoKICAgIEFyZ3M6CiAgICAgICAgZHJ5OiBDbGVhbiBzb3VyY2Ugd2F2ZWZvcm0gW1RdLgogICAgICAgIHJpcjogRnVsbCByb29tIGltcHVsc2UgcmVzcG9uc2UgW1JdLgogICAgICAgIHRhcmdldF9sZW5ndGg6IE91dHB1dCBsZW5ndGguIERlZmF1bHRzIHRvIGxlbihkcnkpLgoKICAgIFJldHVybnM6CiAgICAgICAgUmV2ZXJiZXJhbnQgb2JzZXJ2YXRpb24gW3RhcmdldF9sZW5ndGhdIGZsb2F0MzIuCiAgICAiIiIKICAgIGQgPSBucC5hc2FycmF5KGRyeSwgZHR5cGU9bnAuZmxvYXQzMikuc3F1ZWV6ZSgpCiAgICBoID0gbnAuYXNhcnJheShyaXIsIGR0eXBlPW5wLmZsb2F0MzIpLnNxdWVlemUoKQogICAgd2V0ID0gc2lnbmFsLmZmdGNvbnZvbHZlKGQsIGgsIG1vZGU9ImZ1bGwiKQogICAgbGVuZ3RoID0gaW50KHRhcmdldF9sZW5ndGgpIGlmIHRhcmdldF9sZW5ndGggaXMgbm90IE5vbmUgZWxzZSBkLnNoYXBlWzBdCiAgICByZXR1cm4gX2ZpdF9sZW5ndGgod2V0LCBsZW5ndGgpLmFzdHlwZShucC5mbG9hdDMyKQoKCmRlZiBhcHBseV9yZXZlcmIoCiAgICBtaXh0dXJlOiBDYWxtU2VwTWl4dHVyZSwKICAgIHJpcl9iYW5rOiBSaXJCYW5rLAogICAgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAogICAgdDYwX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICBhbGxvd19zZXZlcmU6IGJvb2wgPSBUcnVlLAogICAgcmVjb3JkOiBSaXJSZWNvcmQgfCBOb25lID0gTm9uZSwKKSAtPiBDYWxtU2VwTWl4dHVyZToKICAgICIiIgogICAgUmV2ZXJiZXJhdGUgZXZlcnkgc3RlbSB3aXRoIG9uZSBzaGFyZWQgcm9vbSwgYW5kIHJlYnVpbGQgdGhlIG1peHR1cmUuCgogICAgQWxsIHNwZWFrZXJzIHNoYXJlIHRoZSBzYW1lIFJJUiBiZWNhdXNlIHRoZXkgYXJlIGluIHRoZSBzYW1lIHJvb20uIERyYXdpbmcKICAgIGEgc2VwYXJhdGUgUklSIHBlciBzcGVha2VyIHdvdWxkIG1vZGVsIGFuIGFjb3VzdGljYWxseSBpbXBvc3NpYmxlIHNjZW5lIGFuZAogICAgd291bGQgZ2l2ZSB0aGUgbW9kZWwgYSBzcHVyaW91cyBwZXItc3BlYWtlciBjdWUgdG8gc2VwYXJhdGUgb24uCgogICAgVGhlIHJldHVybmVkIG1peHR1cmUncyByZWZlcmVuY2VzIGFyZSAqKndldCoqICh0cnVuY2F0ZWQgUklSKSBhbmQgaXRzCiAgICBvYnNlcnZhdGlvbiBpcyAqKmZ1bGx5IHJldmVyYmVyYW50KiogKGZ1bGwgUklSKS4gVGhlIHJlY2lwZSByZWNvcmRzIHRoZQogICAgYWNoaWV2ZWQgVDYwIGFuZCB0aGUgUklSIHBhdGguCgogICAgQXJnczoKICAgICAgICBtaXh0dXJlOiBBIGNsZWFuIENhbG1TZXBNaXh0dXJlLgogICAgICAgIHJpcl9iYW5rOiBMb2FkZWQgc2ltdWxhdGVkIFJJUiBiYW5rLgogICAgICAgIHJuZzogU2VlZGVkIGdlbmVyYXRvci4KICAgICAgICB0NjBfczogVGFyZ2V0IFQ2MC4gRHJhd24gdmlhIHNhbXBsZV90NjAgd2hlbiBOb25lLgogICAgICAgIGFsbG93X3NldmVyZTogUGFzc2VkIHRvIHNhbXBsZV90NjAgZm9yIHRoZSBzZXZlcml0eSBob2xkb3V0LgogICAgICAgIHJlY29yZDogVXNlIHRoaXMgZXhhY3QgUklSIGluc3RlYWQgb2YgZHJhd2luZyBvbmUuIFNldCBieSB0aGUgZml4ZWQKICAgICAgICAgICAgZXZhbHVhdGlvbiBnZW5lcmF0b3Igc28gYW4gZXZhbCBjZWxsIHBpbnMgaXRzIHJvb21zLgoKICAgIFJldHVybnM6CiAgICAgICAgQSBuZXcgQ2FsbVNlcE1peHR1cmUuIFRoZSBpbnB1dCBpcyBub3QgbW9kaWZpZWQuCiAgICAiIiIKICAgIGlmIHJlY29yZCBpcyBOb25lOgogICAgICAgIHRhcmdldCA9IHQ2MF9zIGlmIHQ2MF9zIGlzIG5vdCBOb25lIGVsc2Ugc2FtcGxlX3Q2MChybmcsIGFsbG93X3NldmVyZT1hbGxvd19zZXZlcmUpCiAgICAgICAgcmVjb3JkID0gcmlyX2Jhbmsuc2FtcGxlKHRhcmdldCkKICAgIHJpciA9IHJpcl9iYW5rLmxvYWQocmVjb3JkKQoKICAgIHJlZnMgPSBtaXh0dXJlLnJlZmVyZW5jZXMKICAgIGxlbmd0aCA9IHJlZnMuc2hhcGVbMV0KCiAgICB3ZXRfcmVmcyA9IG5wLnN0YWNrKAogICAgICAgIFttYWtlX3dldF9yZWZlcmVuY2UocmVmc1tpXSwgcmlyLCByZWNvcmQubl9wZWFrLCB0YXJnZXRfbGVuZ3RoPWxlbmd0aCkgZm9yIGkgaW4gcmFuZ2UocmVmcy5zaGFwZVswXSldLAogICAgICAgIGF4aXM9MCwKICAgICkKICAgIHdldF9vYnMgPSBucC5zdGFjaygKICAgICAgICBbbWFrZV93ZXRfbWl4dHVyZShyZWZzW2ldLCByaXIsIHRhcmdldF9sZW5ndGg9bGVuZ3RoKSBmb3IgaSBpbiByYW5nZShyZWZzLnNoYXBlWzBdKV0sCiAgICAgICAgYXhpcz0wLAogICAgKQogICAgb2JzZXJ2ZWQgPSB3ZXRfb2JzLnN1bShheGlzPTApLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIHJlY2lwZSA9IHJlcGxhY2UoCiAgICAgICAgbWl4dHVyZS5yZWNpcGUsCiAgICAgICAgdDYwX3M9cmVjb3JkLnQ2MF9hY2hpZXZlZF9zLAogICAgICAgIHJpcl9maWxlPXJlY29yZC5wYXRoLAogICAgKQogICAgc2FtcGxlID0gTWl4dHVyZVNhbXBsZSgKICAgICAgICBtaXh0dXJlPW9ic2VydmVkLAogICAgICAgIHJlZmVyZW5jZXM9d2V0X3JlZnMsCiAgICAgICAgc2FtcGxlX3JhdGU9bWl4dHVyZS5zYW1wbGUuc2FtcGxlX3JhdGUsCiAgICAgICAgdXR0ZXJhbmNlX2lkPW1peHR1cmUuc2FtcGxlLnV0dGVyYW5jZV9pZCwKICAgICkKICAgIHJldHVybiBDYWxtU2VwTWl4dHVyZShzYW1wbGU9c2FtcGxlLCByZWNpcGU9cmVjaXBlKQoKCmRlZiBhcHBseV9ub2lzZSgKICAgIG1peHR1cmU6IENhbG1TZXBNaXh0dXJlLAogICAgbm9pc2U6IG5wLm5kYXJyYXksCiAgICBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsCiAgICBzbnJfZGI6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICBhbGxvd19zZXZlcmU6IGJvb2wgPSBUcnVlLAogICAgbm9pc2VfZmlsZTogc3RyIHwgTm9uZSA9IE5vbmUsCikgLT4gQ2FsbVNlcE1peHR1cmU6CiAgICAiIiIKICAgIEFkZCBiYWNrZ3JvdW5kIG5vaXNlIGF0IGEgZHJhd24gU05SLCBsZWF2aW5nIHRoZSByZWZlcmVuY2VzIHVudG91Y2hlZC4KCiAgICBUaGUgcmVmZXJlbmNlcyBkbyBub3QgY2hhbmdlOiBub2lzZSBpcyBub3QgYSBzcGVha2VyLCBzbyByZW1vdmluZyBpdCBpcwogICAgcGFydCBvZiB0aGUgdGFzaywgbm90IHBhcnQgb2YgdGhlIHRhcmdldC4gVGhlIFNOUiBpcyBjb21wdXRlZCBhZ2FpbnN0IHRoZQogICAgc3BlZWNoIG1peHR1cmUncyBlbmVyZ3kgc28gdGhlIGxhYmVsIG1lYW5zIHdoYXQgaXQgc2F5cy4KCiAgICBBcmdzOgogICAgICAgIG1peHR1cmU6IEEgQ2FsbVNlcE1peHR1cmUsIGNsZWFuIG9yIGFscmVhZHkgcmV2ZXJiZXJhdGVkLgogICAgICAgIG5vaXNlOiBOb2lzZSB3YXZlZm9ybSBbVF9uXSBhdCB0aGUgbWl4dHVyZSdzIHJhdGUuIExvb3BlZCBvciBjcm9wcGVkIHRvCiAgICAgICAgICAgIHRoZSBtaXh0dXJlJ3MgbGVuZ3RoLgogICAgICAgIHJuZzogU2VlZGVkIGdlbmVyYXRvciwgdXNlZCBmb3IgdGhlIFNOUiBkcmF3IGFuZCB0aGUgbm9pc2Ugb2Zmc2V0LgogICAgICAgIHNucl9kYjogVGFyZ2V0IFNOUi4gRHJhd24gdmlhIHNhbXBsZV9zbnIgd2hlbiBOb25lLgogICAgICAgIGFsbG93X3NldmVyZTogUGFzc2VkIHRvIHNhbXBsZV9zbnIgZm9yIHRoZSBzZXZlcml0eSBob2xkb3V0LgogICAgICAgIG5vaXNlX2ZpbGU6IFBhdGggcmVjb3JkZWQgaW50byB0aGUgcmVjaXBlIGZvciB0cmFjZWFiaWxpdHkuCgogICAgUmV0dXJuczoKICAgICAgICBBIG5ldyBDYWxtU2VwTWl4dHVyZSB3aXRoIG5vaXNlIGFkZGVkIHRvIHRoZSBvYnNlcnZhdGlvbi4KICAgICIiIgogICAgdGFyZ2V0X3NuciA9IHNucl9kYiBpZiBzbnJfZGIgaXMgbm90IE5vbmUgZWxzZSBzYW1wbGVfc25yKHJuZywgYWxsb3dfc2V2ZXJlPWFsbG93X3NldmVyZSkKCiAgICBzcGVlY2ggPSBtaXh0dXJlLm1peHR1cmUKICAgIGxlbmd0aCA9IHNwZWVjaC5zaGFwZVswXQogICAgbm9pc2VfZml0ID0gX2xvb3Bfb3JfY3JvcChucC5hc2FycmF5KG5vaXNlLCBkdHlwZT1ucC5mbG9hdDMyKS5zcXVlZXplKCksIGxlbmd0aCwgcm5nKQoKICAgIHNwZWVjaF9wb3dlciA9IGZsb2F0KG5wLm1lYW4oc3BlZWNoKioyKSkKICAgIG5vaXNlX3Bvd2VyID0gZmxvYXQobnAubWVhbihub2lzZV9maXQqKjIpKQogICAgaWYgbm9pc2VfcG93ZXIgPCBFUFM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibm9pc2Ugc2VnbWVudCBpcyBzaWxlbnQ7IGNhbm5vdCBzY2FsZSBpdCB0byBhIHRhcmdldCBTTlIiKQogICAgaWYgc3BlZWNoX3Bvd2VyIDwgRVBTOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNwZWVjaCBtaXh0dXJlIGlzIHNpbGVudDsgU05SIGlzIHVuZGVmaW5lZCIpCgogICAgIyBzY2FsZSBzbyB0aGF0IDEwKmxvZzEwKHNwZWVjaF9wb3dlciAvIChub2lzZV9wb3dlciAqIHNjYWxlXjIpKSA9PSB0YXJnZXRfc25yCiAgICBzY2FsZSA9IGZsb2F0KG5wLnNxcnQoc3BlZWNoX3Bvd2VyIC8gKG5vaXNlX3Bvd2VyICogMTAuMCAqKiAodGFyZ2V0X3NuciAvIDEwLjApKSkpCiAgICBub2lzeSA9IChzcGVlY2ggKyBub2lzZV9maXQgKiBzY2FsZSkuYXN0eXBlKG5wLmZsb2F0MzIpCgogICAgcmVjaXBlID0gcmVwbGFjZShtaXh0dXJlLnJlY2lwZSwgc25yX2RiPXRhcmdldF9zbnIsIG5vaXNlX2ZpbGU9bm9pc2VfZmlsZSkKICAgIHNhbXBsZSA9IE1peHR1cmVTYW1wbGUoCiAgICAgICAgbWl4dHVyZT1ub2lzeSwKICAgICAgICByZWZlcmVuY2VzPW1peHR1cmUucmVmZXJlbmNlcywKICAgICAgICBzYW1wbGVfcmF0ZT1taXh0dXJlLnNhbXBsZS5zYW1wbGVfcmF0ZSwKICAgICAgICB1dHRlcmFuY2VfaWQ9bWl4dHVyZS5zYW1wbGUudXR0ZXJhbmNlX2lkLAogICAgKQogICAgcmV0dXJuIENhbG1TZXBNaXh0dXJlKHNhbXBsZT1zYW1wbGUsIHJlY2lwZT1yZWNpcGUpCgoKZGVmIGFwcGx5X2NvZGVjKAogICAgbWl4dHVyZTogQ2FsbVNlcE1peHR1cmUsCiAgICBjb2RlY19uYW1lOiBzdHIsCiAgICBiaXRyYXRlX2JwczogaW50LAogICAgdG1wX2Rpcjogc3RyIHwgUGF0aCB8IE5vbmUgPSBOb25lLAopIC0+IENhbG1TZXBNaXh0dXJlOgogICAgIiIiCiAgICBSb3VuZC10cmlwIHRoZSBvYnNlcnZhdGlvbiB0aHJvdWdoIGEgbG9zc3kgY29kZWMsIGxlYXZpbmcgcmVmZXJlbmNlcyBjbGVhbi4KCiAgICBMaWtlIG5vaXNlLCBjb2RlYyBkYW1hZ2UgaXMgc29tZXRoaW5nIHRoZSBzeXN0ZW0gbXVzdCB1bmRvLCBzbyBpdCBpcyBhcHBsaWVkCiAgICB0byB0aGUgb2JzZXJ2YXRpb24gb25seS4gZGF0YS9jb2RlY19hdWdtZW50YXRpb24ucHkgb3ducyB0aGUgZmZtcGVnIGNhbGxzOwogICAgdGhpcyB3cmFwcGVyIGFkYXB0cyB0aGVtIHRvIHRoZSBDYWxtU2VwTWl4dHVyZSB0eXBlIGFuZCByZWNvcmRzIHRoZSBsYWJlbHMuCgogICAgQXJnczoKICAgICAgICBtaXh0dXJlOiBBIENhbG1TZXBNaXh0dXJlIGF0IDgga0h6LgogICAgICAgIGNvZGVjX25hbWU6IE9uZSBvZiAib3B1cyIsICJhYWMiLCAiYW1yLW5iIiwgImFtci13YiIuCiAgICAgICAgYml0cmF0ZV9icHM6IFRhcmdldCBiaXRyYXRlIGluIGJpdHMgcGVyIHNlY29uZC4KICAgICAgICB0bXBfZGlyOiBTY3JhdGNoIGRpcmVjdG9yeSBmb3IgdGhlIGVuY29kZS9kZWNvZGUgcm91bmQgdHJpcC4KCiAgICBSZXR1cm5zOgogICAgICAgIEEgbmV3IENhbG1TZXBNaXh0dXJlIHdpdGggYSBjb2RlYy1kYW1hZ2VkIG9ic2VydmF0aW9uLgogICAgIiIiCiAgICBmcm9tIGRhdGEuY29kZWNfYXVnbWVudGF0aW9uIGltcG9ydCBhcHBseV9jb2RlY19yb3VuZHRyaXAKCiAgICBkYW1hZ2VkID0gYXBwbHlfY29kZWNfcm91bmR0cmlwKAogICAgICAgIGF1ZGlvPW1peHR1cmUubWl4dHVyZSwKICAgICAgICBzYW1wbGVfcmF0ZT1taXh0dXJlLnNhbXBsZS5zYW1wbGVfcmF0ZSwKICAgICAgICBjb2RlYz1jb2RlY19uYW1lLAogICAgICAgIGJpdHJhdGVfYnBzPWJpdHJhdGVfYnBzLAogICAgICAgIHRtcF9kaXI9dG1wX2RpciwKICAgICkKICAgIGRhbWFnZWQgPSBfZml0X2xlbmd0aChkYW1hZ2VkLCBtaXh0dXJlLm1peHR1cmUuc2hhcGVbMF0pLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIHJlY2lwZSA9IHJlcGxhY2UobWl4dHVyZS5yZWNpcGUsIGNvZGVjX25hbWU9Y29kZWNfbmFtZSwgY29kZWNfYml0cmF0ZV9icHM9aW50KGJpdHJhdGVfYnBzKSkKICAgIHNhbXBsZSA9IE1peHR1cmVTYW1wbGUoCiAgICAgICAgbWl4dHVyZT1kYW1hZ2VkLAogICAgICAgIHJlZmVyZW5jZXM9bWl4dHVyZS5yZWZlcmVuY2VzLAogICAgICAgIHNhbXBsZV9yYXRlPW1peHR1cmUuc2FtcGxlLnNhbXBsZV9yYXRlLAogICAgICAgIHV0dGVyYW5jZV9pZD1taXh0dXJlLnNhbXBsZS51dHRlcmFuY2VfaWQsCiAgICApCiAgICByZXR1cm4gQ2FsbVNlcE1peHR1cmUoc2FtcGxlPXNhbXBsZSwgcmVjaXBlPXJlY2lwZSkKCgpkZWYgX2ZpdF9sZW5ndGgoeDogbnAubmRhcnJheSwgbGVuZ3RoOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDcm9wIG9yIHplcm8tcGFkIGEgMS1EIHNpZ25hbCB0byBleGFjdGx5IGBsZW5ndGhgIHNhbXBsZXMuIiIiCiAgICB0ID0geC5zaGFwZVswXQogICAgaWYgdCA9PSBsZW5ndGg6CiAgICAgICAgcmV0dXJuIHgKICAgIGlmIHQgPiBsZW5ndGg6CiAgICAgICAgcmV0dXJuIHhbOmxlbmd0aF0KICAgIHJldHVybiBucC5wYWQoeCwgKDAsIGxlbmd0aCAtIHQpKQoKCmRlZiBfbG9vcF9vcl9jcm9wKG5vaXNlOiBucC5uZGFycmF5LCBsZW5ndGg6IGludCwgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yKSAtPiBucC5uZGFycmF5OgogICAgIiIiCiAgICBGaXQgYSBub2lzZSBjbGlwIHRvIGBsZW5ndGhgIGJ5IGxvb3BpbmcgaXQgb3IgY3JvcHBpbmcgYSByYW5kb20gd2luZG93LgoKICAgIEEgcmFuZG9tIG9mZnNldCBpcyB1c2VkIHJhdGhlciB0aGFuIGFsd2F5cyBzdGFydGluZyBhdCBzYW1wbGUgMCwgc28gYSBsb25nCiAgICBub2lzZSBmaWxlIGNvbnRyaWJ1dGVzIG1hbnkgZGlzdGluY3Qgc2VnbWVudHMgYWNyb3NzIGFuIGVwb2NoIGluc3RlYWQgb2YKICAgIHRoZSBzYW1lIG9wZW5pbmcgZXZlcnkgdGltZS4KICAgICIiIgogICAgbiA9IG5vaXNlLnNoYXBlWzBdCiAgICBpZiBuID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibm9pc2UgY2xpcCBpcyBlbXB0eSIpCiAgICBpZiBuIDwgbGVuZ3RoOgogICAgICAgIHJlcHMgPSBpbnQobnAuY2VpbChsZW5ndGggLyBuKSkKICAgICAgICByZXR1cm4gbnAudGlsZShub2lzZSwgcmVwcylbOmxlbmd0aF0KICAgIHN0YXJ0ID0gaW50KHJuZy5pbnRlZ2VycygwLCBuIC0gbGVuZ3RoICsgMSkpCiAgICByZXR1cm4gbm9pc2Vbc3RhcnQgOiBzdGFydCArIGxlbmd0aF0KCgpkZWYgZGVzY3JpYmVfY29uZGl0aW9uKHJlY2lwZTogTWl4dHVyZVJlY2lwZSkgLT4gc3RyOgogICAgIiIiCiAgICBOYW1lIHRoZSBjb25kaXRpb24gY2VsbCBhIHJlY2lwZSBiZWxvbmdzIHRvLgoKICAgIFVzZWQgdG8ga2V5IHRoZSBldmFsdWF0aW9uIG1hdHJpeCAoQkxVRVBSSU5UIDkuNCkgYW5kIHRvIGNoZWNrIHRoZQogICAgY29uZGl0aW9uLWNvbWJpbmF0aW9uIGhvbGRvdXQgKDcuNSwgaG9sZG91dCAyKS4KCiAgICBSZXR1cm5zOgogICAgICAgIE9uZSBvZiAiY2xlYW4iLCAicmV2ZXJiIiwgIm5vaXNlIiwgImNvZGVjIiwgInJldmVyYitub2lzZSIsCiAgICAgICAgInJldmVyYitjb2RlYyIsICJub2lzZStjb2RlYyIsICJhbGwtdGhyZWUiLgogICAgIiIiCiAgICBwYXJ0czogbGlzdFtzdHJdID0gW10KICAgIGlmIHJlY2lwZS50NjBfcyBpcyBub3QgTm9uZSBhbmQgcmVjaXBlLnQ2MF9zID4gMC4wOgogICAgICAgIHBhcnRzLmFwcGVuZCgicmV2ZXJiIikKICAgIGlmIHJlY2lwZS5zbnJfZGIgaXMgbm90IE5vbmU6CiAgICAgICAgcGFydHMuYXBwZW5kKCJub2lzZSIpCiAgICBpZiByZWNpcGUuY29kZWNfbmFtZSBpcyBub3QgTm9uZToKICAgICAgICBwYXJ0cy5hcHBlbmQoImNvZGVjIikKCiAgICBpZiBub3QgcGFydHM6CiAgICAgICAgcmV0dXJuICJjbGVhbiIKICAgIGlmIGxlbihwYXJ0cykgPT0gMzoKICAgICAgICByZXR1cm4gImFsbC10aHJlZSIKICAgIHJldHVybiAiKyIuam9pbihwYXJ0cykKCgpIRUxEX09VVF9DT01CSU5BVElPTlM6IGZyb3plbnNldFtzdHJdID0gZnJvemVuc2V0KHsicmV2ZXJiK2NvZGVjIiwgIm5vaXNlK2NvZGVjIn0pCiIiIgpCTFVFUFJJTlQgNy41IGhvbGRvdXQgMjogdGhlc2UgY29tYmluYXRpb25zIG5ldmVyIGFwcGVhciBpbiBnYXRlIG9yIGpvaW50CnRyYWluaW5nIGFuZCBleGlzdCBvbmx5IGluIHRoZSBldmFsdWF0aW9uIG1hdHJpeCwgc28gY29tcG9zaXRpb25hbApnZW5lcmFsaXNhdGlvbiBpcyBtZWFzdXJlZCByYXRoZXIgdGhhbiBhc3N1bWVkLiBhc3NlcnRfbm90X2hlbGRfb3V0IGVuZm9yY2VzIGl0LgoiIiIKCgpkZWYgYXNzZXJ0X25vdF9oZWxkX291dChyZWNpcGU6IE1peHR1cmVSZWNpcGUpIC0+IE5vbmU6CiAgICAiIiIKICAgIFJhaXNlIGlmIGEgcmVjaXBlIGJlbG9uZ3MgdG8gYSBoZWxkLW91dCBjb21iaW5hdGlvbiBjZWxsLgoKICAgIENhbGxlZCBieSB0aGUgZ2F0ZSBhbmQgam9pbnQtcG9saXNoIHRyYWluaW5nIGRhdGEgcGlwZWxpbmVzLiBBIGhlbGQtb3V0CiAgICBjb21iaW5hdGlvbiByZWFjaGluZyB0cmFpbmluZyBzaWxlbnRseSBpbnZhbGlkYXRlcyB0aGUgY29tcG9zaXRpb25hbAogICAgZ2VuZXJhbGlzYXRpb24gY2xhaW0gaW4gQkxVRVBSSU5UIDkuNSBhbmFseXNpcyAzLCBzbyB0aGlzIHJhaXNlcy4KICAgICIiIgogICAgY2VsbCA9IGRlc2NyaWJlX2NvbmRpdGlvbihyZWNpcGUpCiAgICBpZiBjZWxsIGluIEhFTERfT1VUX0NPTUJJTkFUSU9OUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmInJlY2lwZSBpcyBpbiBoZWxkLW91dCBjb21iaW5hdGlvbiBjZWxsIHtjZWxsIXJ9OyBpdCBtdXN0IG5vdCBlbnRlciBnYXRlIG9yICIKICAgICAgICAgICAgZiJqb2ludCB0cmFpbmluZyAoQkxVRVBSSU5UIDcuNSBob2xkb3V0IDIpLiBIZWxkIG91dDoge3NvcnRlZChIRUxEX09VVF9DT01CSU5BVElPTlMpfSIKICAgICAgICApCg=='))
open(f'{PROJ}/data/mixer_stub.py','wb').write(base64.b64decode('IiIiCk1pbmltYWwgbWl4ZXIgc3R1YiBmb3IgUGhhc2UgMCBiYXNlbGluZSBydW5zLgoKRGV2IEEgd2lsbCByZXBsYWNlIHRoaXMgd2l0aCB0aGUgZnVsbCBkeW5hbWljIG1peGVyIChgZGF0YS9taXhlci5weWApLgpUaGlzIHN0dWIgbG9hZHMgcHJlLW1peGVkIExpYnJpM01peCB0ZXN0IGZpbGVzIGZyb20gZGlzayBhbmQgeWllbGRzCihtaXh0dXJlLCByZWZlcmVuY2Vfc3RlbXMsIHNhbXBsZV9yYXRlKSB0dXBsZXMgZm9yIHRoZSBiYXNlbGluZSBydW5uZXIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBzb3VuZGZpbGUgYXMgc2YKCgpAZGF0YWNsYXNzCmNsYXNzIE1peHR1cmVTYW1wbGU6CiAgICAiIiJBIHNpbmdsZSBtaXh0dXJlIHdpdGggZ3JvdW5kLXRydXRoIGNsZWFuIHN0ZW1zLiIiIgoKICAgIG1peHR1cmU6IG5wLm5kYXJyYXkKICAgICIiIk1vbm8gbWl4dHVyZSB3YXZlZm9ybSwgc2hhcGUgW1RdLiIiIgoKICAgIHJlZmVyZW5jZXM6IG5wLm5kYXJyYXkKICAgICIiIkNsZWFuIHNvdXJjZSB3YXZlZm9ybXMsIHNoYXBlIFtOLCBUXS4iIiIKCiAgICBzYW1wbGVfcmF0ZTogaW50CiAgICB1dHRlcmFuY2VfaWQ6IHN0cgoKCmRlZiBfbG9hZF93YXYocGF0aDogUGF0aCkgLT4gdHVwbGVbbnAubmRhcnJheSwgaW50XToKICAgIGF1ZGlvLCBzciA9IHNmLnJlYWQoc3RyKHBhdGgpLCBkdHlwZT0iZmxvYXQzMiIsIGFsd2F5c18yZD1UcnVlKQogICAgaWYgYXVkaW8uc2hhcGVbMV0gPiAxOgogICAgICAgIGF1ZGlvID0gYXVkaW8ubWVhbihheGlzPTEsIGtlZXBkaW1zPVRydWUpCiAgICByZXR1cm4gYXVkaW9bOiwgMF0sIHNyCgoKZGVmIGRpc2NvdmVyX2xpYnJpbWl4X3NhbXBsZXMoCiAgICBkYXRhX3Jvb3Q6IHN0ciB8IFBhdGgsCiAgICBzdWJzZXQ6IHN0ciA9ICJ0ZXN0IiwKICAgIG1heF9zYW1wbGVzOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBsaXN0W01peHR1cmVTYW1wbGVdOgogICAgIiIiCiAgICBEaXNjb3ZlciBMaWJyaU5NaXggc2FtcGxlcyAoTj0yLi41KSBmcm9tIGEgc3RhbmRhcmQgTGlicmlNaXggZGlyZWN0b3J5IGxheW91dC4KCiAgICBUaGUgbnVtYmVyIG9mIHNwZWFrZXJzIGlzIGRldGVjdGVkIGF1dG9tYXRpY2FsbHkgYnkgcHJvYmluZyB3aGljaCBzTi8KICAgIGRpcmVjdG9yaWVzIGV4aXN0IHVuZGVyIHRoZSBzdWJzZXQgZm9sZGVyLCBzbyB0aGUgc2FtZSBmdW5jdGlvbiB3b3JrcyBmb3IKICAgIExpYnJpMk1peCwgTGlicmkzTWl4LCBMaWJyaTRNaXgsIGFuZCBMaWJyaTVNaXggd2l0aG91dCBhbnkgZXh0cmEgYXJndW1lbnRzLgoKICAgIEV4cGVjdGVkIGxheW91dCAoMTYga0h6LCBtYXggbW9kZSk6CiAgICAgICAge2RhdGFfcm9vdH0vd2F2MTZrL21heC97c3Vic2V0fS9taXhfYm90aC8gICAjIGFsd2F5cyBwcmVzZW50CiAgICAgICAge2RhdGFfcm9vdH0vd2F2MTZrL21heC97c3Vic2V0fS9zMS8gICAgICAgICAjIE4gPj0gMQogICAgICAgIHtkYXRhX3Jvb3R9L3dhdjE2ay9tYXgve3N1YnNldH0vczIvICAgICAgICAgIyBOID49IDIKICAgICAgICB7ZGF0YV9yb290fS93YXYxNmsvbWF4L3tzdWJzZXR9L3MzLyAgICAgICAgICMgTiA+PSAzCiAgICAgICAge2RhdGFfcm9vdH0vd2F2MTZrL21heC97c3Vic2V0fS9zNC8gICAgICAgICAjIE4gPj0gNAogICAgICAgIHtkYXRhX3Jvb3R9L3dhdjE2ay9tYXgve3N1YnNldH0vczUvICAgICAgICAgIyBOID09IDUKCiAgICBBcmdzOgogICAgICAgIGRhdGFfcm9vdDogUm9vdCBvZiB0aGUgTGlicmlNaXggZGF0YXNldC4KICAgICAgICBzdWJzZXQ6IFNwbGl0IG5hbWUgKCd0cmFpbicsICdkZXYnLCAndGVzdCcpLgogICAgICAgIG1heF9zYW1wbGVzOiBDYXAgdGhlIG51bWJlciBvZiByZXR1cm5lZCBzYW1wbGVzLgoKICAgIFJldHVybnM6CiAgICAgICAgTGlzdCBvZiBNaXh0dXJlU2FtcGxlIG9iamVjdHMuCiAgICAiIiIKICAgIHJvb3QgPSBQYXRoKGRhdGFfcm9vdCkKICAgIHN1YnNldF9kaXIgPSByb290IC8gIndhdjE2ayIgLyAibWF4IiAvIHN1YnNldAogICAgbWl4X2RpciA9IHN1YnNldF9kaXIgLyAibWl4X2JvdGgiCiAgICBpZiBub3QgbWl4X2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJMaWJyaU1peCBtaXggZGlyZWN0b3J5IG5vdCBmb3VuZDoge21peF9kaXJ9XG4iCiAgICAgICAgICAgICJEb3dubG9hZCBMaWJyaU5NaXggYW5kIHNldCBkYXRhX3Jvb3QgaW4gY29uZmlncy9iYXNlbGluZS55YW1sLiIKICAgICAgICApCgogICAgbWl4X2ZpbGVzID0gc29ydGVkKG1peF9kaXIuZ2xvYigiKi53YXYiKSkKICAgIGlmIG1heF9zYW1wbGVzIGlzIG5vdCBOb25lOgogICAgICAgIG1peF9maWxlcyA9IG1peF9maWxlc1s6bWF4X3NhbXBsZXNdCgogICAgIyBBdXRvLWRldGVjdCBzcGVha2VyIGNvdW50IGZyb20gd2hpY2ggc04vIGRpcnMgZXhpc3QgKE49MS4uNSkuCiAgICAjIE9ubHkgcmFpc2UgaWYgdGhlcmUgYXJlIG1peCBmaWxlcyBidXQgbm8gc3RlbSBkaXJzIChjb3JydXB0ZWQgZGF0YXNldCkuCiAgICBtYXhfbiA9IHN1bSgxIGZvciBpIGluIHJhbmdlKDEsIDYpIGlmIChzdWJzZXRfZGlyIC8gZiJze2l9IikuaXNfZGlyKCkpCiAgICBpZiBtaXhfZmlsZXMgYW5kIG1heF9uIDwgMToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJObyBzcGVha2VyIHN0ZW0gZGlyZWN0b3JpZXMgKHMxLy4uczUvKSBmb3VuZCB1bmRlciB7c3Vic2V0X2Rpcn0iCiAgICAgICAgKQoKICAgIHNhbXBsZXM6IGxpc3RbTWl4dHVyZVNhbXBsZV0gPSBbXQogICAgZm9yIG1peF9wYXRoIGluIG1peF9maWxlczoKICAgICAgICB1aWQgPSBtaXhfcGF0aC5zdGVtCiAgICAgICAgcmVmczogbGlzdFtucC5uZGFycmF5XSA9IFtdCiAgICAgICAgc3I6IGludCB8IE5vbmUgPSBOb25lCiAgICAgICAgZm9yIHNwa19pZHggaW4gcmFuZ2UoMSwgbWF4X24gKyAxKToKICAgICAgICAgICAgcmVmX3BhdGggPSBzdWJzZXRfZGlyIC8gZiJze3Nwa19pZHh9IiAvIGYie3VpZH0ud2F2IgogICAgICAgICAgICBpZiBub3QgcmVmX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICByZWYsIHJlZl9zciA9IF9sb2FkX3dhdihyZWZfcGF0aCkKICAgICAgICAgICAgaWYgc3IgaXMgTm9uZToKICAgICAgICAgICAgICAgIHNyID0gcmVmX3NyCiAgICAgICAgICAgIHJlZnMuYXBwZW5kKHJlZikKCiAgICAgICAgaWYgbm90IHJlZnMgb3Igc3IgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgbWl4dHVyZSwgbWl4X3NyID0gX2xvYWRfd2F2KG1peF9wYXRoKQogICAgICAgIGlmIG1peF9zciAhPSBzcjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlNhbXBsZSByYXRlIG1pc21hdGNoIGZvciB7dWlkfTogbWl4PXttaXhfc3J9LCByZWY9e3NyfSIpCgogICAgICAgIG1pbl9sZW4gPSBtaW4obGVuKG1peHR1cmUpLCAqKGxlbihyKSBmb3IgciBpbiByZWZzKSkKICAgICAgICBtaXh0dXJlID0gbWl4dHVyZVs6bWluX2xlbl0KICAgICAgICByZWZzX2FyciA9IG5wLnN0YWNrKFtyWzptaW5fbGVuXSBmb3IgciBpbiByZWZzXSwgYXhpcz0wKQoKICAgICAgICBzYW1wbGVzLmFwcGVuZCgKICAgICAgICAgICAgTWl4dHVyZVNhbXBsZSgKICAgICAgICAgICAgICAgIG1peHR1cmU9bWl4dHVyZS5hc3R5cGUobnAuZmxvYXQzMiksCiAgICAgICAgICAgICAgICByZWZlcmVuY2VzPXJlZnNfYXJyLmFzdHlwZShucC5mbG9hdDMyKSwKICAgICAgICAgICAgICAgIHNhbXBsZV9yYXRlPXNyLAogICAgICAgICAgICAgICAgdXR0ZXJhbmNlX2lkPXVpZCwKICAgICAgICAgICAgKQogICAgICAgICkKCiAgICByZXR1cm4gc2FtcGxlcwoKCmRlZiBpdGVyX21peHR1cmVzKAogICAgZGF0YV9yb290OiBzdHIgfCBQYXRoLAogICAgc3Vic2V0OiBzdHIgPSAidGVzdCIsCiAgICBtYXhfc2FtcGxlczogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtNaXh0dXJlU2FtcGxlXToKICAgICIiIkFsaWFzIGZvciBkaXNjb3Zlcl9saWJyaW1peF9zYW1wbGVzIChiYXNlbGluZSBydW5uZXIgZW50cnkgcG9pbnQpLiIiIgogICAgcmV0dXJuIGRpc2NvdmVyX2xpYnJpbWl4X3NhbXBsZXMoZGF0YV9yb290LCBzdWJzZXQ9c3Vic2V0LCBtYXhfc2FtcGxlcz1tYXhfc2FtcGxlcykK'))
open(f'{PROJ}/data/rir_bank.py','wb').write(base64.b64decode('...'))
open(f'{PROJ}/data/codec_augmentation.py','wb').write(base64.b64decode('IiIiCkNvZGVjIGF1Z21lbnRhdGlvbiBwcm90b3R5cGUgZm9yIENBLU1vU0Ug4oCUIFBoYXNlIDEgcmVzZWFyY2ggZGVsaXZlcmFibGUuCgpTdGFnZSAzIG9mIHRoZSBhdWdtZW50YXRpb24gcGlwZWxpbmU6IHBob25lLWNoYW5uZWwgY29kZWMgZGlzdG9ydGlvbi4KU2ltdWxhdGVzIFdoYXRzQXBwIGFuZCB2b2ljZS1ub3RlIHJlY29yZGluZ3MgYnkgZW5jb2RpbmcgYXVkaW8gdGhyb3VnaApPcHVzIG9yIEFBQyBhdCBsb3cgYml0cmF0ZSAoNuKAkzMyIGticHMpIHRoZW4gZGVjb2RpbmcgYmFjayB0byBQQ00uCgpOYW1lZCBwcm9qZWN0IGNvbnRyaWJ1dGlvbjogbm8gb3RoZXIgdGVhbSBpbiBDQS1Nb1NFIHByZXBhcmVzIGZvciB0aGlzCmRlZ3JhZGF0aW9uLiBDb2RlYyBkaXN0b3J0aW9uIGF0IDbigJMxNiBrYnBzIGJyZWFrcyBoYXJtb25pYyBzdHJ1Y3R1cmUsCmF0dGVudWF0ZXMgaGlnaCBmcmVxdWVuY2llcywgYW5kIGludHJvZHVjZXMgYmxvY2sgYXJ0aWZhY3RzIOKAlCBhIGRpZmZlcmVudApkZWdyYWRhdGlvbiBwcm9maWxlIGZyb20gUklSIHJldmVyYiBvciBXSEFNISBub2lzZSwgcmVxdWlyaW5nIGl0cyBvd24KdHJhaW5pbmcgc2lnbmFsLgoKRGVzaWduIG5vdGU6IFRoaXMgbW9kdWxlIGlzIGludGVudGlvbmFsbHkgc3RhbmRhbG9uZSBzbyBpdCBjYW4gYmUgYXBwZW5kZWQKdG8gQXVnbWVudGF0aW9uUGlwZWxpbmUgaW4gUGhhc2UgNCB3aXRoIG9uZSBsaW5lOgogICAgbWl4dHVyZSDihpIgQXVnbWVudGF0aW9uUGlwZWxpbmUgKFN0YWdlIDErMikg4oaSIENvZGVjQXVnbWVudG9yIChTdGFnZSAzKQoKRXhlY3V0aW9uIHBhdGhzOgogIFByaW1hcnkgIOKAlCBmZm1wZWcgc3VicHJvY2VzcyAocmVhbCBPcHVzL0FBQyk7IHJlcXVpcmVzIGZmbXBlZyBvbiBQQVRILgogIEZhbGxiYWNrIOKAlCBtdS1sYXcgY29tcGFuZGluZyArIDgtYml0IHF1YW50aXNhdGlvbiAoc2NpcHkgb25seSk7IGFsd2F5cwogICAgICAgICAgICAgYXZhaWxhYmxlLCBtaW1pY3MgRy43MTEgdGVsZXBob25lLWNvZGVjIGFydGlmYWN0cy4KClJlYWxpc3RpYyBiaXRyYXRlIHRhcmdldHMgKFdoYXRzQXBwL1RlbGVncmFtIHZvaWNlIG5vdGVzKToKICBPcHVzIDogNuKAkzI0IGticHMgKFdoYXRzQXBwIGNhbGxzIHVzZSA24oCTMTYga2JwcyBhZGFwdGl2ZWx5KQogIEFBQyAgOiA44oCTMzIga2JwcyAoVGVsZWdyYW0sIGdlbmVyaWMgdm9pY2Utbm90ZSBlbmNvZGVycykKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgd2FybmluZ3MKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBzb3VuZGZpbGUgYXMgc2YKCmZyb20gZGF0YS5taXhlcl9zdHViIGltcG9ydCBNaXh0dXJlU2FtcGxlCgpfTVVMQVdfTVU6IGludCA9IDI1NQpfU1VQUE9SVEVEX0NPREVDUyA9ICgib3B1cyIsICJhYWMiLCAiYW1yLW5iIiwgImFtci13YiIpCgojIEFNUi1OQiBmaXhlZCBiaXRyYXRlIG1vZGVzIChicHMpLCBjbG9zZXN0IG1hdGNoIGlzIGNob3NlbiBhdCBlbmNvZGUgdGltZS4KX0FNUl9OQl9CSVRSQVRFU19CUFMgPSAoNDc1MCwgNTE1MCwgNTkwMCwgNjcwMCwgNzQwMCwgNzk1MCwgMTAyMDAsIDEyMjAwKQojIEFNUi1XQiBmaXhlZCBiaXRyYXRlIG1vZGVzIChicHMpLgpfQU1SX1dCX0JJVFJBVEVTX0JQUyA9ICg2NjAwLCA4ODUwLCAxMjY1MCwgMTQyNTAsIDE1ODUwLCAxODI1MCwgMTk4NTAsIDIzMDUwLCAyMzg1MCkKCl9DT0RFQ19GRk1QRUdfTkFNRSA9IHsKICAgICJvcHVzIjogImxpYm9wdXMiLAogICAgImFhYyI6ICJhYWMiLAogICAgImFtci1uYiI6ICJsaWJvcGVuY29yZV9hbXJuYiIsCiAgICAiYW1yLXdiIjogImxpYnZvX2FtcndiZW5jIiwKfQpfQ09ERUNfQ09OVEFJTkVSID0gewogICAgIm9wdXMiOiAiLm9wdXMiLAogICAgImFhYyI6ICIuYWFjIiwKICAgICJhbXItbmIiOiAiLmFtciIsCiAgICAiYW1yLXdiIjogIi5hbXIiLAp9CiMgQU1SLU5CIHJlcXVpcmVzIDgga0h6OyBBTVItV0IgcmVxdWlyZXMgMTYga0h6LgpfQ09ERUNfUkVRVUlSRURfU1I6IGRpY3Rbc3RyLCBpbnQgfCBOb25lXSA9IHsKICAgICJvcHVzIjogTm9uZSwgICAjIGFjY2VwdHMgYW55IHNyCiAgICAiYWFjIjogTm9uZSwKICAgICJhbXItbmIiOiA4XzAwMCwKICAgICJhbXItd2IiOiAxNl8wMDAsCn0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIFB1YmxpYyBoZWxwZXJzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgoKZGVmIGlzX2ZmbXBlZ19hdmFpbGFibGUoKSAtPiBib29sOgogICAgIiIiUmV0dXJuIFRydWUgaWYgZmZtcGVnIGlzIGZvdW5kIG9uIFBBVEggYW5kIGV4aXRzIHN1Y2Nlc3NmdWxseS4iIiIKICAgIHJldHVybiBzaHV0aWwud2hpY2goImZmbXBlZyIpIGlzIG5vdCBOb25lCgoKZGVmIGFwcGx5X2NvZGVjX3JvdW5kdHJpcCgKICAgIGF1ZGlvOiBucC5uZGFycmF5LAogICAgc2FtcGxlX3JhdGU6IGludCwKICAgIGNvZGVjOiBzdHIsCiAgICBiaXRyYXRlX2JwczogaW50IHwgZmxvYXQsCiAgICB0bXBfZGlyOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUsCikgLT4gbnAubmRhcnJheToKICAgICIiIgogICAgRW5jb2RlIOKGkiBkZWNvZGUgYGBhdWRpb2BgIHRocm91Z2ggYSByZWFsIGNvZGVjIGFuZCByZXR1cm4gdGhlIGRhbWFnZWQgc2lnbmFsLgoKICAgIFB1YmxpYyB3cmFwcGVyIHVzZWQgYnkgYGBkYXRhLmRlZ3JhZGF0aW9ucy5hcHBseV9jb2RlY2BgLiBTdXBwb3J0cyAib3B1cyIsCiAgICAiYWFjIiwgImFtci1uYiIsIGFuZCAiYW1yLXdiIi4gQU1SLU5CIHJlcXVpcmVzIDgga0h6IGlucHV0OyBBTVItV0IKICAgIHJlcXVpcmVzIDE2IGtIeiBpbnB1dCDigJQgZmZtcGVnIHdpbGwgcmVzYW1wbGUgaW50ZXJuYWxseSB3aGVuIHRoZSBpbnB1dAogICAgc2FtcGxlIHJhdGUgZGlmZmVycywgYnV0IGNhbGxlcnMgc2hvdWxkIG5vdGUgdGhlIHJlc2FtcGxpbmcuCgogICAgRmFsbHMgYmFjayB0byBtdS1sYXcgY29tcGFuZGluZyAoRy43MTEgYXBwcm94aW1hdGlvbikgd2hlbiBmZm1wZWcgaXMKICAgIHVuYXZhaWxhYmxlIG9yIHRoZSByb3VuZHRyaXAgZmFpbHMuCgogICAgQXJnczoKICAgICAgICBhdWRpbzogMS1EIGZsb2F0MzIgd2F2ZWZvcm0uCiAgICAgICAgc2FtcGxlX3JhdGU6IEF1ZGlvIHNhbXBsZSByYXRlIGluIEh6LgogICAgICAgIGNvZGVjOiBPbmUgb2YgIm9wdXMiLCAiYWFjIiwgImFtci1uYiIsICJhbXItd2IiLgogICAgICAgIGJpdHJhdGVfYnBzOiBUYXJnZXQgYml0cmF0ZSBpbiBiaXRzIHBlciBzZWNvbmQuCiAgICAgICAgdG1wX2RpcjogU2NyYXRjaCBkaXJlY3Rvcnk7IHVzZXMgYSBuZXcgdGVtcGRpciB3aGVuIE5vbmUuCgogICAgUmV0dXJuczoKICAgICAgICBEYW1hZ2VkIHdhdmVmb3JtLCBzYW1lIGxlbmd0aCBhcyBgYGF1ZGlvYGAsIGZsb2F0MzIuCiAgICAiIiIKICAgIGlmIGNvZGVjIG5vdCBpbiBfU1VQUE9SVEVEX0NPREVDUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY29kZWMge2NvZGVjIXJ9IG5vdCBzdXBwb3J0ZWQ7IGNob29zZSBmcm9tIHtfU1VQUE9SVEVEX0NPREVDU30iKQoKICAgIGF1ZGlvX2YzMiA9IG5wLmFzYXJyYXkoYXVkaW8sIGR0eXBlPW5wLmZsb2F0MzIpLnNxdWVlemUoKQogICAgYml0cmF0ZV9rYnBzID0gYml0cmF0ZV9icHMgLyAxMDAwLjAKCiAgICBpZiBub3QgaXNfZmZtcGVnX2F2YWlsYWJsZSgpOgogICAgICAgIHdhcm5pbmdzLndhcm4oCiAgICAgICAgICAgIGYiZmZtcGVnIG5vdCBmb3VuZDsgZmFsbGluZyBiYWNrIHRvIG11LWxhdyBzaW11bGF0aW9uIGZvciBjb2RlYyB7Y29kZWMhcn0uIiwKICAgICAgICAgICAgUnVudGltZVdhcm5pbmcsCiAgICAgICAgICAgIHN0YWNrbGV2ZWw9MiwKICAgICAgICApCiAgICAgICAgcmV0dXJuIF9tdWxhd19mYWxsYmFjayhhdWRpb19mMzIpCgogICAgcmVzdWx0ID0gX2ZmbXBlZ19yb3VuZHRyaXBfc3RhbmRhbG9uZShhdWRpb19mMzIsIHNhbXBsZV9yYXRlLCBjb2RlYywgYml0cmF0ZV9rYnBzLCB0bXBfZGlyKQogICAgaWYgcmVzdWx0IGlzIG5vdCBOb25lOgogICAgICAgIHJldHVybiByZXN1bHQKCiAgICB3YXJuaW5ncy53YXJuKAogICAgICAgIGYiZmZtcGVnIGNvZGVjIHJvdW5kdHJpcCBmYWlsZWQgZm9yIHtjb2RlYyFyfTsgZmFsbGluZyBiYWNrIHRvIG11LWxhdyBzaW11bGF0aW9uLiIsCiAgICAgICAgUnVudGltZVdhcm5pbmcsCiAgICAgICAgc3RhY2tsZXZlbD0yLAogICAgKQogICAgcmV0dXJuIF9tdWxhd19mYWxsYmFjayhhdWRpb19mMzIpCgoKZGVmIF9mZm1wZWdfcm91bmR0cmlwX3N0YW5kYWxvbmUoCiAgICBhdWRpbzogbnAubmRhcnJheSwKICAgIHNyOiBpbnQsCiAgICBjb2RlYzogc3RyLAogICAgYml0cmF0ZV9rYnBzOiBmbG9hdCwKICAgIHRtcF9kaXI6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKKSAtPiBucC5uZGFycmF5IHwgTm9uZToKICAgICIiIk1vZHVsZS1sZXZlbCBmZm1wZWcgcm91bmR0cmlwIHN1cHBvcnRpbmcgYWxsIGZvdXIgY29kZWNzLiBSZXR1cm5zIE5vbmUgb24gZmFpbHVyZS4iIiIKICAgIGZmbXBlZ19uYW1lID0gX0NPREVDX0ZGTVBFR19OQU1FW2NvZGVjXQogICAgY29udGFpbmVyID0gX0NPREVDX0NPTlRBSU5FUltjb2RlY10KICAgIHJlcXVpcmVkX3NyID0gX0NPREVDX1JFUVVJUkVEX1NSW2NvZGVjXQoKICAgICMgU25hcCBBTVIgYml0cmF0ZXMgdG8gdGhlIG5lYXJlc3QgdmFsaWQgbW9kZS4KICAgIGlmIGNvZGVjID09ICJhbXItbmIiOgogICAgICAgIGJpdHJhdGVfa2JwcyA9IF9zbmFwX2Ftcl9iaXRyYXRlKGJpdHJhdGVfa2JwcyAqIDEwMDAsIF9BTVJfTkJfQklUUkFURVNfQlBTKSAvIDEwMDAuMAogICAgZWxpZiBjb2RlYyA9PSAiYW1yLXdiIjoKICAgICAgICBiaXRyYXRlX2ticHMgPSBfc25hcF9hbXJfYml0cmF0ZShiaXRyYXRlX2ticHMgKiAxMDAwLCBfQU1SX1dCX0JJVFJBVEVTX0JQUykgLyAxMDAwLjAKCiAgICBjdHggPSB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBpZiB0bXBfZGlyIGlzIE5vbmUgZWxzZSBOb25lCiAgICB3b3JrID0gUGF0aChjdHgubmFtZSBpZiBjdHggZWxzZSB0bXBfZGlyKQogICAgdHJ5OgogICAgICAgIHNyYyA9IHdvcmsgLyAiaW5wdXQud2F2IgogICAgICAgIHNmLndyaXRlKHN0cihzcmMpLCBhdWRpbywgc3IsIHN1YnR5cGU9IkZMT0FUIikKCiAgICAgICAgZW5jb2RlZCA9IHdvcmsgLyBmImVuY29kZWR7Y29udGFpbmVyfSIKICAgICAgICBkZWNvZGVkID0gd29yayAvICJkZWNvZGVkLndhdiIKCiAgICAgICAgZW5jX2NtZCA9IFsiZmZtcGVnIiwgIi15IiwgIi1pIiwgc3RyKHNyYyksICItY29kZWM6YSIsIGZmbXBlZ19uYW1lLAogICAgICAgICAgICAgICAgICAgIi1iOmEiLCBmIntiaXRyYXRlX2ticHM6LjJmfWsiXQogICAgICAgIGlmIHJlcXVpcmVkX3NyIGlzIG5vdCBOb25lOgogICAgICAgICAgICBlbmNfY21kICs9IFsiLWFyIiwgc3RyKHJlcXVpcmVkX3NyKV0KICAgICAgICBlbmNfY21kLmFwcGVuZChzdHIoZW5jb2RlZCkpCgogICAgICAgIGlmIHN1YnByb2Nlc3MucnVuKGVuY19jbWQsIGNhcHR1cmVfb3V0cHV0PVRydWUsIGNoZWNrPUZhbHNlKS5yZXR1cm5jb2RlICE9IDA6CiAgICAgICAgICAgIHJldHVybiBOb25lCgogICAgICAgIGRlY19jbWQgPSBbImZmbXBlZyIsICIteSIsICItaSIsIHN0cihlbmNvZGVkKV0KICAgICAgICBpZiByZXF1aXJlZF9zciBpcyBub3QgTm9uZToKICAgICAgICAgICAgIyBSZXNhbXBsZSBiYWNrIHRvIG9yaWdpbmFsIHJhdGUgYWZ0ZXIgZGVjb2RpbmcuCiAgICAgICAgICAgIGRlY19jbWQgKz0gWyItYXIiLCBzdHIoc3IpXQogICAgICAgIGRlY19jbWQuYXBwZW5kKHN0cihkZWNvZGVkKSkKCiAgICAgICAgaWYgc3VicHJvY2Vzcy5ydW4oZGVjX2NtZCwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9RmFsc2UpLnJldHVybmNvZGUgIT0gMDoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCiAgICAgICAgb3V0LCBfID0gc2YucmVhZChzdHIoZGVjb2RlZCksIGR0eXBlPSJmbG9hdDMyIiwgYWx3YXlzXzJkPVRydWUpCiAgICAgICAgcmV0dXJuIF9maXRfbGVuZ3RoKG91dC5tZWFuKGF4aXM9MSksIGxlbihhdWRpbykpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25lCiAgICBmaW5hbGx5OgogICAgICAgIGlmIGN0eCBpcyBub3QgTm9uZToKICAgICAgICAgICAgY3R4LmNsZWFudXAoKQoKCmRlZiBfc25hcF9hbXJfYml0cmF0ZShiaXRyYXRlX2JwczogZmxvYXQsIHZhbGlkX3JhdGVzOiB0dXBsZVtpbnQsIC4uLl0pIC0+IGludDoKICAgICIiIlJldHVybiB0aGUgQU1SIGJpdHJhdGUgbW9kZSAoYnBzKSBjbG9zZXN0IHRvIGBgYml0cmF0ZV9icHNgYC4iIiIKICAgIHJldHVybiBtaW4odmFsaWRfcmF0ZXMsIGtleT1sYW1iZGEgcjogYWJzKHIgLSBiaXRyYXRlX2JwcykpCgoKZGVmIF9tdWxhd19mYWxsYmFjayhhdWRpbzogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICIiIkcuNzExIG11LWxhdyBhcHByb3hpbWF0aW9uLCB1c2VkIHdoZW4gZmZtcGVnIGlzIHVuYXZhaWxhYmxlLiIiIgogICAgeCA9IG5wLmNsaXAoYXVkaW8sIC0xLjAsIDEuMCkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBtdSA9IGZsb2F0KF9NVUxBV19NVSkKICAgIGVuY29kZWQgPSBucC5zaWduKHgpICogbnAubG9nMXAobXUgKiBucC5hYnMoeCkpIC8gbnAubG9nMXAobXUpCiAgICBxdWFudGlzZWQgPSBucC5yb3VuZChlbmNvZGVkICogMTI3LjApLmNsaXAoLTEyNywgMTI3KS5hc3R5cGUobnAuaW50OCkKICAgIHFfZmxvYXQgPSBxdWFudGlzZWQuYXN0eXBlKG5wLmZsb2F0NjQpIC8gMTI3LjAKICAgIGRlY29kZWQgPSBucC5zaWduKHFfZmxvYXQpICogKCgxLjAgKyBtdSkgKiogbnAuYWJzKHFfZmxvYXQpIC0gMS4wKSAvIG11CiAgICByZXR1cm4gZGVjb2RlZC5hc3R5cGUobnAuZmxvYXQzMikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENvbmZpZ3VyYXRpb24KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCgpAZGF0YWNsYXNzCmNsYXNzIENvZGVjQ29uZmlnOgogICAgIiIiQ29uZmlndXJhdGlvbiBmb3IgdGhlIGNvZGVjIGRpc3RvcnRpb24gYXVnbWVudG9yLgoKICAgIFJlYWxpc3RpYyBXaGF0c0FwcC92b2ljZS1ub3RlIGJpdHJhdGUgcmFuZ2VzOgogICAgICBPcHVzICA24oCTMjQga2JwcyAgKGRlZmF1bHQgNuKAkzMyIGNvdmVycyByZXNlYXJjaCBoZWFkcm9vbSkKICAgICAgQUFDICAgOOKAkzMyIGticHMKICAgICIiIgoKICAgIGNvZGVjOiBzdHIgPSAicmFuZG9tIgogICAgIiIiQ29kZWMgdG8gYXBwbHk6ICdvcHVzJywgJ2FhYycsIG9yICdyYW5kb20nIChjaG9zZW4gcGVyIHNhbXBsZSkuIiIiCgogICAgYml0cmF0ZV9taW5fa2JwczogZmxvYXQgPSA2LjAKICAgICIiIk1pbmltdW0gZW5jb2RpbmcgYml0cmF0ZSBpbiBrYnBzLiIiIgoKICAgIGJpdHJhdGVfbWF4X2ticHM6IGZsb2F0ID0gMzIuMAogICAgIiIiTWF4aW11bSBlbmNvZGluZyBiaXRyYXRlIGluIGticHMuIiIiCgogICAgY29kZWNfcHJvYjogZmxvYXQgPSAwLjMKICAgICIiIlByb2JhYmlsaXR5IG9mIGFwcGx5aW5nIGNvZGVjIGRpc3RvcnRpb24gcGVyIHNhbXBsZS4iIiIKCiAgICB1c2VfZmZtcGVnOiBib29sID0gVHJ1ZQogICAgIiIiVXNlIHJlYWwgT3B1cy9BQUMgY29kZWMgdmlhIGZmbXBlZy4gRmFsbHMgYmFjayB0byBtdS1sYXcgaWYgRmFsc2Ugb3IKICAgIGlmIGZmbXBlZyBpcyBub3QgaW5zdGFsbGVkLiIiIgoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdmFsaWQgPSAoKl9TVVBQT1JURURfQ09ERUNTLCAicmFuZG9tIikKICAgICAgICBpZiBzZWxmLmNvZGVjIG5vdCBpbiB2YWxpZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYiY29kZWMgbXVzdCBiZSBvbmUgb2Yge3ZhbGlkIXJ9LCBnb3Qge3NlbGYuY29kZWMhcn0iCiAgICAgICAgICAgICkKICAgICAgICBpZiBzZWxmLmJpdHJhdGVfbWluX2ticHMgPD0gMCBvciBzZWxmLmJpdHJhdGVfbWF4X2ticHMgPD0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYml0cmF0ZSB2YWx1ZXMgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgaWYgc2VsZi5iaXRyYXRlX21pbl9rYnBzID4gc2VsZi5iaXRyYXRlX21heF9rYnBzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJiaXRyYXRlX21pbl9rYnBzIG11c3QgYmUgPD0gYml0cmF0ZV9tYXhfa2JwcyIpCiAgICAgICAgaWYgbm90IDAuMCA8PSBzZWxmLmNvZGVjX3Byb2IgPD0gMS4wOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjb2RlY19wcm9iIG11c3QgYmUgaW4gWzAsIDFdIikKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEF1Z21lbnRvcgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmNsYXNzIENvZGVjQXVnbWVudG9yOgogICAgIiIiCiAgICBBcHBsaWVzIHByb2JhYmlsaXN0aWMgcGhvbmUtY2hhbm5lbCBjb2RlYyBkaXN0b3J0aW9uIHRvIGEgTWl4dHVyZVNhbXBsZS4KCiAgICBPbmx5IHRoZSBtaXh0dXJlIGlzIG1vZGlmaWVkOyByZWZlcmVuY2VzIChjbGVhbiBzdGVtcykgYXJlIHByZXNlcnZlZAogICAgc28gdGhhdCBTSS1TRFJpIGNhbiBiZSBjb21wdXRlZCBhZ2FpbnN0IHRoZSBvcmlnaW5hbCBjbGVhbiBzb3VyY2VzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgY29uZmlnOiBDb2RlY0NvbmZpZyB8IE5vbmUgPSBOb25lLAogICAgICAgIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBOb25lOgogICAgICAgIHNlbGYuY29uZmlnID0gY29uZmlnIG9yIENvZGVjQ29uZmlnKCkKICAgICAgICBzZWxmLnJuZyA9IHJuZyBpZiBybmcgaXMgbm90IE5vbmUgZWxzZSBucC5yYW5kb20uZGVmYXVsdF9ybmcoKQogICAgICAgIHNlbGYuX2ZmbXBlZ19vazogYm9vbCB8IE5vbmUgPSBOb25lICAjIGxhemlseSBjaGVja2VkCgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIHNhbXBsZTogTWl4dHVyZVNhbXBsZSkgLT4gTWl4dHVyZVNhbXBsZToKICAgICAgICAiIiJSZXR1cm4gYSBuZXcgTWl4dHVyZVNhbXBsZSB3aXRoIHRoZSBtaXh0dXJlIGNvZGVjLWRpc3RvcnRlZCwgcmVmZXJlbmNlcyB1bmNoYW5nZWQuIiIiCiAgICAgICAgaWYgc2VsZi5ybmcucmFuZG9tKCkgPj0gc2VsZi5jb25maWcuY29kZWNfcHJvYjoKICAgICAgICAgICAgcmV0dXJuIE1peHR1cmVTYW1wbGUoCiAgICAgICAgICAgICAgICBtaXh0dXJlPXNhbXBsZS5taXh0dXJlLmNvcHkoKSwKICAgICAgICAgICAgICAgIHJlZmVyZW5jZXM9c2FtcGxlLnJlZmVyZW5jZXMsCiAgICAgICAgICAgICAgICBzYW1wbGVfcmF0ZT1zYW1wbGUuc2FtcGxlX3JhdGUsCiAgICAgICAgICAgICAgICB1dHRlcmFuY2VfaWQ9c2FtcGxlLnV0dGVyYW5jZV9pZCwKICAgICAgICAgICAgKQoKICAgICAgICBtaXh0dXJlID0gc2VsZi5fZW5jb2RlX2RlY29kZShzYW1wbGUubWl4dHVyZSwgc2FtcGxlLnNhbXBsZV9yYXRlKQogICAgICAgIHJldHVybiBNaXh0dXJlU2FtcGxlKAogICAgICAgICAgICBtaXh0dXJlPW1peHR1cmUsCiAgICAgICAgICAgIHJlZmVyZW5jZXM9c2FtcGxlLnJlZmVyZW5jZXMsCiAgICAgICAgICAgIHNhbXBsZV9yYXRlPXNhbXBsZS5zYW1wbGVfcmF0ZSwKICAgICAgICAgICAgdXR0ZXJhbmNlX2lkPXNhbXBsZS51dHRlcmFuY2VfaWQsCiAgICAgICAgKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIERpc3BhdGNoCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKICAgIGRlZiBfZW5jb2RlX2RlY29kZShzZWxmLCBhdWRpbzogbnAubmRhcnJheSwgc3I6IGludCkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJBcHBseSBjb2RlYyBkaXN0b3J0aW9uOiBmZm1wZWcgaWYgYXZhaWxhYmxlLCBtdS1sYXcgb3RoZXJ3aXNlLiIiIgogICAgICAgIHVzZV9mZm1wZWcgPSBzZWxmLmNvbmZpZy51c2VfZmZtcGVnIGFuZCBzZWxmLl9mZm1wZWdfYXZhaWxhYmxlKCkKICAgICAgICBpZiB1c2VfZmZtcGVnOgogICAgICAgICAgICBjb2RlYyA9IHNlbGYuX3Jlc29sdmVfY29kZWMoKQogICAgICAgICAgICBiaXRyYXRlX2ticHMgPSBmbG9hdCgKICAgICAgICAgICAgICAgIHNlbGYucm5nLnVuaWZvcm0oc2VsZi5jb25maWcuYml0cmF0ZV9taW5fa2Jwcywgc2VsZi5jb25maWcuYml0cmF0ZV9tYXhfa2JwcykKICAgICAgICAgICAgKQogICAgICAgICAgICByZXN1bHQgPSBzZWxmLl9mZm1wZWdfcm91bmR0cmlwKGF1ZGlvLCBzciwgY29kZWMsIGJpdHJhdGVfa2JwcykKICAgICAgICAgICAgaWYgcmVzdWx0IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuIHJlc3VsdAogICAgICAgICAgICB3YXJuaW5ncy53YXJuKAogICAgICAgICAgICAgICAgImZmbXBlZyBjb2RlYyByb3VuZHRyaXAgZmFpbGVkOyBmYWxsaW5nIGJhY2sgdG8gbXUtbGF3IHNpbXVsYXRpb24uIiwKICAgICAgICAgICAgICAgIFJ1bnRpbWVXYXJuaW5nLAogICAgICAgICAgICAgICAgc3RhY2tsZXZlbD0zLAogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHNlbGYuX211bGF3X3JvdW5kdHJpcChhdWRpbykKCiAgICBkZWYgX3Jlc29sdmVfY29kZWMoc2VsZikgLT4gc3RyOgogICAgICAgIGlmIHNlbGYuY29uZmlnLmNvZGVjID09ICJyYW5kb20iOgogICAgICAgICAgICByZXR1cm4gc3RyKHNlbGYucm5nLmNob2ljZShfU1VQUE9SVEVEX0NPREVDUykpCiAgICAgICAgcmV0dXJuIHNlbGYuY29uZmlnLmNvZGVjCgogICAgZGVmIF9mZm1wZWdfYXZhaWxhYmxlKHNlbGYpIC0+IGJvb2w6CiAgICAgICAgaWYgc2VsZi5fZmZtcGVnX29rIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuX2ZmbXBlZ19vayA9IGlzX2ZmbXBlZ19hdmFpbGFibGUoKQogICAgICAgIHJldHVybiBzZWxmLl9mZm1wZWdfb2sKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgIyBQcmltYXJ5IHBhdGg6IHJlYWwgT3B1cyAvIEFBQyB2aWEgZmZtcGVnCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKICAgIGRlZiBfZmZtcGVnX3JvdW5kdHJpcCgKICAgICAgICBzZWxmLAogICAgICAgIGF1ZGlvOiBucC5uZGFycmF5LAogICAgICAgIHNyOiBpbnQsCiAgICAgICAgY29kZWM6IHN0ciwKICAgICAgICBiaXRyYXRlX2ticHM6IGZsb2F0LAogICAgKSAtPiBucC5uZGFycmF5IHwgTm9uZToKICAgICAgICAiIiJFbmNvZGUg4oaSIGRlY29kZSB0aHJvdWdoIGEgcmVhbCBjb2RlYy4gUmV0dXJucyBOb25lIG9uIGZhaWx1cmUuIiIiCiAgICAgICAgcmV0dXJuIF9mZm1wZWdfcm91bmR0cmlwX3N0YW5kYWxvbmUoYXVkaW8sIHNyLCBjb2RlYywgYml0cmF0ZV9rYnBzKQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICAjIEZhbGxiYWNrIHBhdGg6IG11LWxhdyBjb21wYW5kaW5nIChHLjcxMSBhcHByb3hpbWF0aW9uKQogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCiAgICBkZWYgX211bGF3X3JvdW5kdHJpcChzZWxmLCBhdWRpbzogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJHLjcxMSBtdS1sYXcgYXBwcm94aW1hdGlvbjsgcmVxdWlyZXMgbm8gYmluYXJ5IGRlcGVuZGVuY2llcy4iIiIKICAgICAgICByZXR1cm4gX211bGF3X2ZhbGxiYWNrKGF1ZGlvKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgSW50ZXJuYWwgaGVscGVycwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKCmRlZiBfZml0X2xlbmd0aChhdWRpbzogbnAubmRhcnJheSwgdGFyZ2V0OiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJUcmltIG9yIHplcm8tcGFkIGF1ZGlvIHRvIGV4YWN0bHkgYHRhcmdldGAgc2FtcGxlcy4iIiIKICAgIGlmIGxlbihhdWRpbykgPj0gdGFyZ2V0OgogICAgICAgIHJldHVybiBhdWRpb1s6dGFyZ2V0XS5hc3R5cGUobnAuZmxvYXQzMikKICAgIHBhZCA9IG5wLnplcm9zKHRhcmdldCAtIGxlbihhdWRpbyksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUoW2F1ZGlvLmFzdHlwZShucC5mbG9hdDMyKSwgcGFkXSkK'))

for f in ['train/__init__.py', 'train/losses.py', 'train/stage1_single.py', 'train/stage3_gate.py', 'train/stage4_joint.py', 'models/__init__.py', 'models/lora.py', 'models/condition.py', 'models/gate.py', 'data/__init__.py', 'data/calmsep_mixer.py', 'data/degradations.py', 'data/mixer_stub.py', 'data/rir_bank.py', 'data/codec_augmentation.py']:
    sz = Path(f'{PROJ}/{f}').stat().st_size
    print(f'  {f}: {sz} bytes')
print('All files baked.')

In [ ]:
import importlib
import os
import sys
from pathlib import Path

PROJ = '/tmp/calmsep_project'

for link, real in [
    (f'{PROJ}/data/calmsep-8k', AUDIO),
    (f'{PROJ}/checkpoints',     f'{WORK}/checkpoints'),
]:
    if not os.path.exists(link):
        os.makedirs(real, exist_ok=True)
        os.symlink(real, link)

os.chdir(PROJ)
print('cwd:', os.getcwd())

for p in [PROJ, '/tmp/sr_corrnet_src', '/tmp/loguru_stub', '/tmp/rotary_stub']:
    if p not in sys.path: sys.path.insert(0, p)
os.environ['PYTHONPATH'] = ':'.join([
    PROJ, '/tmp/sr_corrnet_src', '/tmp/loguru_stub', '/tmp/rotary_stub'])

for mod in ["models.lora", "models.gate", "models.condition",
            "train.losses", "train.stage1_single", "train.stage3_gate", "train.stage4_joint"]:
    try:
        importlib.import_module(mod)
        print(f'  {mod}: OK')
    except Exception as e:
        print(f'  {mod}: ERROR — {e}')

# Verify all paths
RIR_BANK  = os.path.join(AUDIO, 'rirs/bank.json')
NOISE_DIR = os.path.join(AUDIO, 'noise')
GATE_CKPT = os.path.join(CKPT_DIR, 'best_gate_init.pt')
print('RIR bank:', Path(RIR_BANK).exists(), RIR_BANK)
print('Noise dir:', Path(NOISE_DIR).exists(), NOISE_DIR)
print('Gate ckpt:', Path(GATE_CKPT).exists(), GATE_CKPT)
print('Reverb:', Path(f'{STAGE1_DIR}/best_reverb.pt').exists())
print('Noise:', Path(f'{STAGE1_DIR}/best_noise.pt').exists())
print('Codec:', Path(f'{STAGE1_DIR}/best_codec.pt').exists())


In [ ]:
import importlib
import os
import sys
from pathlib import Path

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ('models', 'train', 'data')):
        del sys.modules[mod]

import argparse

from train.stage4_joint import train_joint

RIR_BANK  = os.path.join(AUDIO, 'rirs/bank.json')
NOISE_DIR = os.path.join(AUDIO, 'noise')
GATE_CKPT = os.path.join(CKPT_DIR, 'best_gate_init.pt')

args = argparse.Namespace(
    librispeech_8k   = AUDIO,
    data_root        = AUDIO,
    rir_bank         = RIR_BANK if Path(RIR_BANK).exists() else '',
    noise_dir        = NOISE_DIR if Path(NOISE_DIR).exists() else '',
    adapter_reverb   = f'{STAGE1_DIR}/best_reverb.pt',
    adapter_noise    = f'{STAGE1_DIR}/best_noise.pt',
    adapter_codec    = f'{STAGE1_DIR}/best_codec.pt',
    stage1_dir       = STAGE1_DIR,
    stage3_dir       = CKPT_DIR,
    gate_checkpoint  = GATE_CKPT,
    output_dir       = CKPT_DIR,
    checkpoint_dir   = CKPT_DIR,
    hf_model         = 'shinuh/sr-corrnet-ss-1ch-wsj-var-2-5spk',
    device           = 'cuda',
    epochs           = 20,
    batch_size       = 4,
    lr_adapter       = 1e-5,
    lr_gate          = 2e-5,
    lr               = 0.0,
    stage1_lr        = 0.0,
    seed             = 42,
    samples_per_epoch= 1000,
    num_workers      = 0,
    bf16             = True,
    max_audio_samples= 8000,
)

print('Stage 4 joint training — 20 epochs, 1000 samples/epoch')
print('lr_adapter=1e-5  lr_gate=2e-5  bf16=True')
train_joint(args)

for name in ['best_joint.pt', 'final_joint.pt']:
    p = Path(CKPT_DIR) / name
    if p.exists():
        print(f'{p}  {p.stat().st_size/1024:.1f} KB')
